# FPLVORTEX V4.6 · Verified private state or provisional research from public FPL data + 10-GW chip analysis

## Minimal-change dependency map — inspected *before editing* V4.1

V4.1 contains **four notebook cells**: 0 = instructions; 1 = one 1,300+ line engine; 2 = user-setting notes; 3 = settings + `run()`.
The engine has these ordered dependency sections (line numbers refer to the original V4.1 Python cell):

| Original cell / lines | Dependency | Responsible functions | V4.2 change |
|---|---|---|---|
| 1: 33–211 | Public FPL collection | `fetch`, `collect`, `determine_events` | Reused; new raw archive wrapper |
| 1: 221–361 | Public news, markets, benchmark lookups | `collect_news_radar`, `collect_rank_benchmarks`, `collect_polymarket_context` | Reused; research signals not confirmations |
| 1: 364–433 | Old GW7 screenshot overlay | `_resolve_screenshot_player`, `apply_gw7_screen_squad` | **Disabled**; verified manager JSON replaces it |
| 1: 436–458 | `Forecast` / `Route` structs | `Forecast`, `Route` | Reused |
| 1: 461–700 | Player history, minutes, per-GW projections | `DecisionEngine`, `create_player_stats`, `player_for_gw` | Reused + explicit component extraction and bounded official role priors |
| 1: 700–790 | XI, formation, transfer paths | `lineup`, `plan`, `transfer_candidates` | Reused; adds independent path/budget audit |
| 1: 790–877 | Chip legality + joint MILP | `optimize_squad`, `chip_availability` | Reused; Wildcard now compared against hold **and** repair |
| 1: 880–961 | Legacy simulation/chip recommendation | `simulate_lineup`, `chip_solver` | Original retained; **audited 5,000-draw sampler supersedes for final actions** |
| 1: 984–1239 | HTML, assets, reporting | `embed_visual_assets`, `make_result`, `render_html` | Reused, additional audited tab + correct scenario contract |
| 1: 1242–1306 | 4K PNG | `draw_four_k` | Kept; run only after legal/fresh/verified state gates |
| 1: 1313–1364 | Google Drive and orchestration | `google_drive_folder`, `run` | Kept for legacy; final cell calls `run_audited` instead |

**MP4 audit:** V4.1 contains **no MP4 encoding/export function or cell**. This upgrade doesn't alter any other notebook's MP4 workflow. Call `assert_pre_render` before adding a separate MP4 renderer; no video is claimed to exist here.

## New cells (in execution order)

- **V4.2 cell 3 — Manager-state, immutable raw/state archives, legality gates, explicit per-GW projection components.**
- **V4.2 cell 4 — 5,000-draw Monte Carlo; hold/repair/Wildcard rolling paths, confidence, minutes-sensitivity, captain choices.**
- **V4.2 cell 5 — Existing-renderer compatibility bridge, clickable audit tab, timestamped diagnostics, versioned projection export, `run_audited()`.**
- **V4.2 cell 6 — Strict predeadline replay capture, subsequent settlement helper and historical CSV/JSON backtest.**

Core HTML/PNG filenames and Google Drive mount/location are unchanged.

## Running safely

1. Upload this `.ipynb` to Google Colab and **Run all**. It mounts Drive and writes `My Drive/FPLVORTEX/Input/manager_state.json`, seeded with the **last public locked** XV for convenience, **not current-GW transfers**.
2. The first run will intentionally **stop** and write `FPLVORTEX_LATEST_DIAGNOSTICS.html`. This is correct: the public API cannot verify unannounced transfers, their exact sale prices, the current display of hit cost or free transfers.
3. Update `Input/manager_state.json` from the current official FPL **Transfers / Pick Team** screens: 15 `{element,selling_price}` rows, bank/value in FPL tenths, FTs before/after moves, count of moves, *displayed* hit cost, chip availability, current `next_event`, and true `ft_after_wildcard` only after FPL confirms it. Set an ISO 8601 UTC `verified_at_utc`, and `confirmed: true` only after checking every field. Preserve the exact player IDs from the public API where they are still on your team.
4. **Run all again**. If required state is still missing/stale/contradictory, no new dashboard recommendation is rendered. See diagnostics. Do not rely on an older HTML file.
5. For historical replay only, set `BACKTEST_ONLY = True` in the last cell and run all. It reads frozen deadline snapshots and post-deadline outcome files—never current aggregate data as historical features.

This engine never logs in to your FPL account and never initiates transfers or activates chips. 6–8 GW is configurable by changing `HORIZON` in the last cell (default 6). The current action alone is actionable; later weeks are tentative.


# FPLVORTEX V4.2 compatibility base (original V4.1 engine) — GW7 Wildcard optimiser (confirmed budget and zero hits) (team #145784)

This notebook uses your *GW7 post-four-transfer screenshot squad*, not the stale GW6 public squad.

**Run:** Upload to Google Colab → `Runtime → Run all` → approve Google Drive access. It creates one interactive HTML report, one 3840 × 2160 PNG, and a reproducible JSON analysis in `My Drive/FPLVORTEX/Reports/`.

**Model decision:** No predefined Haaland/anti-Haaland scenario. The solver chooses the legal 15-man squad, best 11 + formation, captain, and six-GW plan by expected points with minutes/injury adjustments, three-per-club constraint and budget. It weighs playing the remaining first-half Wildcard against a six-GW no-chip transfer path, taking your four transfers (and possible points hits) into account.

**Screenshot-confirmed inputs (11 Oct 2026):** £2.8m available budget, **0 free transfers**, **0 points transfer cost**, GW7 Wildcard available. The displayed squad is reconciled from your GW7 screenshot rather than relying on the GW6 API snapshot. We estimate you used four banked free transfers, but the original balance is inferred; the zero points cost is directly visible. **Final selling prices can differ from player card prices**, so verify affordability in the official FPL transfer screen before confirming any Wildcard transfers. Public FPL API does not return private GW7 transfers; screenshot roster is entered below and checked against player IDs. User decisions and late injury news may change the output. No rank guarantee.


In [ ]:
from __future__ import annotations

import base64
import io
from concurrent.futures import ThreadPoolExecutor, as_completed
import csv
import html
import json
import math
import random
import re
import xml.etree.ElementTree as ET
from urllib.parse import urlencode, urlparse
from scipy.sparse import lil_matrix
import unicodedata
from email.utils import parsedate_to_datetime
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

import numpy as np
import requests
from scipy.optimize import Bounds, LinearConstraint, milp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyBboxPatch, Rectangle
from IPython.display import HTML, display

ENTRY_ID: int = 145784
DRIVE_DIRECTORY: str = 'FPLVORTEX/Reports'
HORIZON: int = 10
# GW7 transfer screenshot: 0 FTs remain and 0 points cost after 4 moves.
FREE_TRANSFERS_ASSUMED: int = 0
BANK_OVERRIDE_TENTHS: int | None = None
REQUEST_DELAY_SECONDS: float = 0.12
MAX_ELEMENT_SUMMARIES: int = 62
SUMMARY_CACHE_HOURS: int = 7
SIMULATIONS: int = 1200
BEAM_WIDTH: int = 10
NUM_RESULTS: int = 10
ENABLE_REAL_HEADSHOTS: bool = True  # From public Premier League CDN when reachable; fallback to teal avatars.
API_BASE: str = 'https://fantasy.premierleague.com/api/'
PRIMARY_RANK_TARGET: int = 5000
BACKUP_RANK_TARGET: int = 10000
# Screenshot taken after 4 GW7 transfers; the public picks endpoint is still GW6.
# The model resolves these against official player IDs and REFUSES ambiguous names.
GW7_SCREENSHOT_SQUAD: tuple[tuple[str,int], ...] = ()  # Legacy screenshot input disabled; current state now comes from verified manager_state.json.
USE_GW7_SCREENSHOT_SQUAD: bool = False
CURRENT_BANK_TENTHS: int | None = None  # Audited workflow reads verified current manager-state input.
SAVED_FTS_BEFORE_GW7_TRANSFERS: int = 0  # Legacy placeholder; not used in audit.
GW7_TRANSFERS_ALREADY_MADE: int = 0  # Legacy placeholder; not used in audit.
GW7_TRANSFER_COST_CONFIRMED_PTS: int = 0  # Legacy placeholder; verified displayed hit read from state.
WILDCARD_OPPORTUNITY_COST_PTS: float = 6.0  # Explicit heuristic, NOT observed market data.

SOURCE_TIMEOUT_SECONDS: float = 9.0
STYLE_BG = '#071826'
STYLE_TEAL = '#1ce7d1'
UTC = timezone.utc
POS_NAME = {1:'GK',2:'DEF',3:'MID',4:'FWD'}
POSITION_COUNTS = {1:2, 2:5, 3:5, 4:3}


def num(value: Any, default: float = 0.0) -> float:
    try:
        value = float(value)
        return value if math.isfinite(value) else default
    except (TypeError, ValueError, OverflowError):
        return default


def integer(value: Any, default: int = 0) -> int:
    try:
        return int(value)
    except (TypeError, ValueError, OverflowError):
        return default


def clamp(v: float, a: float, b: float) -> float:
    return max(a, min(b, v))


def esc(v: Any) -> str:
    return html.escape(str(v), quote=True)


def iso_time(value: Any) -> datetime | None:
    if not value:
        return None
    try:
        out = datetime.fromisoformat(str(value).replace('Z','+00:00'))
        return out.replace(tzinfo=UTC) if out.tzinfo is None else out.astimezone(UTC)
    except (ValueError, TypeError):
        return None


def fetch(session: requests.Session, endpoint: str, optional: bool = False) -> Any:
    if endpoint.startswith('/') or '://' in endpoint:
        raise ValueError('Unsafe FPL endpoint')
    for attempt in range(4):
        try:
            time.sleep(REQUEST_DELAY_SECONDS)
            response = session.get(API_BASE + endpoint, timeout=24)
            if optional and response.status_code in (403, 404):
                return None
            if response.status_code in (429, 500, 502, 503, 504) and attempt < 3:
                time.sleep(min(2 ** (attempt+1), 6))
                continue
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError):
            if attempt == 3:
                if optional:
                    return None
                raise RuntimeError(f'Cannot retrieve official FPL {endpoint}. No speculative live analysis produced.')
            time.sleep(min(2 ** (attempt+1), 5))
    return None


def get_session() -> requests.Session:
    session = requests.Session()
    session.headers.update({'Accept':'application/json','User-Agent':'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/127 Safari/537.36','Referer':'https://fantasy.premierleague.com/'})
    return session


def latest_published_gw(profile: dict[str,Any], history: dict[str,Any]) -> list[int]:
    historical = {integer(x.get('event')) for x in history.get('current',[]) if integer(x.get('event'))>0}
    current = integer(profile.get('current_event'))
    if current: historical.add(current)
    return sorted(historical,reverse=True)


def determine_events(bootstrap: dict[str,Any], now: datetime | None = None) -> list[int]:
    now = now or datetime.now(UTC)
    events = sorted(bootstrap.get('events',[]),key=lambda e:integer(e.get('id')))
    future = [integer(x['id']) for x in events if iso_time(x.get('deadline_time')) and iso_time(x['deadline_time']) > now]
    if not future:
        future = [integer(x['id']) for x in events if x.get('is_next')]
    return future[:HORIZON]


def price_for_owned(pick: dict[str,Any], player: dict[str,Any]) -> int:
    return integer(pick.get('selling_price'),integer(player.get('now_cost')))


def collect(output: Path) -> dict[str,Any]:
    cache_dir = output / 'Cache'
    cache_dir.mkdir(parents=True,exist_ok=True)
    with get_session() as session:
        print(f'📡 FPLVORTEX: reading official FPL data for entry #{ENTRY_ID}')
        bootstrap = fetch(session,'bootstrap-static/')
        profile = fetch(session,f'entry/{ENTRY_ID}/')
        history = fetch(session,f'entry/{ENTRY_ID}/history/')
        fixtures = fetch(session,'fixtures/')
        transfers = fetch(session,f'entry/{ENTRY_ID}/transfers/',optional=True) or []
        if not isinstance(bootstrap,dict) or len(bootstrap.get('elements',[])) < 100 or not isinstance(fixtures,list):
            raise RuntimeError('Invalid player / fixtures payload; refusing to forecast.')
        if integer(profile.get('id')) != ENTRY_ID:
            raise RuntimeError('Wrong FPL entry returned.')
        picks = None
        picked_gw = None
        for gw in latest_published_gw(profile,history):
            candidate = fetch(session,f'entry/{ENTRY_ID}/event/{gw}/picks/',optional=True)
            if isinstance(candidate,dict) and len(candidate.get('picks',[]))==15:
                # A Free Hit squad is temporary; next deadline reverts to the prior squad.
                if candidate.get('active_chip')=='freehit':
                    print(f'  GW{gw} was a Free Hit; using earlier permanent squad.')
                    continue
                picks,picked_gw = candidate,gw
                break
        if picks is None:
            raise RuntimeError('Public 15-player squad unavailable. Cannot infer unseen transfers or recommend a chip.')
        selected = {integer(p.get('element')) for p in picks['picks']}
        players = bootstrap['elements']
        # A rank + price diversified pool protects budget feasibility and limits rate usage.
        for pos in (1,2,3,4):
            eligible=[p for p in players if integer(p.get('element_type'))==pos and p.get('status') not in ('u','s')]
            score=lambda p: .52*num(p.get('form')) + .35*num(p.get('points_per_game')) + .13*(num(p.get('expected_goals'))+num(p.get('expected_assists')))*3
            selected.update(integer(p['id']) for p in sorted(eligible,key=score,reverse=True)[:(10 if pos==1 else 17)])
            cheap=sorted(eligible,key=lambda p:integer(p.get('now_cost')))
            selected.update(integer(p['id']) for p in cheap[:(3 if pos==1 else 5)])
        own={integer(p['element']) for p in picks['picks']}
        owned_order = list(own)
        remaining=sorted((p for p in players if integer(p.get('id')) in selected and integer(p.get('id')) not in own),key=lambda p:num(p.get('form'))+num(p.get('points_per_game')),reverse=True)
        summary_ids=(owned_order + [integer(p['id']) for p in remaining])[:MAX_ELEMENT_SUMMARIES]
        summaries: dict[str,Any]={}
        failures=0
        for idx,pid in enumerate(summary_ids):
            path=cache_dir / f'player_{pid}.json'
            cached=None
            if path.exists() and (time.time()-path.stat().st_mtime)< SUMMARY_CACHE_HOURS*3600:
                try: cached=json.loads(path.read_text(encoding='utf-8'))
                except (OSError,ValueError): pass
            if cached is None:
                cached=fetch(session,f'element-summary/{pid}/',optional=True)
                if isinstance(cached,dict) and isinstance(cached.get('history'),list):
                    path.write_text(json.dumps(cached,ensure_ascii=False),encoding='utf-8')
            if isinstance(cached,dict):summaries[str(pid)]=cached
            else:failures+=1
            if (idx+1)%15==0: print(f'  Player histories: {idx+1}/{len(summary_ids)}')
        print(f'✅ Read {len(summaries)} detailed player histories; {failures} unavailable (season fallback).')
        benchmarks=collect_rank_benchmarks({'profile':profile},session)
    return dict(collected_utc=datetime.now(UTC).isoformat(),entry_id=ENTRY_ID,profile=profile,history=history,bootstrap=bootstrap,fixtures=fixtures,transfers=transfers,picks=picks,picked_gw=picked_gw,summaries=summaries,rank_benchmarks=benchmarks)



# Independent, no-login feeds are treated as REPORTS, not lineup confirmations.
NEWS_FEEDS: tuple[tuple[str,str],...] = (
    ('Guardian Premier League','https://www.theguardian.com/football/premierleague/rss'),
    ('Google News UK','https://news.google.com/rss/search?' + urlencode({'q':'Premier League team news injury expected lineup when:3d', 'hl':'en-GB','gl':'GB','ceid':'GB:en'})),
)

def safe_public_get(url: str, timeout: float = SOURCE_TIMEOUT_SECONDS) -> requests.Response | None:
    hostname = urlparse(url).hostname or ''
    if hostname not in {'www.theguardian.com','news.google.com','gamma-api.polymarket.com'}:
        raise ValueError(f'External-source host not on allowlist: {hostname}')
    try:
        response = requests.get(url,timeout=timeout,headers={'User-Agent':'Mozilla/5.0 FPLVORTEX research (read only)'})
        response.raise_for_status()
        return response
    except requests.RequestException:
        return None


def parse_rss_response(xml_text: str, source: str, now: datetime) -> list[dict[str,Any]]:
    articles=[]
    try: root=ET.fromstring(xml_text)
    except ET.ParseError: return articles
    for item in root.findall('.//item')[:65]:
        title=(item.findtext('title') or '').strip()
        url=(item.findtext('link') or '').strip()
        published_text=(item.findtext('pubDate') or '').strip()
        if not title or not url or not url.startswith('https://'):continue
        try:
            published=parsedate_to_datetime(published_text)
            if published.tzinfo is None:published=published.replace(tzinfo=UTC)
            published=published.astimezone(UTC)
        except (ValueError,TypeError,OverflowError):continue
        age=(now-published).total_seconds()/3600
        if age<-.5 or age>96:continue
        rss_source=item.find('source')
        publisher=(rss_source.text or '').strip() if rss_source is not None else source
        articles.append({'title':title[:250],'url':url,'published_utc':published.isoformat(),
                         'publisher':publisher or source,'feed':source,'age_hours':round(age,1)})
    return articles


def collect_news_radar(snapshot:dict[str,Any], fetch_external:bool=True) -> dict[str,Any]:
    players={integer(p['id']):p for p in snapshot['bootstrap']['elements']}
    picks=snapshot['picks']['picks']
    now=datetime.now(UTC)
    reports=[]; status=[]
    if fetch_external:
        for name,url in NEWS_FEEDS:
            response=safe_public_get(url)
            status.append({'source':name,'ok':response is not None})
            if response is not None:reports.extend(parse_rss_response(response.text,name,now))
    else:
        status=[{'source':name,'ok':False,'note':'External retrieval skipped for offline test'} for name,_ in NEWS_FEEDS]
    unique={(r['publisher'].casefold(),r['title'].casefold()):r for r in reports}
    reports=list(unique.values())
    official=[]; player_reports=[]
    for pick in picks:
        pid=integer(pick['element']); player=players[pid]
        name=str(player.get('web_name') or player.get('second_name') or '')
        status_code=str(player.get('status','a'))
        official.append({'player_id':pid,'name':name,'status':status_code,'news':str(player.get('news') or ''),
                         'chance':player.get('chance_of_playing_next_round'),
                         'news_added':player.get('news_added')})
        # Only strict whole-word matching. Broad names produce more false positives; expose exact headline links.
        matches=[r for r in reports if len(name)>=5 and re.search(r'(?<!\w)'+re.escape(name)+r'(?!\w)',r['title'],flags=re.I)]
        if matches:
            publications=len({r['publisher'].casefold() for r in matches})
            player_reports.append({'player_id':pid,'name':name,'independent_publishers':publications,
                                   'articles':sorted(matches,key=lambda item:item['age_hours'])[:6],
                                   'interpretation':'Possible news coverage, NOT a confirmed lineup or independent verification of a particular claim.'})
    return {'retrieved_utc':now.isoformat(),'feed_status':status,'articles_checked':len(reports),
            'official_flags':official,'headline_matches':player_reports,
            'rule':'Headlines never override starting probabilities; only authenticated official status or manually verified multi-source reports may do so.',
            'x_rumors':'Unauthenticated X searches cannot reliably prove a lineup leak or independent sourcing. No X corroboration claimed.'}


def collect_rank_benchmarks(snapshot:dict[str,Any], session:requests.Session) -> dict[str,Any]:
    output={}
    total=integer(snapshot['profile'].get('summary_overall_points'))
    overall_league=next((integer(league.get('id')) for league in snapshot['profile'].get('leagues',{}).get('classic',[]) if league.get('short_name')=='overall'),0)
    if not overall_league:return output
    for target in (PRIMARY_RANK_TARGET,BACKUP_RANK_TARGET):
        page=max(1,(target-1)//50+1)
        payload=fetch(session,f'leagues-classic/{overall_league}/standings/?page_standings={page}',optional=True)
        records=payload.get('standings',{}).get('results',[]) if isinstance(payload,dict) else []
        candidates=[item for item in records if abs(integer(item.get('rank'))-target)<=25]
        if not candidates:continue
        row=min(candidates,key=lambda record:abs(integer(record.get('rank'))-target))
        rival_points=integer(row.get('total'),-1)
        if rival_points<0:continue
        output[str(target)]={'approx_rank':integer(row['rank']),'approx_points':rival_points,
                             'points_gap':max(0,rival_points-total),'source':'Official overall league standings',
                             'qualification':'Approximate live total; rank / total can change during a live gameweek.'}
    return output


def collect_polymarket_context(snapshot:dict[str,Any], fetch_external:bool=True) -> dict[str,Any]:
    # Display-only research context. Odds from three-way markets must not be misrepresented as xG/player props.
    payload={'source':'Polymarket public Gamma API','available':False,'matches':[],
             'note':'Only matching liquid EPL 1X2 markets are shown. No automatic injury or lineup probabilities.'}
    if not fetch_external:return payload
    fixtures=[f for f in snapshot['fixtures'] if not f.get('finished') and f.get('kickoff_time')]
    teams={integer(t['id']):str(t.get('name','')) for t in snapshot['bootstrap']['teams']}
    url='https://gamma-api.polymarket.com/events?'+urlencode({'active':'true','closed':'false','tag_slug':'soccer','limit':100})
    response=safe_public_get(url)
    if response is None:return payload
    try:events=response.json()
    except (requests.exceptions.JSONDecodeError,ValueError):return payload
    if not isinstance(events,list):return payload
    payload['available']=True
    payload['events_checked']=len(events)
    for fixture in fixtures:
        kickoff=iso_time(fixture.get('kickoff_time'))
        if not kickoff:continue
        hname=teams.get(integer(fixture.get('team_h')),'');aname=teams.get(integer(fixture.get('team_a')),'')
        if not hname or not aname:continue
        for event in events:
            title=str(event.get('title') or '')
            if not all(re.search(r'(?<!\w)'+re.escape(name)+r'(?!\w)',title,re.I) for name in (hname,aname)):
                continue
            start=iso_time(event.get('startDate') or event.get('startTime'))
            if not start or abs((kickoff-start).total_seconds())>60*60*36:continue
            for market in event.get('markets',[]):
                try:
                    outcomes=market.get('outcomes');prices=market.get('outcomePrices')
                    if isinstance(outcomes,str):outcomes=json.loads(outcomes)
                    if isinstance(prices,str):prices=json.loads(prices)
                    if len(outcomes)!=3 or len(prices)!=3:continue
                    result={str(k).casefold().strip():float(v) for k,v in zip(outcomes,prices)}
                    keys=list(result)
                    home_key=next((k for k in keys if k==hname.casefold()),None)
                    away_key=next((k for k in keys if k==aname.casefold()),None)
                    draw_key=next((k for k in keys if k in ('draw','tie')),None)
                    if not (home_key and away_key and draw_key):continue
                    volume=num(market.get('liquidityNum') or market.get('liquidity'))
                    divisor=sum(result.values())
                    if volume<500 or not .85<=divisor<=1.20:continue
                    prices_norm={k:round(result[key]/divisor,3) for k,key in [('home',home_key),('draw',draw_key),('away',away_key)]}
                    payload['matches'].append({'fixture_id':integer(fixture.get('id')),'fixture':f'{hname} vs {aname}',
                                               'gw':integer(fixture.get('event')),'kickoff':kickoff.isoformat(),
                                               'probabilities':prices_norm,'liquidity_usd_approx':round(volume),
                                               'url':'https://polymarket.com/event/'+str(event.get('slug') or ''),
                                               'source':'Three-way contract prices, liquidity filtered; not bookmaker xG.'})
                    break
                except (TypeError,ValueError,KeyError,IndexError):continue
    payload['matches']=list({r['fixture_id']:r for r in payload['matches']}.values())
    return payload


def _squashed_name(value: str) -> str:
    decomposed=unicodedata.normalize('NFKD',str(value)).casefold()
    return ''.join(ch for ch in decomposed if ch.isalnum() and not unicodedata.combining(ch))


def _resolve_screenshot_player(players: list[dict[str,Any]], name: str, position: int) -> dict[str,Any]:
    lookup=_squashed_name(name)
    matches=[]
    for player in players:
        if integer(player.get('element_type'))!=position:continue
        variants=(player.get('web_name',''),player.get('second_name',''),
                  (str(player.get('first_name',''))+' '+str(player.get('second_name',''))).strip())
        first=_squashed_name(str(player.get('first_name','')))
        last=_squashed_name(str(player.get('second_name','')))
        if any(_squashed_name(v)==lookup for v in variants) or (first and last and lookup==first[0]+last):
            matches.append(player)
    if len(matches)!=1:
        label=', '.join(str(p.get('web_name'))+':'+str(p.get('id')) for p in matches)
        raise ValueError(f'Cannot uniquely match screenshot player {name!r} ({POS_NAME[position]}): {label or "NOT FOUND"}. Correct the GW7_SCREENSHOT_SQUAD settings.')
    return matches[0]


def apply_gw7_screen_squad(snapshot: dict[str,Any]) -> dict[str,Any]:
    if not USE_GW7_SCREENSHOT_SQUAD:return snapshot
    players=snapshot['bootstrap']['elements']
    resolved=[_resolve_screenshot_player(players,name,pos) for name,pos in GW7_SCREENSHOT_SQUAD]
    desired={integer(p['id']) for p in resolved}
    if len(desired)!=15:raise ValueError('Screenshot contains repeated FPL player IDs.')
    locked=snapshot['picks']['picks']
    previous={integer(p['element']):p for p in locked}
    now={integer(p['id']):p for p in players}
    outgoing=set(previous)-desired; incoming=desired-set(previous)
    if len(outgoing)!=4 or len(incoming)!=4:
        raise ValueError(f'Expected precisely four squad changes, found {len(outgoing)} OUT and {len(incoming)} IN. Verify the screenshot player list versus GW{snapshot["picked_gw"]} picks; refusing to guess budget.')
    positions=lambda player:integer(player['element_type'])
    for pos in POSITION_COUNTS:
        if sum(positions(p)==pos for p in resolved)!=POSITION_COUNTS[pos]:
            raise ValueError(f'Illegal screenshot squad position count: {POS_NAME[pos]}.')
    clubs=Counter(integer(p['team']) for p in resolved)
    if max(clubs.values())>3:raise ValueError('Screenshot squad has >3 players from one club.')
    # Latest GW6 public selling prices only approximate GW7 sale prices after intraweek rises/drops.
    old_bank=integer(snapshot['profile'].get('last_deadline_bank'))
    estimated_bank=(old_bank + sum(price_for_owned(previous[pid],now[pid]) for pid in outgoing)
                    - sum(integer(now[pid]['now_cost']) for pid in incoming))
    bank=estimated_bank if CURRENT_BANK_TENTHS is None else CURRENT_BANK_TENTHS
    if bank<0:raise ValueError(f'Estimated GW7 bank £{estimated_bank/10:.1f}m is negative. Set CURRENT_BANK_TENTHS to your exact current bank in the notebook; no invented affordable roster.')
    new_picks=[]
    for idx,player in enumerate(resolved):
        pid=integer(player['id'])
        prior=previous.get(pid)
        # New acquisitions are assumed to sell at current price; enter exact bank to correct budget errors.
        selling=price_for_owned(prior,player) if prior is not None else integer(player['now_cost'])
        new_picks.append({'element':pid,'position':idx+1,'selling_price':selling,
                          'purchase_price':selling,'is_captain':False,'is_vice_captain':False,'multiplier':0})
    patched=dict(snapshot)
    patched['picks']={**snapshot['picks'],'picks':new_picks}
    patched['profile']={**snapshot['profile'],'last_deadline_bank':bank}
    patched['squad_provenance']='YOUR GW7 SCREENSHOT (post-four transfers), resolved to public IDs; bank '+('entered manually' if CURRENT_BANK_TENTHS is not None else 'estimated from GW6 selling prices + current FPL prices')
    patched['pending_transfers_meta']={'out_ids':sorted(outgoing),'in_ids':sorted(incoming),
       'out_names':[str(now[x].get('web_name')) for x in sorted(outgoing)],
       'in_names':[str(now[x].get('web_name')) for x in sorted(incoming)],
       'estimated_bank':estimated_bank,'used_bank':bank,
       'bank_verified':CURRENT_BANK_TENTHS is not None,
       'free_transfers_original':SAVED_FTS_BEFORE_GW7_TRANSFERS,
       'transfers_done':GW7_TRANSFERS_ALREADY_MADE,
       'points_hits_pending':GW7_TRANSFER_COST_CONFIRMED_PTS}
    print('📸 Using GW7 screenshot roster, not stale public picks. Transfers IN:',', '.join(patched['pending_transfers_meta']['in_names']))
    print('📸 Transfers OUT:',', '.join(patched['pending_transfers_meta']['out_names']))
    print(f'💷 GW7 BANK: £{bank/10:.1f}m '+('confirmed manual override' if CURRENT_BANK_TENTHS is not None else '(ESTIMATED; verify in FPL Transfers)'))
    return patched

@dataclass
class Forecast:
    xp: float
    p_start: float
    p_play: float
    p60: float
    minutes: float
    cs_probability: float
    xg: float
    xa: float
    uncertainty: float
    source: str
    fixture: str
    fdr: float
    warnings: list[str]=field(default_factory=list)


@dataclass
class Route:
    ids: tuple[int,...]
    bank: int
    ft: int
    score: float
    steps: list[dict[str,Any]]


class DecisionEngine:
    def __init__(self,snapshot:dict[str,Any],input_dir:Path):
        self.snapshot=snapshot
        self.players={integer(p['id']):p for p in snapshot['bootstrap']['elements']}
        self.teams={integer(t['id']):t for t in snapshot['bootstrap']['teams']}
        self.events=determine_events(snapshot['bootstrap'])
        if not self.events:
            raise RuntimeError('No future FPL deadline found; cannot create a 6-GW forecast.')
        self.fixtures=defaultdict(list)
        self.completed=[]
        for f in snapshot['fixtures']:
            gw=integer(f.get('event'))
            if gw and integer(f.get('team_h')) and integer(f.get('team_a')):
                self.fixtures[(gw,integer(f['team_h']))].append((f,True))
                self.fixtures[(gw,integer(f['team_a']))].append((f,False))
                if f.get('finished') and f.get('team_h_score') is not None and f.get('team_a_score') is not None:
                    self.completed.append(f)
        self.news,self.odds,self.news_count,self.odds_count=self.load_overrides(input_dir)
        self.picks=snapshot['picks']['picks']
        self.owned=tuple(sorted(integer(p['element']) for p in self.picks))
        self.owned_prices={integer(p['element']):price_for_owned(p,self.players[integer(p['element'])]) for p in self.picks}
        self.bank=integer(snapshot['profile'].get('last_deadline_bank'),0) if BANK_OVERRIDE_TENTHS is None else BANK_OVERRIDE_TENTHS
        self.bank=max(0,self.bank)
        self.budget=self.bank + sum(self.owned_prices.values())
        self.ft=clamp(FREE_TRANSFERS_ASSUMED,0,5)
        self.ftrisk='After 4 GW7 moves, your Transfers screenshot confirms 0 free transfers and 0 transfer cost'
        self.squad_last_gw=integer(snapshot['picked_gw'])
        self.team_rates=self.rate_teams()
        self.stats={}
        self.forecasts={}
        self.membership_note=snapshot.get('squad_provenance','Last publicly locked squad, not current')
        self.risk_notes=[]
        self.create_player_stats()
        self.make_forecasts()
        self.sample_cache={}
        self.headshots: dict[int,str]={}
        self.club_logos: dict[int,str]={}

    @staticmethod
    def distinct_independent_sources(source: str) -> set[str]:
        # Two citations to the same publisher/article are not corroboration.
        identities=set()
        for raw in re.split(r'[;|\n]+',source):
            raw=raw.strip()
            try:
                parsed=urlparse(raw)
                if parsed.scheme!='https' or not parsed.hostname:continue
                host=parsed.hostname.casefold().removeprefix('www.')
                if host in ('x.com','twitter.com'):
                    account=parsed.path.strip('/').split('/')[0].casefold()
                    if account and account not in ('home','search','i'):
                        identities.add('x:'+account)
                else:
                    identities.add('publisher:'+host)
            except ValueError:
                continue
        return identities

    def load_overrides(self,folder:Path):
        now=datetime.now(UTC)
        news, odds={},{}
        news_path=folder/'news_overrides.csv'
        odds_path=folder/'odds_overrides.csv'
        for path,kind in ((news_path,'news'),(odds_path,'odds')):
            if not path.exists(): continue
            with path.open('r',encoding='utf-8-sig',newline='') as stream:
                for row in csv.DictReader(stream):
                    observed=iso_time(row.get('observed_utc'))
                    if observed is None or not (now-timedelta(hours=48)<=observed<=now+timedelta(minutes=10)):
                        continue
                    source=row.get('source','').strip()
                    if not source or row.get('verified','').lower() not in ('yes','true','1'):
                        continue
                    if kind=='news':
                        pid=integer(row.get('player_id'))
                        pstart=num(row.get('start_probability'),-1)
                        if pid>0 and 0<=pstart<=1 and len(self.distinct_independent_sources(source))>=2:
                            news[pid]={'p_start':pstart,'minutes_if_start':clamp(num(row.get('minutes_if_started'),78),45,95),'source':source,'observed':observed.isoformat(),'note':row.get('note','')}
                    else:
                        gw,team=integer(row.get('gw')),integer(row.get('team_id'))
                        cs=num(row.get('clean_sheet_probability'),-1)
                        lam=num(row.get('team_xg'),-1)
                        if gw>0 and team>0 and (0<=cs<=1 or lam>=0):
                            odds[(gw,team)]={'cs':cs if cs>=0 else None,'lambda':lam if lam>=0 else None,'source':source,'observed':observed.isoformat()}
        return news,odds,len(news),len(odds)

    def rate_teams(self):
        rates={}
        for tid in self.teams:
            matches=[]
            for f in self.completed:
                ishome=integer(f['team_h'])==tid
                if ishome or integer(f['team_a'])==tid:
                    gf=integer(f['team_h_score'] if ishome else f['team_a_score'])
                    ga=integer(f['team_a_score'] if ishome else f['team_h_score'])
                    matches.append((iso_time(f.get('kickoff_time')) or datetime.min.replace(tzinfo=UTC),gf,ga))
            matches.sort(key=lambda x:x[0],reverse=True)
            matches=matches[:10]
            weighted=[max(.35, 1-i*.07) for i in range(len(matches))]
            weight=sum(weighted)
            goals_for=sum(row[1]*w for row,w in zip(matches,weighted))
            goals_against=sum(row[2]*w for row,w in zip(matches,weighted))
            # Shrink early-season results toward league-average rather than fitting six noisy games.
            rates[tid]=((goals_for+1.42*7)/(weight+7),(goals_against+1.42*7)/(weight+7))
        return rates

    def fixture_rates(self, f:dict[str,Any]):
        home,away=integer(f['team_h']),integer(f['team_a'])
        h_for,h_against=self.team_rates[home]
        a_for,a_against=self.team_rates[away]
        hteam,ateam=self.teams[home],self.teams[away]
        # FPL team strengths are a supplementary modest prior; not bookmaker odds.
        strength_h=clamp(num(hteam.get('strength_attack_home'),1100)/max(1,num(ateam.get('strength_defence_away'),1100)),.75,1.30)
        strength_a=clamp(num(ateam.get('strength_attack_away'),1100)/max(1,num(hteam.get('strength_defence_home'),1100)),.75,1.30)
        home_xg=clamp(1.43*(h_for/1.42)**.50*(a_against/1.42)**.50*strength_h**.25, .30, 3.9)
        away_xg=clamp(1.22*(a_for/1.42)**.50*(h_against/1.42)**.50*strength_a**.25, .25, 3.6)
        return home_xg,away_xg

    def create_player_stats(self):
        """Season-shrunk, exponentially aged rates and *separate* starter/cameo state.

        History is used only when present in the official event history; missing match
        logs are explicitly uncertain, never interpreted as zero playing probability.
        These priors are heuristics until replay calibration supplies measured weights.
        """
        for pid, player in self.players.items():
            club = integer(player['team'])
            games = max(1, len({integer(f.get('id')) for f in self.completed
                               if club in (integer(f.get('team_h')), integer(f.get('team_a')))}))
            season_min = num(player.get('minutes'))
            season_starts = num(player.get('starts'), season_min / 82)
            summary = self.snapshot['summaries'].get(str(pid), {})
            rows = [r for r in (summary.get('history', []) if isinstance(summary, dict) else [])
                    if isinstance(r, dict)]
            rows.sort(key=lambda r: (integer(r.get('round')), integer(r.get('fixture'))))
            recent = list(reversed(rows[-10:]))
            # Half-life ~3.3 appearances, including 0-minute appearances as bench evidence.
            weights = np.asarray([0.81**i for i in range(len(recent))], dtype=float)
            minutes = np.asarray([clamp(num(r.get('minutes')), 0, 120) for r in recent])
            recent_starts = minutes >= 55
            wsum = float(weights.sum())
            season_pstart = clamp((season_starts + 1.7)/(games + 3.2), .04, .97)
            rec_start = (float((weights*recent_starts).sum()) + 2.5*season_pstart)/(wsum + 2.5)
            # Small sample -> season/position prior dominates; sample n_eff is explicit.
            n_eff = float(wsum**2/max(1e-8, (weights**2).sum())) if len(recent) else 0
            pstart = clamp((rec_start*wsum + season_pstart*4)/(wsum+4), .025, .97)
            start_samples = [(float(m),float(w)) for m,w in zip(minutes,weights) if m>=55]
            sub_samples = [(float(m),float(w)) for m,w in zip(minutes,weights) if 0<m<55]
            weighted = lambda s,d: float(sum(m*w for m,w in s)/max(1e-8,sum(w for _,w in s))) if s else d
            start_minutes = clamp(weighted(start_samples,clamp(season_min/max(1,season_starts),62,90)), 57, 93)
            bench_minutes = clamp(weighted(sub_samples,20),8,44)
            bench_exposure = float(sum(w for w,m in zip(weights,minutes) if m<55))
            bench_appear = float(sum(w for w,m in zip(weights,minutes) if 0<m<55))
            p_sub_given_bench = clamp((bench_appear+0.25)/(bench_exposure+1.5), .045,.75)
            p60 = clamp((sum(w for w,m in zip(weights, minutes) if m>=60)+.8)/
                        (sum(w for w,m in zip(weights,minutes) if m>=55)+1.2), .35,.98)
            # Empirical Bayes: recent xG/xA per90 is shrunk by ~900 prior minutes.
            # Very short cameos carry low exposure naturally through minutes/90.
            prior_minutes = max(360, season_min)
            def rate(field: str, cap: float) -> float:
                season_total = num(player.get(field))
                season90 = 90*season_total/max(360,season_min)
                wmins = sum(w*num(r.get('minutes')) for r,w in zip(recent,weights))
                wrate = 90*sum(w*num(r.get(field)) for r,w in zip(recent,weights))/max(1,wmins)
                credibility = wmins/(wmins+900)
                return clamp((1-credibility)*season90+credibility*wrate,0,cap)
            xg90,xa90 = rate('expected_goals', 2.4),rate('expected_assists', 1.8)
            if xg90==0 and xa90==0 and season_min<180:
                xg90={1:.005,2:.055,3:.15,4:.28}.get(integer(player['element_type']),.12)
                xa90={1:0,2:.025,3:.12,4:.09}.get(integer(player['element_type']),.08)
            dc_history=[(num(r.get('defensive_contribution')),num(r.get('minutes')),w)
                        for r,w in zip(recent,weights) if r.get('defensive_contribution') is not None]
            pos=integer(player['element_type']); thresh=10 if pos==2 else 12
            dc90=sum(v*w for v,m,w in dc_history)*90/max(90,sum(m*w for v,m,w in dc_history)) if dc_history else thresh*.55
            # Poisson threshold tail uses sample-size shrinkage, not a fixed 10% reward.
            dc90=(dc90*len(dc_history)+thresh*.55*5)/(len(dc_history)+5)
            self.stats[pid]=dict(p_start=pstart,p_sub_given_bench=p_sub_given_bench,
                start_minutes=start_minutes,bench_minutes=bench_minutes,p60=p60,xg90=xg90,xa90=xa90,
                bonus90=rate('bonus',1.5),saves90=rate('saves',9),yc90=rate('yellow_cards',1),
                dc90=dc90,dcprob=clamp(dc90/max(1,thresh)*.4,0,.7),
                history_available=bool(rows),recent_sample_effective=n_eff,starts=season_starts)

    def player_for_gw(self,pid:int,gw:int)->Forecast:
        player=self.players[pid]; st=self.stats[pid]
        games=self.fixtures.get((gw,integer(player['team'])),[])
        if not games:
            return Forecast(0,0,0,0,0,0,0,0,0,'FPL: blank/no confirmed fixture','BLANK',5,['No fixture scheduled'])
        warnings=[]
        pstart=st['p_start']; psub=st['p_sub_given_bench']
        availability=player.get('status','a')
        chance=player.get('chance_of_playing_next_round') if gw==self.events[0] else None
        if availability in ('i','s','u'):
            pstart*=.03 if availability in ('s','u') else .12
            psub*=.05 if availability in ('s','u') else .20
            warnings.append(f'Official status {availability}; verify injury/suspension.')
        elif chance is not None:
            factor=clamp(num(chance)/100,0,1)
            pstart*=factor; psub*=factor
            warnings.append(f'Official chance of playing {chance}%')
        elif availability=='d':
            pstart*=.60;psub*=.70
            warnings.append('Official doubtful flag')
        if pid in self.news and gw==self.events[0]:
            # Existing verified, fresh, two-source overrides are never bypassed.
            pstart=self.news[pid]['p_start']
            warnings.append(f"Verified override: {self.news[pid]['source']}")
        if not st['history_available']:warnings.append('Minutes based on season totals / priors; no match log')
        kickoff=[iso_time(f.get('kickoff_time')) for f,h in games]
        kickoff=[k for k in kickoff if k is not None]
        if len(kickoff)>=2 and (max(kickoff)-min(kickoff)).total_seconds()<5*86400:
            pstart*=.94
            warnings.append('Congested double Gameweek: rotation discount')
        startmin=self.news[pid]['minutes_if_start'] if gw==self.events[0] and pid in self.news else st['start_minutes']
        pstart=clamp(pstart,0,.995);psub=clamp(psub,0,1)
        cameo=(1-pstart)*psub
        pplay=clamp(pstart+cameo,0,.999)
        p60=st['p60']
        start_exposure=pstart*startmin; sub_exposure=cameo*st['bench_minutes']
        xpts=0.; cs_sum=0.;xgs=0.;xas=0.;fdrs=[];labels=[]
        pos=integer(player['element_type'])
        # Published ordering can lag changing set-piece duties: small, bounded
        # role prior decays toward zero over ~5 GWs; never assert the role as fact.
        offset=max(0,self.events.index(gw)) if gw in self.events else 0
        role_decay=math.exp(-offset/4.0)
        penalty=integer(player.get('penalties_order'))
        corners=integer(player.get('corners_and_indirect_freekicks_order'))
        fk=integer(player.get('direct_freekicks_order'))
        goal_role=(.06 if penalty==1 else .02 if penalty==2 else 0)*role_decay
        assist_role=((.06 if corners==1 else .02 if corners==2 else 0)+
                     (.025 if fk==1 else .007 if fk==2 else 0))*role_decay
        for fixture,home in games:
            home_xg,away_xg=self.fixture_rates(fixture)
            attack=home_xg if home else away_xg
            conceded=away_xg if home else home_xg
            odds=self.odds.get((gw,integer(player['team'])))
            cs=math.exp(-conceded)
            if odds:
                if odds['cs'] is not None:cs=clamp(odds['cs'],0,1)
                if odds['lambda'] is not None:attack=odds['lambda']
            cs_sum+=cs
            opp=integer(fixture['team_a'] if home else fixture['team_h'])
            label=self.teams[opp].get('short_name',self.teams[opp].get('name','?'))
            labels.append(f'{label} ({"H" if home else "A"})')
            fdrs.append(integer(fixture.get('team_h_difficulty' if home else 'team_a_difficulty'),3))
            minutes_factor=(start_exposure+sub_exposure)/90.
            attack_factor=clamp(attack/1.42,.55,1.95)
            xg=st['xg90']*minutes_factor*attack_factor*(1+goal_role)
            xa=st['xa90']*minutes_factor*attack_factor*(1+assist_role)
            xgs+=xg;xas+=xa
            appearance=pstart*(1+p60)+cameo
            cs_pts=(4 if pos in (1,2) else 1 if pos==3 else 0)*pstart*p60*cs
            goal_pts={1:10,2:6,3:5,4:4}.get(pos,4)*xg
            assist_pts=3*xa
            # Opponent xG drives clean sheet and likelihood of 2+ goals conceded.
            concede_2plus=max(0,conceded/2-(1-math.exp(-2*conceded))/4)
            conceded_pts=-concede_2plus*pstart*p60 if pos in (1,2) else 0
            saves=st['saves90']*minutes_factor/3 if pos==1 else 0
            defensive=0.
            if pos in (2,3,4):
                threshold=10 if pos==2 else 12
                # More opposition possession/pressure gives defenders more opportunities.
                intensity=clamp(.78+.23*conceded,.60,1.55)
                expected_dc=st['dc90']*startmin/90*intensity
                tail=1-sum(math.exp(-expected_dc)*expected_dc**k/math.factorial(k) for k in range(threshold))
                defensive=2*pstart*clamp(tail,0,1)
            bonus=st['bonus90']*minutes_factor*.75
            cards=-st['yc90']*minutes_factor
            xpts+=appearance+cs_pts+goal_pts+assist_pts+conceded_pts+saves+defensive+bonus+cards
        uncertainty=clamp(1.5+.40*xpts+3.2*pstart*(1-pstart)+2*(1-pplay)+
                          1.6/math.sqrt(1+st['recent_sample_effective']),2.3,9.5)
        source='Bookmaker CSV' if (gw,integer(player['team'])) in self.odds else 'Official FPL + shrinkage model'
        return Forecast(xp=round(max(0,xpts),3),p_start=round(pstart,4),p_play=round(pplay,4),
            p60=p60,minutes=round((start_exposure+sub_exposure)*len(games),1),
            cs_probability=cs_sum/len(games),xg=xgs,xa=xas,uncertainty=uncertainty,source=source,
            fixture=' + '.join(labels),fdr=sum(fdrs)/len(fdrs),warnings=warnings)

    def make_forecasts(self):
        for gw in self.events:
            for pid in self.players:
                self.forecasts[(pid,gw)]=self.player_for_gw(pid,gw)

    def xp(self,pid:int,gw:int)->float:
        return self.forecasts[(pid,gw)].xp

    def lineup(self,ids:tuple[int,...],gw:int)->dict[str,Any]:
        posgroups={pos:sorted([pid for pid in ids if integer(self.players[pid]['element_type'])==pos],key=lambda pid:self.xp(pid,gw),reverse=True) for pos in POS_NAME}
        if any(len(posgroups[p])!=POSITION_COUNTS[p] for p in POSITION_COUNTS):
            raise ValueError('Illegal squad position counts')
        keeper=posgroups[1][0]
        best=-1e10;selected=[];shape=(0,0,0)
        for nd in (3,4,5):
            for nm in range(2,6):
                nf=10-nd-nm
                if not 1<=nf<=3:continue
                row=[keeper]+posgroups[2][:nd]+posgroups[3][:nm]+posgroups[4][:nf]
                total=sum(self.xp(pid,gw) for pid in row)
                if total>best:
                    best=total;selected=row;shape=(nd,nm,nf)
        captain=max(selected,key=lambda pid:self.xp(pid,gw))
        second=max((pid for pid in selected if pid!=captain),key=lambda pid:self.xp(pid,gw))
        bench_outfield=sorted((pid for pid in ids if pid not in selected and integer(self.players[pid]['element_type'])!=1),key=lambda pid:self.xp(pid,gw),reverse=True)
        reserve_gk=posgroups[1][1]
        return dict(xi=selected,bench=[reserve_gk]+bench_outfield,captain=captain,vice=second,formation=f'{shape[0]}-{shape[1]}-{shape[2]}',expected=round(best+self.xp(captain,gw),2))

    def valid_squad(self,ids:tuple[int,...])->bool:
        if len(ids)!=15 or len(set(ids))!=15:return False
        if any(sum(integer(self.players[pid]['element_type'])==pos for pid in ids)!=count for pos,count in POSITION_COUNTS.items()):return False
        return all(count<=3 for count in Counter(integer(self.players[pid]['team']) for pid in ids).values())

    def squad_value(self,ids:tuple[int,...],bank:int)->int:
        return bank+sum(self.owned_prices.get(pid,integer(self.players[pid]['now_cost'])) for pid in ids)

    def squad_score(self,ids:tuple[int,...],start_index:int=0)->float:
        return sum((.93**idx)*self.lineup(ids,gw)['expected'] for idx,gw in enumerate(self.events[start_index:],start=start_index))

    def transfer_candidates(self,route:Route,gw:int)->list[tuple[int,int,int,float]]:
        current=set(route.ids)
        clubs=Counter(integer(self.players[pid]['team']) for pid in route.ids)
        recent=self.events.index(gw)
        scoring=lambda pid:sum(.90**offset*self.xp(pid,event) for offset,event in enumerate(self.events[recent:]))
        outgoing=sorted(route.ids,key=scoring)[:7]
        pool_by_position={pos:[] for pos in POS_NAME}
        for pid,player in self.players.items():
            if pid in current or player.get('status') in ('s','u'):continue
            pos=integer(player['element_type'])
            if pos not in pool_by_position:continue
            pool_by_position[pos].append(pid)
        for pos in pool_by_position:
            pool_by_position[pos].sort(key=scoring,reverse=True)
            pool_by_position[pos]=pool_by_position[pos][:20]
        moves=[]
        for outgoing_id in outgoing:
            sold=self.owned_prices.get(outgoing_id,integer(self.players[outgoing_id]['now_cost']))
            # An acquired player must later sell at purchase price at best, adjusted conservatively.
            for incoming_id in pool_by_position[integer(self.players[outgoing_id]['element_type'])]:
                newcost=integer(self.players[incoming_id]['now_cost'])
                if newcost>route.bank+sold:continue
                team=integer(self.players[incoming_id]['team'])
                if clubs[team]>=3 and team!=integer(self.players[outgoing_id]['team']):continue
                new_ids=tuple(sorted((set(route.ids)-{outgoing_id})|{incoming_id}))
                if not self.valid_squad(new_ids):continue
                delta=scoring(incoming_id)-scoring(outgoing_id)
                if delta<=-.1:continue
                moves.append((outgoing_id,incoming_id,route.bank+sold-newcost,delta))
        return sorted(moves,key=lambda m:m[3],reverse=True)[:14]

    def plan(self)->Route:
        states=[Route(self.owned,self.bank,integer(self.ft),0.0,[])]
        for idx,gw in enumerate(self.events):
            pool=[]
            for route in states:
                baseline=self.lineup(route.ids,gw)['expected']
                pool.append(Route(route.ids,route.bank,min(5,route.ft+1),route.score+(.93**idx)*baseline,route.steps+[dict(gw=gw,action='ROLL / HOLD',out=None,into=None,hit=0,xp=baseline,free_transfers_before=route.ft,bank=route.bank,ids=route.ids)]))
                for sold,bought,newbank,delta in self.transfer_candidates(route,gw):
                    ids=tuple(sorted((set(route.ids)-{sold})|{bought}))
                    projected=self.lineup(ids,gw)['expected']
                    transfer_cost=0 if route.ft>=1 else 4
                    if transfer_cost>0 and delta<5.0:continue
                    new_ft=min(5,max(0,route.ft-1)+1)
                    newscore=route.score+(.93**idx)*(projected-transfer_cost-1.0)  # stability cost discourages low-value churn
                    pool.append(Route(ids,newbank,new_ft,newscore,route.steps+[dict(gw=gw,action='TRANSFER',out=sold,into=bought,hit=transfer_cost,xp=projected,free_transfers_before=route.ft,bank=newbank,ids=ids)]))
            pool.sort(key=lambda x:x.score+(.93**(idx+1))*self.squad_score(x.ids,idx+1)*.10,reverse=True)
            # Different paths may have identical squads but different FT counts and banks.
            seen=set();pruned=[]
            for route in pool:
                key=(route.ids,route.ft,route.bank)
                if key not in seen:
                    seen.add(key);pruned.append(route)
                if len(pruned)>=BEAM_WIDTH:break
            states=pruned
            if not states:raise RuntimeError('No valid six-GW route')
        return max(states,key=lambda s:s.score)

    def optimize_squad(self,goal:str, gw:int|None=None)->dict[str,Any]:
        gw=gw or self.events[0]
        if goal not in ('wildcard','freehit','benchboost'):
            raise ValueError(f'Unknown squad optimization goal: {goal}')
        weeks=[event for event in self.events if event >= gw] if goal=='wildcard' else [gw]
        h=len(weeks)
        risk_penalty=0.10
        score=lambda pid:sum((.94**t)*max(0,self.xp(pid,event)-risk_penalty*self.forecasts[(pid,event)].uncertainty) for t,event in enumerate(weeks))
        eligible=[p for p in self.players.values() if integer(p.get('element_type')) in POSITION_COUNTS and (p.get('status') not in ('u','s') or integer(p['id']) in self.owned)]
        pool=set(self.owned)
        for pos in POSITION_COUNTS:
            group=[integer(p['id']) for p in eligible if integer(p['element_type'])==pos]
            group.sort(key=score,reverse=True)
            pool.update(group[:(12 if pos==1 else 22 if pos in (2,3) else 18)])
            group.sort(key=lambda pid:integer(self.players[pid]['now_cost']))
            pool.update(group[:(5 if pos==1 else 8)])
        ids=sorted(pool)
        n=len(ids)
        budget=self.budget
        costs=np.asarray([self.owned_prices.get(pid,integer(self.players[pid]['now_cost'])) for pid in ids],dtype=float)
        if budget<=0:raise RuntimeError('Invalid total selling value. Check current bank before optimization.')
        # Binary x(player) = own; y(player,week) = start; c(player,week) = captain.
        ix=lambda j:j
        iy=lambda week,j:n+week*n+j
        ic=lambda week,j:n+h*n+week*n+j
        count=n*(1+2*h)
        objective=np.zeros(count)
        for j,pid in enumerate(ids):
            objective[ix(j)]=-.12*score(pid)  # Modest bench strength, never equal to starter points.
            for week, event in enumerate(weeks):
                f=self.forecasts[(pid,event)]
                projected=max(0,f.xp-risk_penalty*f.uncertainty)
                objective[iy(week,j)]=-(.94**week)*projected
                objective[ic(week,j)]=-(.94**week)*projected
        constraints=[]
        def add(values:dict[int,float], minimum:float, maximum:float)->None:
            constraints.append((values,minimum,maximum))
        add({ix(j):1.0 for j in range(n)},15,15)
        for pos,qty in POSITION_COUNTS.items():
            add({ix(j):1.0 for j,pid in enumerate(ids) if integer(self.players[pid]['element_type'])==pos},qty,qty)
        for team_id in self.teams:
            keys={ix(j):1.0 for j,pid in enumerate(ids) if integer(self.players[pid]['team'])==team_id}
            if keys:add(keys,0,3)
        add({ix(j):costs[j] for j in range(n)},0,float(budget))
        for week in range(h):
            add({iy(week,j):1 for j in range(n)},11,11)
            add({ic(week,j):1 for j in range(n)},1,1)
            for pos,low,high in ((1,1,1),(2,3,5),(3,2,5),(4,1,3)):
                keys={iy(week,j):1 for j,pid in enumerate(ids) if integer(self.players[pid]['element_type'])==pos}
                add(keys,low,high)
            for j in range(n):
                add({iy(week,j):1,ix(j):-1},-np.inf,0)
                add({ic(week,j):1,iy(week,j):-1},-np.inf,0)
        matrix=lil_matrix((len(constraints),count),dtype=float)
        lower=np.zeros(len(constraints));upper=np.zeros(len(constraints))
        for index,(items,low,high) in enumerate(constraints):
            for col,value in items.items():matrix[index,col]=value
            lower[index]=low;upper[index]=high
        answer=milp(objective,integrality=np.ones(count),bounds=Bounds(np.zeros(count),np.ones(count)),
            constraints=LinearConstraint(matrix.tocsr(),lower,upper),
            options={'time_limit':25.0,'mip_rel_gap':0.025})
        if answer.x is None:
            return dict(success=False,reason=str(answer.message),selected=self.owned)
        selected=tuple(sorted(ids[j] for j in range(n) if answer.x[ix(j)]>.5))
        spent=sum(self.owned_prices.get(pid,integer(self.players[pid]['now_cost'])) for pid in selected)
        if not self.valid_squad(selected) or spent>budget:
            return dict(success=False,reason='Result failed hard squad / budget validation',selected=self.owned)
        for week,event in enumerate(weeks):
            formation=[ids[j] for j in range(n) if answer.x[iy(week,j)]>.5]
            caps=[ids[j] for j in range(n) if answer.x[ic(week,j)]>.5]
            if len(formation)!=11 or len(caps)!=1 or caps[0] not in formation:
                return dict(success=False,reason='Optimizer lineup/captain proof failed.',selected=self.owned)
        return dict(success=True,exact_within_candidate_pool=answer.status==0,status=str(answer.message),
            selected=selected,remaining_bank=budget-spent,
            in_ids=sorted(set(selected)-set(self.owned)),out_ids=sorted(set(self.owned)-set(selected)),
            optimization='Joint legal XV + best XI/formation + captain for all six weeks (no mandatory player names).',
            has_haaland=any(_squashed_name(self.name(pid))=='haaland' for pid in selected))

    def chip_availability(self)->dict[str,bool]:
        gw=self.events[0]
        used=self.snapshot['history'].get('chips',[])
        half=1 if gw<=19 else 2
        within=lambda row:((1<=integer(row.get('event'))<=19) if half==1 else (20<=integer(row.get('event'))<=38))
        usednames={str(c.get('name')) for c in used if within(c)}
        # The API uses 'wildcard', 'freehit', '3xc', 'bboost'.
        eligibility={name:code not in usednames for name,code in [('wildcard','wildcard'),('freehit','freehit'),('triplecaptain','3xc'),('benchboost','bboost')]}
        if gw==20 and any(c.get('name')=='freehit' and integer(c.get('event'))==19 for c in used):
            eligibility['freehit']=False  # consecutive Free Hits are prohibited across halves
        return eligibility

    def simulate_points(self,pid:int,gw:int,n:int)->tuple[np.ndarray,np.ndarray]:
        key=(pid,gw,n)
        if key in self.sample_cache:return self.sample_cache[key]
        forecast=self.forecasts[(pid,gw)]
        rng=np.random.default_rng(11_900 + pid*700 + gw*179)
        if forecast.xp<=0:
            arr=np.zeros(n);played=np.zeros(n,dtype=bool)
        else:
            start=rng.random(n)<forecast.p_start
            cameo=(~start)&(rng.random(n)<clamp((forecast.p_play-forecast.p_start)/max(.0001,1-forecast.p_start),0,1))
            played=start|cameo
            mean_cond=forecast.xp/max(.05,forecast.p_start+(.33*(forecast.p_play-forecast.p_start)))
            shape=clamp((mean_cond/max(2.0,forecast.uncertainty))**2,.85,3)
            arr=rng.gamma(shape,mean_cond/shape,size=n)*start + rng.gamma(1.2,mean_cond*.26/1.2,size=n)*cameo
            arr=np.clip(arr,0,35)
        self.sample_cache[key]=(arr,played)
        return arr,played

    def simulate_lineup(self,ids:tuple[int,...],gw:int,chip:str|None=None,n:int=SIMULATIONS)->np.ndarray:
        lineup=self.lineup(ids,gw)
        samples={pid:self.simulate_points(pid,gw,n) for pid in ids}
        total=np.zeros(n)
        starting=set(lineup['xi'])
        for pid in starting: total+=samples[pid][0]
        if chip=='benchboost':
            for pid in lineup['bench']: total+=samples[pid][0]
        else:
            gk=lineup['xi'][0]
            missing_gk=~samples[gk][1]
            total+=samples[lineup['bench'][0]][0]*missing_gk
            # Bench automations obey 3 DEF, 2 MID, 1 FWD minimums for every simulated outcome.
            initial_outfield=[pid for pid in lineup['xi'] if integer(self.players[pid]['element_type'])!=1]
            substitutes=lineup['bench'][1:]
            for iteration in range(n):
                taken=[pid for pid in initial_outfield if samples[pid][1][iteration]]
                count=Counter(integer(self.players[pid]['element_type']) for pid in taken)
                available=[pid for pid in substitutes if samples[pid][1][iteration]]
                needed=10-len(taken)
                for slot in range(needed):
                    choice=None
                    for candidate in available:
                        pos=integer(self.players[candidate]['element_type'])
                        proposed=count.copy();proposed[pos]+=1
                        remaining=needed-slot-1
                        minimum=sum(max(0,requirements-proposed[key]) for key,requirements in ((2,3),(3,2),(4,1)))
                        if minimum<=remaining:
                            choice=candidate;break
                    if choice is None:break
                    total[iteration]+=samples[choice][0][iteration]
                    count[integer(self.players[choice]['element_type'])]+=1
                    available.remove(choice)
        captain=lineup['captain']; vice=lineup['vice']
        cap_scores,cap_play=samples[captain]
        vice_scores,vice_play=samples[vice]
        cap_mult=2 if chip=='triplecaptain' else 1
        total+=cap_mult*np.where(cap_play,cap_scores,np.where(vice_play,vice_scores,0))
        return total

    def chip_solver(self,route:Route)->dict[str,Any]:
        first=self.events[0]
        availability=self.chip_availability()
        pending=self.snapshot.get('pending_transfers_meta',{})
        hit_points=integer(pending.get('points_hits_pending',0))
        n=SIMULATIONS
        base=np.zeros(n)
        for step in route.steps:
            base+=self.simulate_lineup(step['ids'],step['gw'],n=n)-step['hit']
        base-=hit_points
        scenarios={key:dict(available=availability[key],gw=first,mean_gain=None,p10_gain=None,p90_gain=None,
                           hold_reason=('Used this half' if not availability[key] else ''),in_ids=[],out_ids=[])
                   for key in ('wildcard','freehit','triplecaptain','benchboost')}
        recommended='HOLD WILDCARD'
        optimal=None
        if availability['wildcard']:
            optimal=self.optimize_squad('wildcard')
            if optimal['success']:
                wc=np.zeros(n)
                for event in self.events:
                    wc+=self.simulate_lineup(optimal['selected'],event,n=n)
                delta=wc-base
                mean=float(np.mean(delta));positive=float(np.mean(delta>0))
                scenarios['wildcard'].update(mean_gain=round(mean,2),p10_gain=round(float(np.quantile(delta,.10)),2),
                    p90_gain=round(float(np.quantile(delta,.90)),2),positive_probability=round(positive,3),
                    horizon_gain=round(mean,2),in_ids=optimal['in_ids'],out_ids=optimal['out_ids'],
                    solver_status=optimal['status'],exact_within_candidate_pool=optimal['exact_within_candidate_pool'],
                    selected_ids=list(optimal['selected']),has_haaland=optimal['has_haaland'])
                if mean>WILDCARD_OPPORTUNITY_COST_PTS and positive>.55:
                    recommended='PLAY WILDCARD (MODEL LEAN)'
                else:
                    recommended='HOLD WILDCARD (MODEL LEAN)'
            else:
                scenarios['wildcard']['hold_reason']=optimal.get('reason','Optimizer failed')
        return dict(availability=availability,baseline_gw=round(float(np.mean(self.simulate_lineup(self.owned,first,n=n))),2),
            normal_6gw=round(float(np.mean(base)),2),scenarios=scenarios,recommendation=recommended,
            simulation_count=n,pending_hits=hit_points,
            wildcard_option_value_heuristic=WILDCARD_OPPORTUNITY_COST_PTS,
            decided_by='Simulated six-GW expected points, risk probability, current roster & automatic 15-man/captain optimizer. Haaland is not forced.',
            optimal=optimal)

    def player(self,pid:int)->dict[str,Any]:return self.players[pid]
    def name(self,pid:int)->str:return str(self.player(pid).get('web_name') or self.player(pid).get('second_name') or pid)
    def team_name(self,pid:int)->str:
        return str(self.teams[integer(self.player(pid)['team'])].get('name','?'))


def embed_visual_assets(engine: DecisionEngine, destination: Path) -> None:
    if not ENABLE_REAL_HEADSHOTS:
        return
    store=destination/'Cache'/'VisualAssets'
    store.mkdir(parents=True,exist_ok=True)
    tasks=[]
    for pid in engine.owned:
        code=integer(engine.player(pid).get('code'))
        if code:
            tasks.append(('player',pid,code,[f'https://resources.premierleague.com/premierleague/photos/players/110x140/p{code}.png']))
    for team_id in sorted({integer(engine.player(pid)['team']) for pid in engine.owned}):
        code=integer(engine.teams[team_id].get('code'))
        if code:
            tasks.append(('club',team_id,code,[f'https://resources.premierleague.com/premierleague/badges/t{code}.png']))
    def load_asset(item:tuple[str,int,int,list[str]]) ->tuple[str,int,str|None]:
        kind,item_id,code,urls=item
        path=store/f'{kind}_{code}.png'
        content=None
        if path.exists() and 16<len_bytes_safe(path)<350000:
            try:content=path.read_bytes()
            except OSError:pass
        if content is None:
            for url in urls:
                try:
                    response=requests.get(url,timeout=6,headers={'User-Agent':'Mozilla/5.0'})
                    if response.status_code==200 and response.headers.get('content-type','').startswith('image/') and 16<len(response.content)<350000:
                        from PIL import Image
                        with Image.open(io.BytesIO(response.content)) as test:
                            test.verify()
                        content=response.content
                        path.write_bytes(content)
                        break
                except (OSError,ValueError,requests.RequestException):
                    continue
        return kind,item_id,'data:image/png;base64,'+base64.b64encode(content).decode('ascii') if content else None
    def len_bytes_safe(path:Path)->int:
        try:return path.stat().st_size
        except OSError:return 0
    with ThreadPoolExecutor(max_workers=7) as executor:
        for done in as_completed([executor.submit(load_asset,item) for item in tasks]):
            try: kind,item_id,uri=done.result()
            except (ValueError,OSError):continue
            if uri:
                if kind=='player':engine.headshots[item_id]=uri
                else:engine.club_logos[item_id]=uri
    print(f'🖼️ Official player headshots: {len(engine.headshots)}/{len(engine.owned)}; official club badges: {len(engine.club_logos)}. Teal avatar fallback elsewhere.')


def data_news_status(snapshot:dict[str,Any]) -> list[dict[str,Any]]:
    return snapshot.get('news_radar',{}).get('feed_status',[])

def make_result(engine:DecisionEngine)->dict[str,Any]:
    plan=engine.plan()
    chip=engine.chip_solver(plan)
    selected_chip=chip['scenarios']['wildcard']
    preferred_ids=tuple(selected_chip.get('selected_ids',[])) if chip['recommendation'].startswith('PLAY WILDCARD') else engine.owned
    views={gw:engine.lineup(preferred_ids,gw) for gw in engine.events}

    roster=[]
    for pid in engine.owned:
        p=engine.player(pid)
        pview={'id':pid,'name':engine.name(pid),'team':engine.team_name(pid),'pos':POS_NAME[integer(p['element_type'])], 'price':integer(p['now_cost'])/10,'status':p.get('status'), 'official_news':p.get('news') or '', 'official_chance':p.get('chance_of_playing_next_round'),'owned_selling_price':engine.owned_prices[pid]/10,'projections':[]}
        for gw in engine.events:
            f=engine.forecasts[(pid,gw)]
            pview['projections'].append(dict(gw=gw,points=round(f.xp,2),p_start=f.p_start,minutes=f.minutes,cs=round(f.cs_probability,3),fixture=f.fixture,fdr=f.fdr,warnings=f.warnings,source=f.source))
        roster.append(pview)
    rows=[]
    for gw in engine.events:
        step=next(item for item in plan.steps if item['gw']==gw)
        ids=plan.ids if gw==engine.events[-1] else None
        rows.append({**step,'out_name': ', '.join(engine.name(i) for i in step.get('out_ids', [step['out']] if step['out'] else [])), 'in_name': ', '.join(engine.name(i) for i in step.get('in_ids', [step['into']] if step['into'] else [])), 'captain':engine.name(engine.lineup(step['ids'],gw)['captain']), 'baseline_xp':views[gw]['expected']})
    result={'generated_utc':datetime.now(UTC).isoformat(),'team_id':ENTRY_ID,'team_name':engine.snapshot['profile']['name'],'overall_rank':engine.snapshot['profile'].get('summary_overall_rank'),'overall_points':engine.snapshot['profile'].get('summary_overall_points'),'last_published_gw':engine.squad_last_gw,'events':engine.events,'bank':engine.bank/10,'free_transfers_assumed':integer(engine.ft),'free_transfers_verified':False,'ownership_current_verified':False,'picks_stale_warning':engine.membership_note,'data_sources':{'official_fpl':True,'player_summaries':len(engine.snapshot['summaries']),'verified_news_overrides':engine.news_count,'odds_overrides':engine.odds_count,'clean_sheet_odds':('Imported bookmaker CSV' if engine.odds_count else 'Poisson estimates, NOT bookmaker odds'),'top10k_ownership':'Unavailable: global ownership only','news_feeds':data_news_status(engine.snapshot),'polymarket':engine.snapshot.get('markets',{}).get('available',False)},'primary_target':PRIMARY_RANK_TARGET,'backup_target':BACKUP_RANK_TARGET,'rank_benchmarks':engine.snapshot.get('rank_benchmarks',{}),'news_radar':engine.snapshot.get('news_radar',{}),'markets':engine.snapshot.get('markets',{}),'roster':roster,'lineups':{str(gw):views[gw] for gw in engine.events},'plan':rows,'plan_total_projected':round(plan.score,2),'chip':chip,
        'recommended_squad_ids':list(preferred_ids),'pending_transfers':engine.snapshot.get('pending_transfers_meta',{}),
        'bank_verified':CURRENT_BANK_TENTHS is not None,'raw_snapshot':engine.snapshot}
    return result

AVATAR = '''<svg viewBox="0 0 80 80" xmlns="http://www.w3.org/2000/svg" aria-hidden="true"><circle cx="40" cy="26" r="15" fill="#36ebda"/><path d="M7 76q1-30 33-31 32 1 33 31" fill="#13a99f"/><circle cx="40" cy="26" r="14" fill="#074857" opacity=".45"/></svg>'''
LOGO = '''<svg viewBox="0 0 72 80" xmlns="http://www.w3.org/2000/svg" aria-label="FPLVORTEX badge"><path d="M36 5 65 17 61 50 36 72 11 50 7 17Z" fill="#043443" stroke="#1ce7d1" stroke-width="4"/><circle cx="36" cy="38" r="17" fill="#1ce7d1"/><circle cx="36" cy="38" r="12" fill="#062337"/><text x="36" y="44" text-anchor="middle" font-size="14" fill="#fff" font-weight="800">VX</text><path d="m23 5 5-5 8 7 8-7 5 5" fill="none" stroke="#1ce7d1" stroke-width="4"/></svg>'''


def button(label: str, target: str) -> str:
    return f'<button class="button" type="button" data-go="{esc(target)}">{esc(label)} →</button>'


def badge(text: str, tone: str='teal') -> str:
    return f'<span class="badge {tone}">{esc(text)}</span>'


def row_player(engine:DecisionEngine,pid:int,gw:int) -> str:
    p=engine.player(pid); f=engine.forecasts[(pid,gw)]
    team=engine.teams[integer(p['team'])]
    abbrev=team.get('short_name') or (team.get('name','')[:3]).upper()
    news=(str(p.get('news') or '')[:160])
    warn=badge('Check news','warn') if f.warnings and (p.get('status') not in (None,'a') or pid in engine.news) else badge('Minutes ✓','teal')
    crest=(f'<img class="clubmark" src="{engine.club_logos[integer(p["team"])]}" alt="{esc(team.get("name",""))} badge"/>' if integer(p['team']) in engine.club_logos else f'<span class="clubmark" title="{esc(team.get("name",""))}">{esc(abbrev)}</span>')
    return f'''<tr><td><div class="player-cell">{crest}<div><b>{esc(engine.name(pid))}</b><small>{esc(team.get('name',''))} · {POS_NAME[integer(p['element_type'])]}</small></div></div></td><td>{esc(f.fixture)}</td><td><strong>{f.xp:.1f}</strong></td><td>{f.p_start*100:.0f}%</td><td>{f.minutes:.0f}</td><td>{f.cs_probability*100:.0f}%</td><td>{warn}</td></tr>'''


def player_tile(engine:DecisionEngine,pid:int,gw:int,role:str='') -> str:
    f=engine.forecasts[(pid,gw)]
    p=engine.player(pid);team=engine.teams[integer(p['team'])]
    abbrev=team.get('short_name') or team.get('name','')[:3].upper()
    alert=('<span class="player-alert">!</span>' if p.get('status') not in ('a',None) else '')
    portrait=(f'<img class="headshot" src="{engine.headshots[pid]}" alt="{esc(engine.name(pid))} official headshot" />' if pid in engine.headshots else AVATAR)
    club_crest=(f'<img class="club-chic" src="{engine.club_logos[integer(p["team"])]}" alt="{esc(team.get("name",""))} badge" />' if integer(p['team']) in engine.club_logos else f'<span class="club-chic">{esc(abbrev)}</span>')
    return f'''<div class="pitch-player" title="{esc(f.fixture)}; {f.xp:.2f} xPts; {f.p_start:.0%} projected start">
    <span class="player-portrait">{portrait}{club_crest}{alert}</span>
    <span class="pitch-name">{esc(engine.name(pid))[:16]}{' <em>('+role+')</em>' if role else ''}</span><span class="pitch-meta">{esc(f.fixture)} · {f.xp:.1f} xP</span></div>'''


def pitch_view(engine:DecisionEngine,gw:int,ids:tuple[int,...]|None=None) -> str:
    lineup=engine.lineup(ids if ids is not None else engine.owned,gw)
    positional={p:[pid for pid in lineup['xi'] if integer(engine.player(pid)['element_type'])==p] for p in POS_NAME}
    capt=lineup['captain'];vice=lineup['vice']
    rows=[]
    for pos in (1,2,3,4):
        tiles=''.join(player_tile(engine,pid,gw,'C' if pid==capt else 'VC' if pid==vice else '') for pid in positional[pos])
        rows.append(f'<div class="pitch-row pitch-row-{pos}">{tiles}</div>')
    bench=''.join(player_tile(engine,pid,gw,'GK' if i==0 else str(i)) for i,pid in enumerate(lineup['bench']))
    return f'''<div class="stadium"><div class="stadium-top">{LOGO}<div><b>FPLVORTEX</b><small>MY TEAM. SMARTER DECISIONS.</small></div>{badge('GW '+str(gw),'teal')}</div>
    <div class="pitch"><div class="pitch-centerline"></div><div class="pitch-circle"></div><div class="pitch-box pitch-box-top"></div><div class="pitch-box pitch-box-bottom"></div><div class="pitch-inner">{''.join(rows)}</div></div>
    <div class="bench-box"><b>SUBSTITUTES</b><div class="bench-grid">{bench}</div></div></div>'''


def metrics(result:dict[str,Any],engine:DecisionEngine) -> str:
    gw=result['events'][0]
    deadline=next((x.get('deadline_time') for x in engine.snapshot['bootstrap']['events'] if integer(x.get('id'))==gw),'Unknown')
    cards=[('Overall rank',f"#{integer(result['overall_rank']):,}"),('Total points',str(integer(result['overall_points']))),('Bank',f"£{result['bank']:.1f}m"),('FT left before WC',f"{result['free_transfers_assumed']} ⚠"),('Primary target','TOP 5K')]
    return '<div class="metrics">'+''.join(f'<div class="stat"><small>{esc(label)}</small><strong>{esc(value)}</strong></div>' for label,value in cards)+'</div>'+f'<div class="alert">Next deadline: <b>GW{gw}</b> — {esc(deadline)} (UTC). Latest locked squad: GW{result["last_published_gw"]}. Public picks might differ from your actual pre-deadline team. Free transfers assumed and unverified.</div>'


def render_html(result:dict[str,Any],engine:DecisionEngine,png:Path)->str:
    first=result['events'][0]
    preferred_ids=tuple(result['recommended_squad_ids'])
    wildcard_active=chip_active=result['chip']['recommendation'].startswith('PLAY WILDCARD')
    lineup=engine.lineup(preferred_ids,first)
    img='data:image/png;base64,'+base64.b64encode(png.read_bytes()).decode('ascii')
    cap=engine.name(lineup['captain']);vice=engine.name(lineup['vice'])
    player_table='<div class="table-scroll"><table><thead><tr><th>Player</th><th>Fixture</th><th>xPts</th><th>P(start)</th><th>Minutes</th><th>CS</th><th>News</th></tr></thead><tbody>'+''.join(row_player(engine,pid,first) for pid in engine.owned)+'</tbody></table></div>'
    rows=[]
    for step in result['plan']:
        move=f'{step["out_name"]} → {step["in_name"]}' if step['action']=='TRANSFER' else 'Roll / hold';color='up' if step['action']=='TRANSFER' else 'neutral'
        rows.append(f'''<tr><td><b>GW{step['gw']}</b></td><td>{badge(move,color)}</td><td>{esc(step['captain'])}</td><td>{step['xp']:.1f}</td><td>{step['hit']} pts</td><td>£{step['bank']/10:.1f}m</td><td>{'1-2 GWs tentative' if step['gw']>=first+2 else 'Review news before lock'}</td></tr>''')
    route_table='<div class="table-scroll"><table><thead><tr><th>GW</th><th>Suggested move</th><th>Captain</th><th>XI xPts</th><th>Hit</th><th>Bank after</th><th>Timing</th></tr></thead><tbody>'+''.join(rows)+'</tbody></table></div>'
    first_move=result['plan'][0]
    headline=('ACTIVATE WILDCARD & BUILD OPTIMAL XV' if wildcard_active else
        (f"{first_move['out_name']} → {first_move['in_name']}" if first_move['action']=='TRANSFER' else 'HOLD / ROLL TRANSFER'))
    wc_details=''
    wc_six_table=''
    if wildcard_active:
        option=result['chip']['scenarios']['wildcard']
        ins=', '.join(engine.name(pid) for pid in option['in_ids']) or 'No changes'
        outs=', '.join(engine.name(pid) for pid in option['out_ids']) or 'No changes'
        projected_budget=f"£{engine.budget/10:.1f}m selling-value budget"
        wc_details=f'''<div class="card"><h2>Automatic Wildcard recommendation — verified current squad</h2>
        <div class="big-call">PLAY WILDCARD • GW{first}</div>
        <p><b>Buy:</b> {esc(ins)}</p><p><b>Sell:</b> {esc(outs)}</p>
        <p>GW7 expected XI + captain <b>{lineup['expected']:.1f} xPts</b> ·
        Projected 6-GW advantage after current transfer hits <b>{option['mean_gain']:+.1f} pts</b> (model estimate).</p>
        <p>{esc(projected_budget)} · WC cash remaining £{(engine.budget - sum(engine.owned_prices.get(pid,integer(engine.players[pid]['now_cost'])) for pid in preferred_ids))/10:.1f}m.</p>
        <p class="mini">Prices/cash are UNVERIFIED unless exact current bank is supplied. The model does not force Haaland; it {'included' if option.get('has_haaland') else 'did not include'} him after evaluating the complete candidate pool.</p></div>'''
        captain_plan=[]
        for gw in result['events']:
            line=engine.lineup(preferred_ids,gw)
            captain_plan.append(f'''<tr><td>GW{gw}</td><td>{esc(engine.name(line['captain']))}</td><td>{esc(line['formation'])}</td>
                <td>{line['expected']:.1f}</td><td>{esc(', '.join(engine.name(pid) for pid in line['bench']))}</td></tr>''')
        wc_six_table='<div class="card"><h2>Recommended Wildcard squad: six-week XI and captaincy plan</h2><div class="table-scroll"><table><thead><tr><th>GW</th><th>Captain</th><th>Formation</th><th>Projected xPts</th><th>Bench</th></tr></thead><tbody>'+''.join(captain_plan)+'</tbody></table></div>'+pitch_view(engine,first,preferred_ids)+'</div>'

    top_owned=sorted(engine.owned,key=lambda pid:sum(engine.xp(pid,gw) for gw in engine.events),reverse=True)[:5]
    weak=sorted(engine.owned,key=lambda pid:sum(engine.xp(pid,gw) for gw in engine.events))[:4]
    risk_list=[]
    for pid in engine.owned:
        f=engine.forecasts[(pid,first)]
        if f.p_start<.70 or engine.player(pid).get('status') not in ('a',None):
            risk_list.append(f'<div class="risk-row"><b>{esc(engine.name(pid))}</b><small>{f.p_start*100:.0f}% projected starts • {esc(engine.player(pid).get("news") or "Review minutes / rotation")}</small></div>')
    if not risk_list:risk_list=['<p class="muted">No major official injury flags found. Check late team news anyway.</p>']
    cap_cards=''.join(f'<div class="small-card"><b>{esc(engine.name(pid))}</b><span>{engine.xp(pid,first):.1f} xPts · {engine.forecasts[(pid,first)].p_start*100:.0f}% start</span></div>' for pid in sorted(lineup['xi'],key=lambda pid:engine.xp(pid,first),reverse=True)[:5])
    chip=result['chip']
    chip_labels={'triplecaptain':'Triple Captain','benchboost':'Bench Boost','freehit':'Free Hit','wildcard':'Wildcard'}
    chip_cards=[]
    for key,title in chip_labels.items():
        scenario=chip['scenarios'][key]
        available=scenario['available']
        value='UNAVAILABLE' if not available else ('NOT SOLVED' if scenario['mean_gain'] is None else f"{scenario['mean_gain']:+.1f} estimated pts")
        extra=(f"Simulated P10 / P90: {scenario['p10_gain']:+.1f} / {scenario['p90_gain']:+.1f}. P(positive): {scenario['positive_probability']*100:.0f}%" if scenario['mean_gain'] is not None else scenario['hold_reason'])
        ins=', '.join(engine.name(pid) for pid in scenario['in_ids']) or '—'
        outs=', '.join(engine.name(pid) for pid in scenario['out_ids']) or '—'
        chip_cards.append(f'''<div class="chip-card {'disabled' if not available else ''}"><div class="split"><h3>{title}</h3>{badge('Eligible' if available else 'Used this half','teal' if available else 'warn')}</div><div class="chip-gain">{esc(value)}</div><p class="muted">{esc(extra)}</p><p class="mini">Potential IN: {esc(ins)}</p><p class="mini">Potential OUT: {esc(outs)}</p><button class="button" type="button" data-sim="{key}" {'disabled' if not available or scenario['mean_gain'] is None else ''}>Run {title} simulation</button></div>''')
    matrix='<div class="table-scroll"><table><thead><tr><th>Player</th>'+''.join(f'<th>GW{gw}</th>' for gw in result['events'])+'<th>6GW total</th></tr></thead><tbody>'
    for pid in sorted(engine.owned,key=lambda pid:sum(engine.xp(pid,gw) for gw in engine.events),reverse=True):
        matrix+=f'<tr><th>{esc(engine.name(pid))} <small>{POS_NAME[integer(engine.player(pid)["element_type"])]}</small></th>'
        for gw in engine.events:
            f=engine.forecasts[(pid,gw)]
            tone='good' if f.xp>=5 else 'poor' if f.xp<2.1 else 'mid'
            matrix+=f'<td class="{tone}"><b>{f.xp:.1f}</b><small>{esc(f.fixture)}</small></td>'
        matrix+=f'<td><b>{sum(engine.xp(pid,gw) for gw in engine.events):.1f}</b></td></tr>'
    matrix+='</tbody></table></div>'
    # The source is deliberately short JSON, not the raw API snapshot.
    browser_data={'scenario':{key:{a:opt.get(a) for a in ('mean_gain','p10_gain','p90_gain','positive_probability','available')} for key,opt in chip['scenarios'].items()}, 'count':chip['simulation_count']}
    script_data=json.dumps(browser_data,ensure_ascii=False,separators=(',',':')).replace('</','<\\/')
    links=[('overview','Overview'),('decisions','Decision Room'),('next','Next GW Plan'),('six','6-GW Plan'),('chips','Chip Simulator'),('news','Team News & Odds'),('audit','Data & Limits')]
    nav=''.join(f'<button class="tab" role="tab" type="button" data-tab="{key}" aria-selected="false">{name}</button>' for key,name in links)
    v=metrics(result,engine)
    unused=[key for key,yes in chip['availability'].items() if yes]
    warning='Only '+', '.join(chip_labels.get(k,k) for k in unused)+' available this half.' if unused else 'All chips from this half were used.'

    radar=result.get('news_radar',{})
    feeds=radar.get('feed_status',[])
    checked=int(radar.get('articles_checked',0))
    official_flags=radar.get('official_flags',[])
    source_report='<div class="small-card"><b>News feed coverage</b><span>'+esc(', '.join(f"{row['source']}: {'OK' if row['ok'] else 'unavailable'}" for row in feeds) or 'Not checked')+'</span></div>'
    source_report+=f'<div class="small-card"><b>Independent headlines screened</b><span>{checked}</span></div>'
    for item in official_flags:
        report=next((r for r in radar.get('headline_matches',[]) if r.get('player_id')==item['player_id']),None)
        state=badge('Official flag','warn') if item.get('status')!='a' else badge('FPL available','up')
        news_text=esc(item.get('news') or 'No official status warning')
        source_report+=f'<div class="mini-card" style="margin-bottom:8px"><b>{esc(item["name"])}</b> {state}<p>{news_text}</p>'
        if report:
            source_report+=f'<p class="mini">{int(report["independent_publishers"])} publisher(s) mention player in a headline. This does not verify a lineup.</p>'
            for article in report['articles']:
                source_report+=f'<p class="mini">◦ <a rel="noopener noreferrer" target="_blank" style="color:#45f1cd" href="{esc(article["url"])}">{esc(article["title"])}</a> · {esc(article["publisher"])} · {article["age_hours"]}h ago</p>'
        source_report+='</div>'
    markets=result.get('markets',{})
    market_rows=[]
    for market in markets.get('matches',[]):
        prob=market['probabilities']
        market_rows.append(f'<tr><td>{esc(market["fixture"])}</td><td>{prob["home"]:.0%}</td><td>{prob["draw"]:.0%}</td><td>{prob["away"]:.0%}</td><td><a href="{esc(market["url"])}" target="_blank" rel="noopener noreferrer" style="color:#45f1cd">Market</a></td></tr>')
    market_table=('<div class="table-scroll"><table><thead><tr><th>Fixture</th><th>Home</th><th>Draw</th><th>Away</th><th>Source</th></tr></thead><tbody>'+''.join(market_rows)+'</tbody></table></div>') if market_rows else '<p class="muted">No exact, liquid EPL match markets found or public Gamma feed unavailable. No odds probabilities were invented.</p>'
    rank_benchmarks=result.get('rank_benchmarks',{})
    gap_parts=[]
    for desired in (PRIMARY_RANK_TARGET,BACKUP_RANK_TARGET):
        data=rank_benchmarks.get(str(desired))
        if data: gap_parts.append(f'Rank ~{desired:,}: {data["approx_points"]} pts; gap ~{data["points_gap"]} pts')
    rank_box='<div class="card"><h2>Rank mission — primary 5K, backup 10K</h2><p>'+esc(' · '.join(gap_parts) if gap_parts else 'Live rank-target cutoffs not available from the sampled standings; no imaginary points gap.')+'</p><p class="muted">Relative target gap may change throughout matches and the season. Optimize actual expected returns, not arbitrary rank jumps.</p></div>'
    css='''
:root{color-scheme:dark;--bg:#061524;--panel:#0a2940;--panel2:#0d344a;--line:#205367;--teal:#16e6ce;--text:#eafcff;--muted:#9cbcca}*{box-sizing:border-box}body{margin:0;background:radial-gradient(circle at 55% 0,#0d3750,#061524 68%);font:15px/1.4 Inter,system-ui,Segoe UI,Arial,sans-serif;color:var(--text)}button{font:inherit;cursor:pointer}.wrap{width:min(1800px,98vw);margin:auto}.top{display:flex;align-items:center;justify-content:space-between;gap:20px;padding:12px 2px;position:sticky;top:0;z-index:30;background:#061928ef;backdrop-filter:blur(12px);border-bottom:1px solid #22495a}.identity{display:flex;align-items:center;gap:10px;min-width:200px}.identity svg{width:56px;height:63px}.identity b{font-size:24px;letter-spacing:.03em}.identity small,.mini{font-size:11px;color:var(--muted)}.tab-nav{display:flex;gap:4px;flex-wrap:wrap;background:#092638;border-radius:12px;border:1px solid #17465a;padding:4px}.tab{border:0;background:transparent;padding:13px 16px;color:#c8dfec;border-radius:9px;font-weight:650}.tab[aria-selected=true],.tab:hover{background:var(--teal);color:#002934}.page{display:none}.page.active{display:block}.metrics{display:grid;grid-template-columns:repeat(5,minmax(0,1fr));gap:12px;margin:14px 0}.stat{padding:15px 17px;border-radius:14px;background:#f0fcff;color:#073249;min-height:77px;display:grid;gap:6px}.stat small{color:#4d6a78}.stat strong{font-size:23px}.alert{background:#123749;border:1px solid #247184;color:#b9dfeb;padding:10px 16px;border-radius:10px;margin-bottom:14px}.grid{display:grid;grid-template-columns:minmax(0,1.95fr) minmax(310px,1fr);gap:15px}.card{border:1px solid #17485a;border-radius:15px;background:linear-gradient(125deg,#09293f,#082236);padding:18px;margin-bottom:14px;overflow:hidden}.card h2{margin:0 0 10px;font-size:23px}.card h3{margin:0 0 8px;font-size:18px}.muted{color:var(--muted)}p{margin:8px 0}.button{background:var(--teal);border:1px solid var(--teal);color:#053144;font-weight:800;padding:10px 14px;border-radius:9px;white-space:normal}.button:hover{filter:brightness(1.14)}.button:disabled{opacity:.4;cursor:not-allowed}.button.alt{background:#133e50;color:#d1f9ff;border:1px solid #32859a}.button-row{display:flex;gap:10px;flex-wrap:wrap;margin:14px 0}.badge{display:inline-block;padding:3px 9px;border-radius:7px;font-size:11px;font-weight:850;background:#064c4e;color:#23f1ca;border:1px solid #139a91}.badge.warn{background:#553726;color:#ffd38d;border-color:#826341}.badge.up{background:#075a47;color:#34f3a2}.badge.neutral{background:#17394a;color:#a8cde0}.table-scroll{overflow:auto}table{border-collapse:collapse;width:100%;min-width:570px;font-size:13px}th,td{border-bottom:1px solid #1b4758;text-align:left;padding:11px 9px}th{color:#aee9f0;background:#0b354a}td small,th small{display:block;color:#89b7c7;font-size:10px;font-weight:400}.player-cell{display:flex;align-items:center;gap:8px}.clubmark,.club-chic{font-size:10px;background:#116a69;color:white;border:1px solid #2ce7d1;display:grid;place-items:center;width:34px;height:34px;border-radius:50%;font-weight:900;flex-shrink:0}.club-chic{position:absolute;bottom:0;right:1px;width:23px;height:23px;font-size:8px}.split{display:flex;justify-content:space-between;gap:14px;align-items:center}.small-card{display:flex;justify-content:space-between;align-items:center;background:#0b394a;border-radius:8px;padding:11px;margin:7px 0;gap:12px}.small-card span{color:#80e9da}.split-tiles{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px}.mini-card{background:#0b3444;border:1px solid #1e5869;padding:13px;border-radius:11px}.big-call{font-size:24px;font-weight:900;color:#38f3c7}.risk-row{border-left:3px solid #f4b752;margin:10px 0;padding:7px 10px;background:#143546}.risk-row small{display:block;color:#f2c2a8}.stadium{background:radial-gradient(circle at 50% 8%,#136480,#071e32 70%);border:1px solid #24566a;border-radius:16px;overflow:hidden;padding:12px}.stadium-top{display:flex;align-items:center;gap:12px;padding:2px 12px 10px}.stadium-top svg{width:43px;height:44px}.stadium-top div{flex:1}.stadium-top small{display:block;font-size:9px;color:#a1ddd5}.pitch{height:590px;position:relative;border:4px solid #98f8c655;border-radius:12px;background:repeating-linear-gradient(0deg,#0c9259 0px,#0c9259 68px,#0da35f 68px,#0da35f 136px);box-shadow:inset 0 0 85px #002a32;overflow:hidden}.pitch-centerline{position:absolute;top:49%;width:100%;border-bottom:2px solid #eefff980}.pitch-circle{position:absolute;top:43%;left:40%;width:20%;height:14%;border:2px solid #eefff990;border-radius:50%}.pitch-box{position:absolute;left:27%;width:46%;height:15%;border:2px solid #eefff98a}.pitch-box-top{top:0;border-top:0}.pitch-box-bottom{bottom:0;border-bottom:0}.pitch-inner{position:relative;z-index:2;height:100%;display:flex;flex-direction:column;justify-content:space-around;align-items:center;padding:15px 0}.pitch-row{display:flex;justify-content:space-around;gap:4px;width:100%}.pitch-player{display:flex;flex-direction:column;align-items:center;min-width:0;max-width:120px;width:110px;text-align:center;filter:drop-shadow(0 8px 7px #003a33b0)}.player-portrait{position:relative;background:linear-gradient(140deg,#004051,#19c7b0);width:56px;height:60px;border-radius:14px 14px 3px 3px;border:1px solid #4cffdc}.player-portrait svg{width:55px;height:55px}.player-portrait .headshot{width:100%;height:100%;object-fit:cover;border-radius:12px 12px 1px 1px;display:block}.player-alert{background:#ffa928;color:#312a1c;border-radius:50%;position:absolute;top:-4px;right:-7px;width:18px}.pitch-name{width:100%;padding:5px 4px 1px;background:#effcff;border-radius:4px 4px 0 0;color:#073847;font-weight:850;font-size:12px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}.pitch-name em{font-style:normal;color:#b86519}.pitch-meta{width:100%;padding:2px;background:#c7e7ec;color:#185468;font-size:10px;border-radius:0 0 7px 7px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}.bench-box{margin-top:10px;background:#0b766344;padding:10px;border:1px solid #53dcc36c;border-radius:10px}.bench-grid{display:flex;justify-content:space-around;gap:8px}.bench-grid .pitch-player{width:100px}.good{background:#10583d66;color:#68f0bb}.mid{background:#77602e33;color:#ffecaf}.poor{background:#803b3940;color:#fcbdb3}.chip-cards{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px}.chip-card{border:1px solid #216578;border-radius:12px;background:#0a3446;padding:16px}.chip-card.disabled{opacity:.65}.chip-gain{font-size:24px;font-weight:900;color:#26efc7;margin-top:13px}.chip-gain+p{min-height:40px}.route-note{background:#0f3848;border-left:4px solid var(--teal);padding:14px;border-radius:8px}.summary-result{padding:16px;background:#044956;border:1px solid #2fe7c8;border-radius:12px;margin:15px 0}.simulation-output{font-size:15px;margin-top:9px}.hero-image{width:100%;height:auto;border-radius:10px;border:1px solid #366479}.footer{padding:18px 0 35px;color:#8baebc;font-size:12px}.headline{font-weight:900;font-size:32px;margin:12px 0}.hidden{display:none!important}@media(max-width:1050px){.grid{grid-template-columns:1fr}.metrics{grid-template-columns:repeat(3,minmax(0,1fr))}.top{position:relative;flex-direction:column;align-items:stretch}.tab-nav{overflow:auto;flex-wrap:nowrap}.tab{white-space:nowrap}.pitch{height:560px}.chip-cards{grid-template-columns:1fr}}@media(max-width:600px){.metrics{grid-template-columns:repeat(2,minmax(0,1fr))}.stat strong{font-size:18px}.card{padding:12px}.pitch{height:490px}.pitch-player{width:58px}.player-portrait{width:40px;height:43px}.player-portrait svg{width:39px;height:40px}.pitch-name{font-size:9px;padding:4px 1px}.pitch-meta{font-size:7px}.bench-grid .pitch-player{width:60px}.pitch-row{gap:2px}.identity b{font-size:18px}.split-tiles{grid-template-columns:1fr}.club-chic{width:17px;height:17px;font-size:7px}}
'''
    content=f'''<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><meta name="color-scheme" content="dark"><title>FPLVORTEX • Top-5K Mission Decision Engine V3</title><style>{css}</style></head><body><div class="wrap">
<header class="top"><div class="identity">{LOGO}<div><b>FPLVORTEX</b><div class="mini">PRIMARY: TOP 5K · BACKUP: TOP 10K</div></div></div><nav class="tab-nav" role="tablist" aria-label="FPLVORTEX tabs">{nav}</nav></header>
{v}
<section class="page active" id="page-overview" role="tabpanel"><div class="grid"><div>{pitch_view(engine,first,preferred_ids)}</div><div><div class="card"><h2>What to do for GW{first}</h2><div class="big-call">{esc(headline)}</div><p class="muted">Modelled 6-GW decision; verify the bank and lineup news before confirming.</p><div class="small-card"><b>Captain</b><span>{esc(cap)} — {engine.xp(lineup['captain'],first):.1f} xPts</span></div><div class="small-card"><b>Vice-captain</b><span>{esc(vice)}</span></div><div class="small-card"><b>Formation</b><span>{esc(lineup['formation'])}</span></div><div class="button-row">{button('Next GW action plan','next')}{button('Run chip simulator','chips')}</div></div><div class="card"><h2>Important team risks</h2>{''.join(risk_list)}</div><div class="card"><h2>Chip status</h2><p>{esc(warning)}</p><p class="muted">{esc(chip['recommendation'])}</p>{button('See chip verdict','chips')}</div></div></div><div class="card"><h2>Full squad projection and official availability</h2>{player_table}</div><div class="card"><h2>4K static report</h2><img class="hero-image" src="{img}" alt="FPLVORTEX 4K report"/><p class="mini">4K image is a snapshot. Click the tabs above for forecasts and chip comparisons.</p></div></section>
<section class="page" id="page-decisions" role="tabpanel"><h1 class="headline">Decision Room</h1><div class="grid"><div><div class="card"><h2>Captaincy intelligence</h2><p class="muted">Minutes-adjusted attacking contribution + opponent strength. Higher expectation is not a guarantee.</p>{cap_cards}</div><div class="card"><h2>Full XV minute and xPoints projections</h2>{player_table}</div></div><div><div class="card"><h2>Immediate recommendation</h2><div class="big-call">{esc(headline)}</div><p>{'Additional proposed transfer hit: '+str(first_move['hit'])+' pts' if first_move['hit'] else 'No hits projected for your next decision.'}</p><p class="muted">6-GW beam-search evaluates feasible same-position upgrades while respecting squad rules, bank and transfer costs. Later moves change as news emerges.</p>{button('See six-week roadmap','six')}</div><div class="card"><h2>Watchlist · weaker holdings</h2>{''.join(f'<div class="small-card"><b>{esc(engine.name(pid))}</b><span>{sum(engine.xp(pid,gw) for gw in engine.events):.1f} xP / 6 GW</span></div>' for pid in weak)}</div><div class="card"><h2>High projected current players</h2>{''.join(f'<div class="small-card"><b>{esc(engine.name(pid))}</b><span>{sum(engine.xp(pid,gw) for gw in engine.events):.1f} xP / 6 GW</span></div>' for pid in top_owned)}</div></div></div></section>
<section class="page" id="page-next" role="tabpanel"><h1 class="headline">Next Gameweek • GW{first}</h1>{wc_details}<div class="grid"><div><div class="card"><h2>Your automatically selected action</h2><div class="summary-result"><div class="mini">PRIMARY RECOMMENDATION</div><div class="big-call">{esc(headline)}</div><p>Expected XI + captain: <b>{lineup['expected']:.1f} xPts</b>. These are estimates, not official expected points.</p></div><div class="small-card"><b>Captain</b><span>{esc(cap)}</span></div><div class="small-card"><b>Vice-captain</b><span>{esc(vice)}</span></div><div class="small-card"><b>Formation</b><span>{esc(lineup['formation'])}</span></div><div class="small-card"><b>Bench: GK then 1–3 outfield</b><span>{esc(' / '.join(engine.name(pid) for pid in lineup['bench']))}</span></div><p class="route-note">Before submitting transfers: confirm actual 15 players, latest press conferences and likely starting XI, your real free-transfer count, and the price at confirmation. This notebook cannot see private transfers or guaranteed leaks.</p></div><div class="card"><h2>Chip recommendation</h2><div class="big-call">{esc(chip['recommendation'])}</div><p>{esc(warning)}</p>{button('Simulate all eligible chips','chips')}</div></div><div><div class="card"><h2>Key availability concerns</h2>{''.join(risk_list)}</div><div class="card"><h2>How strong is this decision?</h2><p class="muted">The model uses event-wise xG/xA, minutes probability, status, opposition xG and Poisson clean sheet estimates. It does not pretend to have verified line-up leaks, betting prices or top-10K ownership unless supplied.</p><p class="mini">Official player news is read at every Colab run. Optional verified news and odds CSV files are read from Drive.</p></div></div></div></section>
<section class="page" id="page-six" role="tabpanel"><h1 class="headline">Your six-gameweek roadmap (GW{first}–GW{result['events'][-1]})</h1>{wc_six_table}<div class="card"><h2>Alternative: route without a Wildcard (for reference)</h2><p class="muted">A beam search maintains one consistent squad/bank path. Each later-GW action is provisional—not six independent transfers.</p>{route_table}</div><div class="card"><h2>Player projection & fixture matrix</h2>{matrix}</div><div class="card"><h2>Next decision</h2><div class="big-call">{esc(headline)}</div><p>Projected weighted six-GW route score: {result['plan_total_projected']:.1f}. Confidence decreases further into the horizon. Actual fixtures, injuries and prices can change.</p><div class="button-row">{button('Chip simulation for this route','chips')}{button('Return to GW plan','next')}</div></div></section>
<section class="page" id="page-chips" role="tabpanel"><h1 class="headline">Chip Solver & Monte Carlo Simulation</h1><div class="card"><div class="split"><div><h2>{esc(chip['recommendation'])}</h2><p class="muted">Official chip availability by season half. Simulations: {chip['simulation_count']:,} per scenario. First-half chips expire at GW19; 1 chip per GW.</p></div>{badge('Eligibility checked','teal')}</div></div><div class="chip-cards">{''.join(chip_cards)}</div><div class="card" style="margin-top:15px"><h2>Simulation workspace</h2><p>Click an eligible chip's simulation button. This re-samples its previously solved scenario with the <b>same model inputs</b> — it does not fetch news. Re-run the Colab notebook for fresh data.</p><div class="simulation-output" id="sim-output" role="status" aria-live="polite">Select a chip above to compare forecast outcomes.</div><div class="button-row"><button class="button alt" type="button" id="compare-chips">Compare available chips</button>{button('View planned transfers','six')}</div></div><div class="card"><h2>How scenarios work</h2><p><b>Wildcard:</b> Legal 15-man permanent squad, cost/3-club/position constraints, 6-GW return versus the no-chip transfer route. <b>Free Hit:</b> Legal 15-man one-week squad, reverting afterwards. <b>Bench Boost:</b> Real bench + appearance forecasts. <b>Triple Captain:</b> Extra captain's score. Model uncertainty and opportunity cost beyond GW{result['events'][-1]} are not fully captured.</p><p class="muted">Optimizers search a broad but trimmed candidate pool; optimality is guaranteed only within that candidate pool when the MILP solver reports success.</p></div></section>
<section class="page" id="page-news" role="tabpanel"><h1 class="headline">Team News, Injury Radar & Open Market Context</h1>{rank_box}<div class="grid"><div class="card"><h2>Verified FPL player status + independent headlines</h2><p class="muted">Official FPL flags inform projections; RSS headlines do NOT independently verify rumors or confirm predicted elevens. Trust published club XIs at kickoff.</p>{source_report}</div><div><div class="card"><h2>Free public prediction-market context</h2><p class="muted">Three-outcome prices, when exactly matched to the listed fixtures, are research signals only; not player-goal odds, bookmaker quotes, or modeled projections.</p>{market_table}</div><div class="card"><h2>Deadline rumor verification protocol</h2><p>Check an official club / manager quote first. For X rumors require two independent named accounts citing different underlying evidence, correct timestamps, and no recycled screenshot. Treat as UNCONFIRMED until official XI. A live RSS article referencing a player is <b>not</b> two corroborating lineup sources.</p><p>Near deadline, re-run Colab; probabilities cannot be automatically adjusted from rumor-only headlines.</p></div></div></div></section>
<section class="page" id="page-audit" role="tabpanel"><h1 class="headline">Data provenance, assumptions & limits</h1><div class="card"><h2>Data freshness</h2><p>Collected at: <b>{esc(result['generated_utc'])}</b> · official FPL API. Latest published picks: GW{result['last_published_gw']}. Next deadline GW{first}. Player history logs fetched: {result['data_sources']['player_summaries']}.</p><div class="split-tiles"><div class="mini-card"><h3>Team strength & clean sheets</h3><p>Modelled recent match goals, home/away ratings and Poisson probabilities. <b>{esc(result['data_sources']['clean_sheet_odds'])}</b>. FPL 1–5 FDR is shown, but not treated as a bookmaker probability.</p></div><div class="mini-card"><h3>Minutes / rotation / injuries</h3><p>Official status, chance of playing, last five match minutes where accessible, and limited schedule congestion. <b>European / cup rotation is not automatically available</b>.</p></div><div class="mini-card"><h3>Team news and leaks</h3><p>FPL 'news' field plus {engine.news_count} verified manual overrides from news_overrides.csv. Unverified or older-than-48h reports are ignored. No automatic social-media leak feed claimed.</p></div><div class="mini-card"><h3>Odds & sources</h3><p>{engine.odds_count} externally verified odds/clean-sheet overrides loaded. If no odds CSV is supplied, the dashboard shows model-estimated probabilities, not bookmaker odds.</p></div><div class="mini-card"><h3>Account private data</h3><p>True FTs, private GW{first} changes and planned chips require user confirmation. Assumed FTs: {integer(engine.ft)}. Current Bank: £{result['bank']:.1f}m, estimated from GW7 screenshot transfers unless overridden.</p></div><div class="mini-card"><h3>Why rankings are uncertain</h3><p>Top-10K effective ownership, manager decisions, bonus points, final team news and injury outcomes are not reliably available. A top-10K finish or winning the league is never guaranteed.</p></div></div></div><div class="card"><h2>Optional data inputs (Google Drive / FPLVORTEX / Input)</h2><p><b>news_overrides.csv</b> (verified=yes AND source contains at least two independent evidence URLs separated with a semicolon) columns: player_id,start_probability,minutes_if_started,source,observed_utc,verified,note</p><p><b>odds_overrides.csv</b> columns: gw,team_id,clean_sheet_probability,team_xg,source,observed_utc,verified</p><p class="muted">Timestamp observed_utc is ISO 8601 UTC. Include a verifiable source and verified=yes. Any row older than 48 hours is ignored. Keep these files optional; engine runs from official FPL data alone.</p></div></section>
<footer class="footer">FPLVORTEX V4 • Entry #{ENTRY_ID} • Top-5K / Top-10K are targets, not a prediction or guarantee. HTML buttons compare calculated scenarios; only Colab Run all fetches new public data.</footer></div>
<script>const STORE={script_data};
(function(){{"use strict";const tabs=[...document.querySelectorAll('[data-tab]')];const keys=tabs.map(x=>x.dataset.tab);function activate(key){{let tab=keys.includes(key)?key:'overview';for(const btn of tabs)btn.setAttribute('aria-selected',String(btn.dataset.tab===tab));for(const panel of document.querySelectorAll('.page'))panel.classList.toggle('active',panel.id==='page-'+tab);window.location.hash=tab;}}for(const el of tabs)el.addEventListener('click',()=>activate(el.dataset.tab));for(const el of document.querySelectorAll('[data-go]'))el.addEventListener('click',()=>activate(el.dataset.go));window.addEventListener('hashchange',()=>{{const dest=window.location.hash.slice(1);if(keys.includes(dest))activate(dest);}});activate(window.location.hash.slice(1));
const labels={{wildcard:'Wildcard',freehit:'Free Hit',triplecaptain:'Triple Captain',benchboost:'Bench Boost'}};
function normalRandom(){{const u=Math.max(1e-10,Math.random());return Math.sqrt(-2*Math.log(u))*Math.cos(2*Math.PI*Math.random());}}
function simulation(name){{const s=STORE.scenario[name];const output=document.getElementById('sim-output');if(!s||!s.available||s.mean_gain===null){{output.textContent='Chip unavailable or scenario failed legal optimization.';return;}}const stdev=Math.max(.3,(s.p90_gain-s.p10_gain)/2.563);const samples=[];for(let i=0;i<1500;i++)samples.push(s.mean_gain+stdev*normalRandom());samples.sort((a,b)=>a-b);const avg=samples.reduce((a,b)=>a+b,0)/samples.length;const positive=samples.filter(x=>x>0).length/samples.length;output.innerHTML='<b>'+labels[name]+' — model forecast re-sample</b><br>Expected marginal '+avg.toFixed(1)+' points, 10th–90th percentile '+samples[150].toFixed(1)+' to '+samples[1349].toFixed(1)+'. Probability > 0: '+(positive*100).toFixed(0)+'%.<p class="mini">Illustrative normal approximation of the Colab Monte Carlo delta, not a fresh API run. Wildcard spans up to 6 GW; other chips are GW-only comparisons.</p>';}}
for(const btn of document.querySelectorAll('[data-sim]'))btn.addEventListener('click',()=>simulation(btn.dataset.sim));document.getElementById('compare-chips').addEventListener('click',()=>{{const eligible=Object.entries(STORE.scenario).filter(([k,v])=>v.available&&v.mean_gain!==null).sort((a,b)=>b[1].mean_gain-a[1].mean_gain);const out=document.getElementById('sim-output');out.innerHTML=eligible.length?'<b>Available chips by modelled marginal value:</b><br>'+eligible.map(([k,v])=>labels[k]+': '+(v.mean_gain>=0?'+':'')+v.mean_gain.toFixed(1)+' pts').join('<br>'):'No eligible chip solved for this gameweek.';}});
}})();</script></body></html>'''
    return content


def draw_four_k(result:dict[str,Any],engine:DecisionEngine,path:Path)->None:
    gw=engine.events[0]
    lineup=engine.lineup(tuple(result['recommended_squad_ids']),gw)
    fig=plt.figure(figsize=(19.2,10.8),dpi=200)
    ax=fig.add_axes([0,0,1,1])
    fig.patch.set_facecolor(STYLE_BG);ax.set_facecolor(STYLE_BG)
    ax.set_xlim(0,1920);ax.set_ylim(0,1080);ax.axis('off')
    def rect(x,y,w,h,fill,edge='#2b6371',rounding=18):
        ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle=f'round,pad=0,rounding_size={rounding}',edgecolor=edge,facecolor=fill,linewidth=1.8))
    def text(x,y,s,size=16,color='white',bold=False,ha='left'):
        ax.text(x,y,s,fontsize=size,color=color,fontweight='bold' if bold else 'normal',ha=ha,va='center',family='DejaVu Sans')
    rect(16,984,1888,82,'#0b3043');text(40,1025,'FPLVORTEX  •  TEAM 145784',30,STYLE_TEAL,True)
    text(1110,1025,f'GW{gw}    |    TOP 10K DECISION ENGINE',22,'#e9fbff',True)
    rect(20,885,1210,83,'#f1fcfd');rect(1245,885,658,83,'#f1fcfd')
    text(42,928,f'GAMEWEEK {gw}  |  RANK #{integer(result["overall_rank"]):,}',29,'#09293e',True)
    text(1270,928,f'BANK £{result["bank"]:.1f}M    |    {integer(engine.ft)} FTs*',22,'#09293e',True)
    rect(20,70,1209,797,'#0c5b48',edge='#49ddbd');rect(1245,70,658,797,'#0b2537',edge='#24546e')
    for band in range(0,800,75):
        ax.add_patch(Rectangle((35,79+band),1180,75,color='#087d56' if (band//75)%2==0 else '#0a8b57',alpha=.9))
    ax.add_patch(Rectangle((36,82),1175,776,fill=False,edgecolor='#a5eec9',linewidth=3))
    ax.plot([36,1211],[469,469],color='#b9fadc',linewidth=2)
    ax.add_patch(Circle((625,469),90,fill=False,edgecolor='#b9fadc',linewidth=2))
    ax.add_patch(Rectangle((375,750),500,107,fill=False,edgecolor='#b9fadc',linewidth=2))
    positions={1:[(625,774)],2:[],3:[],4:[]}
    for pos,y in ((2,631),(3,450),(4,260)):
        group=[pid for pid in lineup['xi'] if integer(engine.player(pid)['element_type'])==pos]
        width=1010
        xs=np.linspace(625-width/2,625+width/2,len(group)) if len(group)>1 else np.array([625])
        positions[pos]=list(zip(xs,[y]*len(xs)))
    all_ids={pos:[pid for pid in lineup['xi'] if integer(engine.player(pid)['element_type'])==pos] for pos in POS_NAME}
    for pos in POS_NAME:
        for (x,y),pid in zip(positions[pos],all_ids[pos]):
            x=float(x)
            rect(x-86,y-44,172,93,'#e1fcfa',edge='#043f4e',rounding=12)
            ax.add_patch(Circle((x,y+22),17,color='#15ab9c'))
            text(x,y-2,engine.name(pid)[:14],14,'#003547',True,'center')
            f=engine.forecasts[(pid,gw)]
            text(x,y-26,f'{f.fixture[:11]}  •  {f.xp:.1f}',11,'#285569',ha='center')
            if pid==lineup['captain']:text(x+66,y+32,'C',17,'#c08103',True)
    rect(60,101,1110,92,'#064439',edge='#37d9b5');text(80,166,'BENCH',17,'#a3fbed',True)
    for i,pid in enumerate(lineup['bench']):
        text(265+230*i,143,f'{engine.name(pid)[:14]}   {engine.xp(pid,gw):.1f} xP',13,'#eaffff',True,'center')
    text(1270,826,'THIS WEEK  •  ACTION',24,STYLE_TEAL,True)
    step=result['plan'][0]
    action=f"{engine.name(step['out'])}  →  {engine.name(step['into'])}" if step['out'] else 'ROLL / HOLD TRANSFER'
    text(1270,785,action,21,'#ffffff',True)
    text(1270,744,f'Captain: {engine.name(lineup["captain"])}',18,'#e9fffa')
    text(1270,704,f'Vice: {engine.name(lineup["vice"])}',18,'#e9fffa')
    text(1270,664,f'Formation: {lineup["formation"]}',18,'#e9fffa')
    text(1270,608,'SIX-GAMEWEEK ROADMAP',24,STYLE_TEAL,True)
    for idx,step in enumerate(result['plan']):
        y=565-idx*64
        rect(1260,y-28,625,54,'#103b4e',edge='#296175',rounding=7)
        text(1275,y,f'GW{step["gw"]}',16,STYLE_TEAL,True)
        st=f'{step["out_name"]} → {step["in_name"]}' if step['out'] else 'Hold / roll'
        text(1368,y,st[:27],15,'#ffffff')
        text(1835,y,f"{step['xp']:.1f} xP",14,'#bef6e6',True,'right')
    text(1260,110,chip_text_4k(result)[:73],13,'#ffd99c')
    path.parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(path,dpi=200,facecolor=STYLE_BG)
    plt.close(fig)
    from PIL import Image
    with Image.open(path) as check:
        if check.size!=(3840,2160):
            raise RuntimeError(f'Incorrect 4K output size: {check.size}')


def chip_text_4k(result:dict[str,Any])->str:
    return f"CHIP: {result['chip']['recommendation']}   •   FTs verified in manager-state audit; reconfirm before deadline"


def google_drive_folder() ->Path:
    try:
        from google.colab import drive
    except ImportError as error:
        raise RuntimeError('Open this notebook in Google Colab to auto-mount Google Drive.') from error
    drive.mount('/content/drive',force_remount=False)
    folder=Path('/content/drive/MyDrive')/DRIVE_DIRECTORY
    folder.mkdir(parents=True,exist_ok=True)
    optional_dir=folder.parent/'Input'
    optional_dir.mkdir(parents=True,exist_ok=True)
    headers={
        'news_overrides.csv':'player_id,start_probability,minutes_if_started,source,observed_utc,verified,note\n',
        'odds_overrides.csv':'gw,team_id,clean_sheet_probability,team_xg,source,observed_utc,verified\n',
    }
    for filename,header in headers.items():
        path=optional_dir/filename
        if not path.exists(): path.write_text(header,encoding='utf-8')
    return folder


def run(output:Path|None=None,snapshot:dict[str,Any]|None=None)->tuple[Path,Path,Path]:
    destination=output if output is not None else google_drive_folder()
    destination.mkdir(parents=True,exist_ok=True)
    data=snapshot if snapshot is not None else collect(destination)
    if snapshot is None:
        data['news_radar']=collect_news_radar(data,fetch_external=True)
        data['markets']=collect_polymarket_context(data,fetch_external=True)
    else:
        data.setdefault('news_radar',collect_news_radar(data,fetch_external=False))
        data.setdefault('markets',collect_polymarket_context(data,fetch_external=False))
    if snapshot is None:
        data=apply_gw7_screen_squad(data)
    model=DecisionEngine(data,destination.parent/'Input')
    if snapshot is None:
        embed_visual_assets(model,destination)
    result=make_result(model)
    png=destination/'FPLVORTEX_TOP5K_LATEST_4K.png'
    json_path=destination/'FPLVORTEX_TOP5K_LATEST.json'
    html_path=destination/'FPLVORTEX_TOP5K_AUTOPILOT_V4.html'
    draw_four_k(result,model,png)
    html_text=render_html(result,model,png)
    if html_text.count('data-tab=')<7 or 'data-sim=' not in html_text:
        raise RuntimeError('Interactive navigation or chip buttons missing')
    json_path.write_text(json.dumps(result,ensure_ascii=False,separators=(',',':'),default=str),encoding='utf-8')
    html_path.write_text(html_text,encoding='utf-8')
    print('\n✅ V4 interactive report:',html_path)
    print('✅ 4K image (3840 × 2160):',png)
    print('✅ Model, data, scenarios & assumptions:',json_path)
    print(f"🎯 GW{result['events'][0]} | Top-5K mission (Top-10K backup) | {result['chip']['recommendation']}")
    print('⚠️ You must confirm real FTs, current squad, lineup news and any chips before applying advice.')
    display(HTML(f'<div style="font-family:Arial;padding:12px;background:#0a3242;color:white"><h3>FPLVORTEX V4 ready</h3><a href="{html_path.as_uri()}" target="_blank" style="color:#45f1cd">Open saved HTML from Google Drive or download it and open in Chrome</a><br>Next gameweek: GW{result["events"][0]}</div>'))
    return html_path,png,json_path



## Added cell 1 — Verified manager state and hard validation
Live FPL HTTP, Drive and V4.1 engine remain untouched. On missing/conflicting private state, stop rather than estimate a transfer hit.

In [ ]:
# V4.2 cell additions: uses V4.1's DecisionEngine, fetch, collect, render_html and draw_four_k.
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import re
from dataclasses import asdict, dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Mapping

import numpy as np

AUDIT_VERSION = '4.5.0'
AUDIT_MONTE_CARLO = 5000
AUDIT_MAX_AGE_HOURS = 36
AUDIT_HORIZON = HORIZON  # Inherits the existing configurable rolling horizon (6–8).
AUDIT_MIN_WILDCARD_WIN_RATE = 0.70
AUDIT_MIN_STABILITY = 3
AUDIT_MIN_MEDIAN_SURPLUS_PTS = 3.0


@dataclass(frozen=True)
class Diagnostic:
    severity: str
    code: str
    message: str


class AuditBlocked(RuntimeError):
    def __init__(self, diagnostics: list[Diagnostic]):
        self.diagnostics = diagnostics
        super().__init__('; '.join(d.message for d in diagnostics if d.severity == 'ERROR'))


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def canonical_json(value: Any) -> bytes:
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':'), default=str).encode('utf-8')


def atomic_json(path: Path, value: Any, *, immutable: bool = False) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = canonical_json(value)
    if immutable:
        with path.open('xb') as stream:
            stream.write(payload)
    else:
        temp = path.with_name(path.name + '.tmp')
        with temp.open('wb') as stream:
            stream.write(payload)
        temp.replace(path)


def schema_template(input_folder: Path, official: dict[str, Any] | None = None) -> Path:
    """Create a blank, explicitly unverified private-state form; never fabricate values."""
    input_folder.mkdir(parents=True, exist_ok=True)
    path = input_folder / 'manager_state.json'
    if not path.exists():
        provisional_picks = (official or {}).get('picks', {}).get('picks', [])
        provisional_squad = ([{'element': integer(row.get('element')),
                               'selling_price': integer(row.get('selling_price'))}
                              for row in provisional_picks] if len(provisional_picks) == 15 else [])
        atomic_json(path, {
            'entry_id': integer((official or {}).get('profile', {}).get('id'), 0) or None, 'verified_at_utc': None, 'next_event': None,
            'source': 'FPL Transfers / Pick Team, checked by manager',
            'confirmed': False,
            'current_squad': provisional_squad,
            'bank_tenths': None, 'team_value_tenths': None,
            'ft_before_moves': None, 'transfers_made_this_gw': None,
            'displayed_hit_cost': None, 'ft_after_moves': None,
            'ft_after_wildcard': None,
            'chip_available': {'wildcard': None, 'freehit': None, 'triplecaptain': None, 'benchboost': None},
            'active_chip': None,
            'notes': 'PROVISIONAL OLD LOCKED SQUAD: public FPL picks do NOT include current GW transfers. Edit every changed player and verify all 15 selling prices, FT counts, and chip status in official FPL. Set confirmed true only then.',
        })
    return path


def load_private_state(path: Path, official: dict[str, Any], events: list[int], now: datetime) -> tuple[dict[str, Any] | None, list[Diagnostic]]:
    issues: list[Diagnostic] = []
    if not path.exists():
        return None, [Diagnostic('ERROR', 'PRIVATE_STATE_MISSING', f'Missing verified current manager state: {path}')]
    try:
        state = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError) as error:
        return None, [Diagnostic('ERROR', 'PRIVATE_STATE_INVALID_JSON', str(error))]
    if not isinstance(state, dict):
        return None, [Diagnostic('ERROR', 'PRIVATE_STATE_TYPE', 'manager_state.json must be an object')]
    timestamp = iso_time(state.get('verified_at_utc'))
    if timestamp is None or timestamp > now + timedelta(minutes=10) or timestamp < now - timedelta(hours=AUDIT_MAX_AGE_HOURS):
        issues.append(Diagnostic('ERROR', 'PRIVATE_STATE_STALE', f'Current FPL Transfers/Pick Team screen must be rechecked within {AUDIT_MAX_AGE_HOURS} hours.'))
    if state.get('confirmed') is not True:
        issues.append(Diagnostic('ERROR', 'PRIVATE_STATE_UNCONFIRMED', 'Explicit confirmed=true is required after checking FPL Transfers.'))
    if state.get('public_provisional') is True or state.get('unobservable_fields'):
        issues.append(Diagnostic('ERROR', 'PROVISIONAL_FIELDS_PRESENT',
            'Public/provisional state or unobservable fields cannot pass the private-state gate.'))
    if any(isinstance(row,dict) and ('proxy' in str(row.get('selling_price_source','')).lower()
             or 'estimate' in str(row.get('selling_price_source','')).lower())
           for row in (state.get('current_squad') or []) if isinstance(state.get('current_squad'),list)):
        issues.append(Diagnostic('ERROR','PUBLIC_PRICE_ESTIMATES',
            'A public selling-price estimate is not a verified current selling price.'))
    if state.get('source') not in ('FPL Transfers / Pick Team, checked by manager', 'Authenticated FPL account state'):
        issues.append(Diagnostic('ERROR', 'PRIVATE_STATE_SOURCE', 'Source must be authenticated FPL account or user-verified official screenshots.'))
    if state.get('entry_id') != integer(official['profile'].get('id')):
        issues.append(Diagnostic('ERROR', 'ENTRY_MISMATCH', 'Private state entry does not match public API entry.'))
    if not events or len(events) > AUDIT_HORIZON or state.get('next_event') != (events[0] if events else None):
        issues.append(Diagnostic('ERROR', 'EVENT_MISMATCH', f'Private next_event must equal live next event {events[0] if events else "missing"}.'))
    rows = state.get('current_squad')
    all_players = {integer(p['id']): p for p in official['bootstrap']['elements']}
    if not isinstance(rows, list) or len(rows) != 15:
        issues.append(Diagnostic('ERROR', 'SQUAD_COUNT', 'Exactly 15 current player rows required, including verified selling prices.'))
        rows = []
    ids: list[int] = []
    missing_prices: list[str] = []
    counts: dict[int, int] = {}
    pos: dict[int, int] = {}
    for row in rows:
        if not isinstance(row, dict) or type(row.get('element')) is not int or type(row.get('selling_price')) is not int:
            issues.append(Diagnostic('ERROR', 'SQUAD_ROW', 'Each player requires integer element and selling_price in 0.1m units.'))
            continue
        player_id, selling = row['element'], row['selling_price']
        player = all_players.get(player_id)
        if player is None:
            issues.append(Diagnostic('ERROR', 'SQUAD_PLAYER', f'FPL player ID {player_id} is not in the official player list.'))
            continue
        # Price verification does not affect the identity/formation checks.
        ids.append(player_id)
        counts[integer(player['team'])] = counts.get(integer(player['team']), 0) + 1
        p = integer(player['element_type'])
        pos[p] = pos.get(p, 0) + 1
        if not 25 <= selling <= 300:
            missing_prices.append(str(player.get('web_name') or player_id))
    if missing_prices:
        issues.append(Diagnostic('ERROR', 'SELLING_PRICES_UNVERIFIED',
                                 f'{len(missing_prices)} selling prices have not been verified in FPL Transfers: '
                                 + ', '.join(missing_prices[:8])
                                 + (f' and {len(missing_prices)-8} more' if len(missing_prices)>8 else '') + '.'))
    if len(ids) != len(set(ids)) or any(count > 3 for count in counts.values()):
        issues.append(Diagnostic('ERROR', 'SQUAD_CLUB', 'Duplicate players or more than three from one club.'))
    if pos != POSITION_COUNTS:
        issues.append(Diagnostic('ERROR', 'SQUAD_POSITION', f'Position counts must be {POSITION_COUNTS}.'))
    for key in ('bank_tenths', 'team_value_tenths', 'ft_before_moves', 'transfers_made_this_gw', 'displayed_hit_cost', 'ft_after_moves'):
        if type(state.get(key)) is not int or state[key] < 0:
            issues.append(Diagnostic('ERROR', 'STATE_FIELD', f'{key} must be a verified nonnegative integer.'))
    wc_ft = state.get('ft_after_wildcard')
    if wc_ft is not None and (type(wc_ft) is not int or not 0 <= wc_ft <= 5):
        issues.append(Diagnostic('ERROR', 'WC_FT_INVALID', 'ft_after_wildcard must be a verified FT count or null.'))
    chip_status = state.get('chip_available')
    if not isinstance(chip_status, dict) or set(chip_status) != {'wildcard', 'freehit', 'triplecaptain', 'benchboost'} or not all(type(chip_status[key]) is bool for key in chip_status):
        issues.append(Diagnostic('ERROR', 'CHIP_STATE', 'chip_available requires four explicitly confirmed boolean fields.'))
    if state.get('active_chip') not in (None, 'wildcard', 'freehit', 'triplecaptain', 'benchboost'):
        issues.append(Diagnostic('ERROR', 'ACTIVE_CHIP', 'Unknown active chip.'))
    if state.get('active_chip') is not None:
        issues.append(Diagnostic('ERROR', 'ACTIVE_CHIP_PRESENT', 'Chip already selected; recommendation requires a fresh state/simulation.'))
    if type(state.get('bank_tenths')) is int and len(rows) == 15 and len(ids) == 15:
        selling_budget = state['bank_tenths'] + sum(row['selling_price'] for row in rows)
        public_current_cost = state['bank_tenths'] + sum(integer(all_players[pid]['now_cost']) for pid in ids)
        if selling_budget > public_current_cost + 3:
            issues.append(Diagnostic('ERROR', 'SELLING_BUDGET_IMPLAUSIBLE',
                                      'Verified selling budget exceeds current market-cost estimate. Verify current selling prices.'))
        # The displayed team value is NOT the same as liquidation value used for WC transfers.
        # Preserve both separately; never force an equality that would reject legitimate paper gains.
    if type(state.get('ft_before_moves')) is int and type(state.get('ft_after_moves')) is int and type(state.get('transfers_made_this_gw')) is int:
        if not 0 <= state['ft_after_moves'] <= 5 or not 0 <= state['ft_before_moves'] <= 5:
            issues.append(Diagnostic('ERROR', 'FT_OUT_OF_RANGE', 'FT counts must be between zero and five.'))
        if state['ft_before_moves'] - min(state['ft_before_moves'], state['transfers_made_this_gw']) != state['ft_after_moves']:
            issues.append(Diagnostic('ERROR', 'FT_CONTRADICTION', 'FT before, transfers made and FT after contradict the official display.'))
    # Never infer actual hits: only check that screenshot's confirmed value is plausible.
    if type(state.get('displayed_hit_cost')) is int and state['displayed_hit_cost'] % 4 != 0:
        issues.append(Diagnostic('ERROR', 'HIT_COST_INVALID', 'Displayed transfer hit must be in four-point increments.'))
    used_chip_names = {str(c.get('name')) for c in official['history'].get('chips', []) if isinstance(c, dict)}
    next_gw = events[0] if events else -1
    used_in_half = {str(c.get('name')) for c in official['history'].get('chips', []) if (1 <= integer(c.get('event')) <= 19 if next_gw <= 19 else 20 <= integer(c.get('event')) <= 38)}
    names = {'wildcard': 'wildcard', 'freehit': 'freehit', 'triplecaptain': '3xc', 'benchboost': 'bboost'}
    if isinstance(chip_status, dict):
        for name, code in names.items():
            if chip_status.get(name) is True and code in used_in_half:
                issues.append(Diagnostic('ERROR', 'CHIP_CONTRADICTION', f'{name} marked available but official history shows it used this half.'))
    if not issues:
        issues.append(Diagnostic('INFO', 'STATE_VERIFIED', f'Manager state verified: {timestamp.isoformat()}; displayed hit {state["displayed_hit_cost"]}pts (not inferred).'))
    return state, issues


def apply_verified_state(data: dict[str, Any], state: dict[str, Any]) -> dict[str, Any]:
    output = dict(data)
    output['picks'] = dict(data['picks'])
    previous = {integer(x['element']): x for x in data['picks']['picks']}
    private_ids = {row['element'] for row in state['current_squad']}
    new_picks = []
    for index, row in enumerate(state['current_squad'], 1):
        pid = row['element']
        original = previous.get(pid, {})
        new_picks.append({**original, 'element': pid, 'position': index, 'selling_price': row['selling_price'],
                          'purchase_price': integer(original.get('purchase_price'), row['selling_price']),
                          'multiplier': 0, 'is_captain': False, 'is_vice_captain': False})
    output['picks']['picks'] = new_picks
    output['profile'] = {**data['profile'], 'last_deadline_bank': state['bank_tenths']}
    output['manager_state'] = copy.deepcopy(state)
    output['squad_provenance'] = 'Verified current Transfers/Pick Team snapshot, not public last-deadline picks'
    output['pending_transfers_meta'] = {
        'out_ids': sorted(set(previous) - private_ids), 'in_ids': sorted(private_ids - set(previous)),
        'free_transfers_original': state['ft_before_moves'],
        'transfers_done': state['transfers_made_this_gw'],
        'points_hits_pending': state['displayed_hit_cost'],
        'bank_verified': True,
    }
    return output


def save_raw_archive(data: dict[str, Any], destination: Path, fetched_at: datetime) -> tuple[Path, str]:
    immutable_data = {key: data.get(key) for key in ('bootstrap', 'profile', 'history', 'fixtures', 'transfers', 'picks', 'picked_gw', 'summaries', 'rank_benchmarks')}
    immutable_data['entry_id'] = integer(data['profile']['id'])
    immutable_data['retrieved_utc'] = fetched_at.isoformat()
    digest = hashlib.sha256(canonical_json(immutable_data)).hexdigest()
    folder = destination / 'Archive'
    path = folder / (f'raw_{fetched_at.strftime("%Y%m%dT%H%M%S%fZ")}_{digest[:12]}.json')
    atomic_json(path, immutable_data, immutable=True)
    atomic_json(path.with_suffix('.sha256.json'), {'sha256': digest, 'file': path.name, 'retrieved_utc': fetched_at.isoformat()}, immutable=True)
    return path, digest


def save_state_archive(state: dict[str, Any], destination: Path, now: datetime) -> tuple[Path, str]:
    immutable = copy.deepcopy(state)
    immutable['archived_utc'] = now.isoformat()
    digest = hashlib.sha256(canonical_json(immutable)).hexdigest()
    path = destination / 'Archive' / f'manager_{now.strftime("%Y%m%dT%H%M%S%fZ")}_{digest[:12]}.json'
    atomic_json(path, immutable, immutable=True)
    return path, digest


def assert_pre_render(engine: Any, state: dict[str, Any], raw_time: datetime) -> list[Diagnostic]:
    issues: list[Diagnostic] = []
    now = utc_now()
    if not (now - timedelta(hours=AUDIT_MAX_AGE_HOURS) <= raw_time <= now + timedelta(minutes=10)):
        issues.append(Diagnostic('ERROR', 'API_STALE', 'Official API fetch is missing or stale.'))
    if not engine.events or len(engine.events) > AUDIT_HORIZON or engine.events[0] != state['next_event']:
        issues.append(Diagnostic('ERROR', 'GW_MISMATCH', 'Next Gameweek or full rolling horizon mismatch.'))
    if not engine.valid_squad(engine.owned):
        issues.append(Diagnostic('ERROR', 'ILLEGAL_XV', 'Current squad has illegal player/position/club counts.'))
    if engine.bank != state['bank_tenths']:
        issues.append(Diagnostic('ERROR', 'BANK_MISMATCH', 'Optimizer bank is not the verified bank.'))
    verified_selling_budget = state['bank_tenths'] + sum(row['selling_price'] for row in state['current_squad'])
    if engine.budget != verified_selling_budget:
        issues.append(Diagnostic('ERROR', 'BUDGET_MISMATCH', 'Selling-budget calculation does not reconcile with verified sale prices + bank.'))
    if integer(engine.ft) != state['ft_after_moves']:
        issues.append(Diagnostic('ERROR', 'FT_MISMATCH', 'Optimizer FTs do not match verified displayed free transfers.'))
    if engine.snapshot.get('pending_transfers_meta', {}).get('points_hits_pending') != state['displayed_hit_cost']:
        issues.append(Diagnostic('ERROR', 'HIT_MISMATCH', 'Optimizer hits must match the verified displayed points cost.'))
    for gw in engine.events:
        try:
            line = engine.lineup(engine.owned, gw)
            xi = line['xi']
            if len(xi) != 11 or len(set(xi)) != 11 or line['captain'] not in xi or line['vice'] not in xi or line['vice'] == line['captain']:
                raise ValueError('Invalid XI/captain/vice')
            formations = {pos: sum(integer(engine.players[pid]['element_type']) == pos for pid in xi) for pos in POSITION_COUNTS}
            if not (formations[1] == 1 and 3 <= formations[2] <= 5 and 2 <= formations[3] <= 5 and 1 <= formations[4] <= 3):
                raise ValueError('Invalid legal formation')
        except (ValueError, KeyError, IndexError) as error:
            issues.append(Diagnostic('ERROR', 'LINEUP_INVALID', f'GW{gw}: {error}'))
    return issues


def minutes_profile(engine: Any, pid: int, gw: int, scale: float = 1.0) -> dict[str, float]:
    forecast = engine.forecasts[(pid, gw)]
    stat = engine.stats[pid]
    p_start = min(0.999, max(0.0, forecast.p_start * scale))
    p_sub_given_bench = min(1.0, max(0.0, stat.get('p_sub_given_bench', 0.15)))
    start_minutes = min(95, max(1, stat['start_minutes']))
    history = engine.snapshot.get('summaries', {}).get(str(pid), {}).get('history', [])
    sub_minutes = [num(row.get('minutes')) for row in history[-12:] if 0 < num(row.get('minutes')) < 45]
    substitute_minutes = stat.get('bench_minutes', float(np.median(sub_minutes)) if len(sub_minutes) >= 2 else 18.0)
    substitute_minutes = max(5.0, min(45.0, substitute_minutes))
    return {'p_start': p_start, 'p_sub_given_bench': p_sub_given_bench,
            'expected_start_minutes': start_minutes, 'expected_sub_minutes': substitute_minutes,
            'expected_minutes': p_start * start_minutes + (1 - p_start) * p_sub_given_bench * substitute_minutes}


def apply_role_priors(engine: Any) -> None:
    """Compatibility hook: role effects are already applied in player_for_gw.

    Avoid adding set-piece priors twice; news and odds override checks are unchanged.
    """
    return None


def explicit_components(engine: Any, pid: int, gw: int) -> dict[str, Any]:
    forecast = engine.forecasts[(pid, gw)]
    profile = minutes_profile(engine, pid, gw)
    player = engine.players[pid]
    pos = integer(player['element_type'])
    pstart = profile['p_start']
    cameo = (1 - pstart) * profile['p_sub_given_bench']
    p60 = forecast.p60
    appearance = pstart * (1 + p60) + cameo
    goal_xp = {1: 10, 2: 6, 3: 5, 4: 4}[pos] * forecast.xg
    assist_xp = 3 * forecast.xa
    clean_sheet = (4 if pos in (1, 2) else 1 if pos == 3 else 0) * pstart * p60 * forecast.cs_probability
    saves = engine.stats[pid]['saves90'] * (profile['expected_minutes'] / 90) / 3 if pos == 1 else 0
    defensive = 2 * engine.stats[pid]['dcprob'] * pstart if pos in (2, 3, 4) else 0
    cards = -engine.stats[pid]['yc90'] * profile['expected_minutes'] / 90
    bonus = engine.stats[pid]['bonus90'] * profile['expected_minutes'] / 90 * 0.75
    # Preserve existing xP as authoritative while explicitly exposing components and unmatched model terms.
    subtotal = appearance + goal_xp + assist_xp + clean_sheet + saves + defensive + cards + bonus
    adjustments = forecast.xp - subtotal
    return {
        'player_id': pid, 'gw': gw, 'fixture': forecast.fixture,
        'expected_minutes': profile,
        'appearance': round(appearance, 4), 'goals': round(goal_xp, 4),
        'assists': round(assist_xp, 4), 'clean_sheets': round(clean_sheet, 4),
        'goalkeeper_saves': round(saves, 4), 'defensive_contributions': round(defensive, 4),
        'cards': round(cards, 4), 'bonus': round(bonus, 4),
        'conceded_and_other_adjustments': round(adjustments, 4),
        'mean_from_existing_forecast': forecast.xp,
        'role': POS_NAME[pos],
        'penalties_order': player.get('penalties_order'),
        'direct_freekicks_order': player.get('direct_freekicks_order'),
        'corners_and_indirect_freekicks_order': player.get('corners_and_indirect_freekicks_order'),
        'team_attack_defence': 'V4.1 fixture_rates / recent historical fixture goals',
        'opponent_difficulty': forecast.fdr,
        'recency': 'V4.1 season+recent match history blend',
        'availability': player.get('status'),
        'sources': [forecast.source, 'FPL bootstrap-static', 'FPL element-summary when available'],
    }




## Added cell 2 — Per-gameweek uncertainty and rolling optimization
Paired 5,000 scenario simulations share fixture goals. Hold, repair and Wildcard follow legal XV, budget and FT constraints.

In [ ]:
class AuditedMonteCarlo:
    """Paired, correlated draw engine. Samples player outcomes and shared fixture CS results."""

    def __init__(self, engine: Any, sample_count: int = AUDIT_MONTE_CARLO, seed: int = 420097):
        if sample_count < 5000:
            raise ValueError('Audited decision requires at least 5,000 Monte Carlo samples')
        self.engine = engine
        self.n = sample_count
        self.seed = seed
        self.player_cache: dict[tuple[int, int, float], tuple[np.ndarray, np.ndarray, np.ndarray]] = {}
        self.goal_cache: dict[int, tuple[np.ndarray, np.ndarray]] = {}
        self.team_shock: dict[tuple[int, int], np.ndarray] = {}
        self.fixture_map = {integer(x['id']): x for x in engine.snapshot['fixtures'] if x.get('id') is not None}

    def shared_fixture(self, fixture: dict[str, Any]) -> tuple[np.ndarray, np.ndarray]:
        key = integer(fixture['id'])
        if key not in self.goal_cache:
            home_xg, away_xg = self.engine.fixture_rates(fixture)
            rng = np.random.default_rng(np.random.SeedSequence([self.seed, 9, key]))
            home_shock = rng.lognormal(-.5 * .12 ** 2, .12, size=self.n)
            away_shock = rng.lognormal(-.5 * .12 ** 2, .12, size=self.n)
            self.goal_cache[key] = (rng.poisson(max(.01, home_xg) * home_shock),
                                    rng.poisson(max(.01, away_xg) * away_shock))
            home, away = integer(fixture['team_h']), integer(fixture['team_a'])
            self.team_shock[(key, home)] = home_shock
            self.team_shock[(key, away)] = away_shock
        return self.goal_cache[key]

    def player(self, pid: int, gw: int, minute_scale: float = 1.0) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        key = (pid, gw, round(minute_scale, 3))
        if key in self.player_cache:
            return self.player_cache[key]
        games = self.engine.fixtures.get((gw, integer(self.engine.players[pid]['team'])), [])
        total = np.zeros(self.n, dtype=float)
        played = np.zeros(self.n, dtype=bool)
        total_minutes = np.zeros(self.n, dtype=float)
        if not games:
            self.player_cache[key] = (total, played, total_minutes)
            return self.player_cache[key]
        forecast = self.engine.forecasts[(pid, gw)]
        profile = minutes_profile(self.engine, pid, gw, minute_scale)
        stat = self.engine.stats[pid]
        pos = integer(self.engine.players[pid]['element_type'])
        per_game_goals = max(0.0, forecast.xg / len(games))
        per_game_assists = max(0.0, forecast.xa / len(games))
        for fixture, home in games:
            fixture_id = integer(fixture['id'])
            rng = np.random.default_rng(np.random.SeedSequence([self.seed, 17, pid, gw, fixture_id, round(minute_scale * 1000)]))
            start = rng.random(self.n) < profile['p_start']
            substitute = (~start) & (rng.random(self.n) < profile['p_sub_given_bench'])
            appeared = start | substitute
            mins = np.where(start, np.clip(rng.normal(profile['expected_start_minutes'], 9, self.n), 1, 96),
                            np.where(substitute, np.clip(rng.normal(profile['expected_sub_minutes'], 8, self.n), 1, 45), 0))
            p60 = mins >= 60
            appeared_early = mins > 0
            contribution = appeared_early.astype(float) + p60.astype(float)
            home_goals, away_goals = self.shared_fixture(fixture)
            opponent_goals = away_goals if home else home_goals
            cs = opponent_goals == 0
            if pos in (1, 2):
                contribution += 4 * (cs & p60)
                contribution -= np.floor(opponent_goals / 2) * p60
            elif pos == 3:
                contribution += (cs & p60).astype(float)
            expected_minutes_per_game = max(1, minutes_profile(self.engine, pid, gw, 1.0)['expected_minutes'])
            minute_ratio = np.minimum(2, mins / expected_minutes_per_game)
            shock = self.team_shock[(fixture_id, integer(self.engine.players[pid]['team']))]
            goals = rng.poisson(np.maximum(0, per_game_goals * minute_ratio * shock))
            assists = rng.poisson(np.maximum(0, per_game_assists * minute_ratio * shock))
            contribution += {1: 10, 2: 6, 3: 5, 4: 4}[pos] * goals + 3 * assists
            yellow = (rng.random(self.n) < np.minimum(1, stat['yc90'] * mins / 90))
            contribution -= yellow.astype(float)
            red_rate = max(0.0, num(self.engine.players[pid].get('red_cards'))) * 90 / max(500, num(self.engine.players[pid].get('minutes')))
            contribution -= 3 * (rng.random(self.n) < np.minimum(.25, red_rate * mins / 90))
            contribution += rng.poisson(max(0, stat['bonus90']) * mins / 90 * .75)
            if pos == 1:
                saves = rng.poisson(max(0, stat['saves90']) * mins / 90)
                contribution += np.floor(saves / 3)
            if pos in (2, 3, 4):
                # Opponent-adjusted threshold event; paired fixture shock is reused.
                # Unlike a fixed 10% event, this responds to minutes and pressure.
                home_xg,away_xg = self.engine.fixture_rates(fixture)
                opponent_xg = away_xg if home else home_xg
                pressure=clamp(.78+.23*opponent_xg,.60,1.55)
                threshold=10 if pos==2 else 12
                dc_lam=np.maximum(0,stat['dc90']*mins/90*pressure)
                contribution += 2*(rng.poisson(dc_lam)>=threshold)
            total += contribution
            played |= appeared
            total_minutes += mins
        self.player_cache[key] = (total, played, total_minutes)
        return self.player_cache[key]

    def lineup(self, ids: tuple[int, ...], gw: int, minute_scale: float = 1.0, chip: str | None = None) -> np.ndarray:
        line = self.engine.lineup(ids, gw)
        start = line['xi']
        bench = line['bench']
        draws = {pid: self.player(pid, gw, minute_scale) for pid in ids}
        score = sum((draws[pid][0] for pid in start), np.zeros(self.n))
        if chip == 'benchboost':
            return score + sum((draws[pid][0] for pid in bench), np.zeros(self.n)) + draws[line['captain']][0]
        missing_gk = ~draws[start[0]][1]
        score += draws[bench[0]][0] * missing_gk
        field = [pid for pid in start if integer(self.engine.players[pid]['element_type']) != 1]
        for i in range(self.n):
            playing = [pid for pid in field if draws[pid][1][i]]
            position_count = {pos: sum(integer(self.engine.players[pid]['element_type']) == pos for pid in playing) for pos in (2, 3, 4)}
            substitutions = [pid for pid in bench[1:] if draws[pid][1][i]]
            slots = 10 - len(playing)
            for slot in range(slots):
                chosen = None
                for candidate in substitutions:
                    p = integer(self.engine.players[candidate]['element_type'])
                    trial = dict(position_count)
                    trial[p] += 1
                    still_missing = sum(max(0, needed - trial[position]) for position, needed in ((2, 3), (3, 2), (4, 1)))
                    if still_missing <= slots - slot - 1:
                        chosen = candidate
                        break
                if chosen is None:
                    break
                score[i] += draws[chosen][0][i]
                position_count[integer(self.engine.players[chosen]['element_type'])] += 1
                substitutions.remove(chosen)
        captain = line['captain']
        vice = line['vice']
        cap_extra = np.where(draws[captain][1], draws[captain][0], np.where(draws[vice][1], draws[vice][0], 0))
        score += (2 if chip == 'triplecaptain' else 1) * cap_extra
        return score

    def route(self, steps: list[dict[str, Any]], scale: float = 1.0, initial_hit: int = 0) -> np.ndarray:
        scores = np.zeros(self.n)
        for step in steps:
            scores += self.lineup(tuple(step['ids']), integer(step['gw']), minute_scale=scale)
            scores -= integer(step.get('hit', 0))
        return scores - initial_hit

    def player_projections(self, pid: int, gw: int) -> dict[str, Any]:
        points, played, minutes = self.player(pid, gw)
        return {
            **explicit_components(self.engine, pid, gw),
            'mean': round(float(points.mean()), 3),
            'median': round(float(np.median(points)), 3),
            'p25': round(float(np.quantile(points, .25)), 3),
            'p75': round(float(np.quantile(points, .75)), 3),
            'standard_deviation': round(float(points.std()), 3),
            'simulated_play_probability': round(float(np.mean(played)), 4),
            'simulated_minutes_mean': round(float(minutes.mean()), 2),
            'scenarios': self.n,
        }


def hold_route(engine: Any) -> list[dict[str, Any]]:
    ft = integer(engine.ft)
    return [dict(gw=gw, action='HOLD', hit=0, out=None, into=None, ids=engine.owned,
                 bank=engine.bank, free_transfers_before=min(5, ft + index))
            for index, gw in enumerate(engine.events)]


def wc_repair_route(engine: Any, optimal: dict[str, Any], wc_ft: int) -> list[dict[str, Any]]:
    """Run the unchanged V4.1 transfer beam from GW+2; don't allow an extra GW+1 transfer."""
    squad = tuple(sorted(optimal['selected']))
    original_owned, original_bank, original_ft, original_prices, original_budget, original_events = (
        engine.owned, engine.bank, engine.ft, engine.owned_prices, engine.budget, engine.events)
    start = {'gw': original_events[0], 'ids': squad, 'action': 'WILDCARD', 'hit': 0,
             'out': None, 'into': None, 'bank': integer(optimal['remaining_bank']),
             'free_transfers_before': wc_ft}
    try:
        engine.owned = squad
        engine.bank = integer(optimal['remaining_bank'])
        engine.ft = min(5, wc_ft + 1)
        engine.owned_prices = {pid: original_prices.get(pid, integer(engine.players[pid]['now_cost'])) for pid in squad}
        engine.budget = engine.bank + sum(engine.owned_prices.values())
        engine.events = original_events[1:]
        if not engine.events:
            return [start]
        result = engine.plan()
        return [start] + result.steps
    finally:
        (engine.owned, engine.bank, engine.ft, engine.owned_prices,
         engine.budget, engine.events) = (original_owned, original_bank, original_ft, original_prices,
                                          original_budget, original_events)


def route_audit(engine: Any, route_steps: list[dict[str, Any]], verified_ft: int) -> list[Diagnostic]:
    issues: list[Diagnostic] = []
    current_ids = engine.owned
    bank = engine.bank
    balance_ft = verified_ft
    effective_prices = dict(engine.owned_prices)
    for index, step in enumerate(route_steps):
        event = engine.events[index]
        ids = tuple(sorted(step['ids']))
        if not engine.valid_squad(ids) or step['gw'] != event:
            issues.append(Diagnostic('ERROR', 'ROUTE_LEGALITY', f'GW{event} squad or event invalid.'))
        if step['action'] == 'TRANSFER':
            outgoing, incoming = step.get('out'), step.get('into')
            if outgoing not in current_ids or incoming in current_ids or not outgoing or not incoming:
                issues.append(Diagnostic('ERROR', 'ROUTE_TRANSFER', f'GW{event} illegal transfer.'))
            else:
                cost = integer(engine.players[incoming]['now_cost'])
                bank = bank + effective_prices[outgoing] - cost
                effective_prices.pop(outgoing)
                effective_prices[incoming] = cost
                expected_hit = 0 if balance_ft >= 1 else 4
                if step.get('hit') != expected_hit or bank < 0:
                    issues.append(Diagnostic('ERROR', 'ROUTE_BUDGET', f'GW{event} bank or hit mismatch.'))
                balance_ft = max(0, balance_ft - 1)
        elif step['action'] not in ('HOLD', 'ROLL / HOLD'):
            issues.append(Diagnostic('ERROR', 'ROUTE_ACTION', f'Unexpected non-WC route step: GW{event}.'))
        current_ids = ids
        balance_ft = min(5, balance_ft + 1)
    return issues


def median_stats(values: np.ndarray) -> dict[str, float]:
    return {'mean': round(float(np.mean(values)), 2), 'median': round(float(np.median(values)), 2),
            'p25': round(float(np.quantile(values, .25)), 2), 'p75': round(float(np.quantile(values, .75)), 2),
            'standard_deviation': round(float(np.std(values)), 2)}


def get_captains(engine: Any, monte_carlo: AuditedMonteCarlo, cohort: str = 'Global FPL selected_by_percent (NOT top-10k)') -> list[dict[str, Any]]:
    gw = engine.events[0]
    lineup = engine.lineup(engine.owned, gw)
    rows = []
    for pid in lineup['xi']:
        sampled = monte_carlo.player(pid, gw)[0]
        owned = num(engine.players[pid].get('selected_by_percent'), -1)
        # Global selected_by_percent is ownership, not effective ownership; EO unavailable for the cohort.
        rows.append({'id': pid, 'name': engine.name(pid), 'xP': round(engine.xp(pid, gw), 2),
                     'p25': round(float(np.quantile(sampled, .25)), 2),
                     'p75': round(float(np.quantile(sampled, .75)), 2),
                     'global_ownership_percent': owned if owned >= 0 else None,
                     'effective_ownership_percent': None,
                     'eo_reference_cohort': 'Unavailable',
                     'leverage_proxy': round((float(sampled.mean()) - 0.7 * np.std(sampled)) * (1 - max(0, owned) / 100), 3) if owned >= 0 else None,
                     'ownership_reference_cohort': cohort})
    safe = max(rows, key=lambda x: (x['xP'], x['p25']))
    risk_pool = [r for r in rows if r['xP'] >= safe['xP'] - 1.25 and r['id'] != safe['id']]
    aggressive = max(risk_pool, key=lambda x: ((x['leverage_proxy'] or 0), x['p75'])) if risk_pool else safe
    return [{'type': 'Safe', **safe, 'tradeoff': 'Higher raw expected points and more stable downside, regardless of high ownership.'},
            {'type': 'Aggressive', **aggressive, 'tradeoff': 'Near-top xP with higher potential rank leverage; higher variance. Global ownership is not EO.'}]


def adaptive_option_cost(engine: Any) -> dict[str, float]:
    horizon = len(engine.events)
    minutes_risks = sum(engine.forecasts[(pid, engine.events[0])].p_start < .75 for pid in engine.owned)
    fixture_swings = 0
    for pid in engine.owned:
        difficulties = [engine.forecasts[(pid, gw)].fdr for gw in engine.events]
        if len(difficulties) > 2 and max(difficulties) - min(difficulties) >= 2:
            fixture_swings += 1
    # This is a transparent prior on future option value, not observed hidden injury information.
    future_option = 2.5 + .65 * min(minutes_risks, 6) + .45 * min(fixture_swings, 8) + .4 * horizon
    return {'future_wildcard_option_value_points': round(future_option, 2),
            'future_flexibility_estimate': round(1.25 * integer(engine.ft) + .10 * max(0, engine.bank), 2),
            'method': 'Explicit heuristic: option value grows with GW horizon, rotation risk and fixture swing. Sensitivity checked; not fitted/backtested.'}


def compare_strategies(engine: Any, state: dict[str, Any], monte: AuditedMonteCarlo) -> tuple[dict[str, Any], list[Diagnostic]]:
    issues: list[Diagnostic] = []
    hold_steps = hold_route(engine)
    repair = engine.plan()
    issues.extend(route_audit(engine, repair.steps, state['ft_after_moves']))
    if any(d.severity == 'ERROR' for d in issues):
        return {}, issues
    initial_hits = state['displayed_hit_cost']
    hold_samples = monte.route(hold_steps, initial_hit=initial_hits)
    repair_samples = monte.route(repair.steps, initial_hit=initial_hits)
    best_nonchip = 'repair' if np.median(repair_samples) > np.median(hold_samples) else 'hold'
    nonchip_samples = repair_samples if best_nonchip == 'repair' else hold_samples
    option = adaptive_option_cost(engine)
    wc_info: dict[str, Any] = {'available': state['chip_available']['wildcard'], 'feasible': False}
    wc_ft_verified = type(state.get('ft_after_wildcard')) is int
    wc_ft = state['ft_after_wildcard'] if wc_ft_verified else state['ft_after_moves']
    if not wc_ft_verified and state['chip_available']['wildcard']:
        issues.append(Diagnostic('WARNING', 'WC_FT_LOWER_BOUND',
                                  'FT after a hypothetical Wildcard is not privately verifiable before activation; conservatively using current FT count as a lower bound.'))
    wc_samples: np.ndarray | None = None
    wc_steps: list[dict[str, Any]] | None = None
    if state['chip_available']['wildcard']:
        optimal = engine.optimize_squad('wildcard')
        if not optimal.get('success'):
            issues.append(Diagnostic('ERROR', 'WILDCARD_OPTIMIZER_FAILED', f'Cannot compare Wildcard path: {optimal.get("reason", "unknown")}.'))
            return {}, issues
        wc_steps = wc_repair_route(engine, optimal, wc_ft)
        if len(wc_steps) != len(engine.events) or tuple(sorted(wc_steps[0]['ids'])) != tuple(sorted(optimal['selected'])):
            issues.append(Diagnostic('ERROR', 'WC_FIRST_WEEK', 'Wildcard route does not start with the optimized XV.'))
            return {}, issues
        # Post-WC GW2+ repair planner is exploratory; GW1 must be the exact optimized WC XV.
        wc_samples = monte.route(wc_steps)
        wc_info = {'available': True, 'feasible': True, 'selected_ids': list(optimal['selected']),
                   'bank_after_wildcard_tenths': optimal['remaining_bank'], 'wildcard_future_ft_assumption': wc_ft,
                   'wildcard_future_ft_verified': wc_ft_verified, 'first_week_action': 'Wildcard',
                   'first_week_lineup': engine.lineup(tuple(optimal['selected']), engine.events[0]),
                   'path': [{'gw': step['gw'], 'action': step['action'], 'out': step['out'], 'into': step['into'], 'hit': step['hit']} for step in wc_steps],
                   'path_full': wc_steps,
                   'solver_status': optimal.get('status'), 'exact_within_candidate_pool': optimal.get('exact_within_candidate_pool')}
        if not engine.valid_squad(tuple(optimal['selected'])):
            issues.append(Diagnostic('ERROR', 'WC_ILLEGAL', 'Wildcard XV violates FPL squad rules.'))
        if sum(engine.owned_prices.get(pid, integer(engine.players[pid]['now_cost'])) for pid in optimal['selected']) + optimal['remaining_bank'] != engine.budget:
            issues.append(Diagnostic('ERROR', 'WC_BUDGET', 'Wildcard optimized budget does not reconcile.'))
    deltas: dict[str, Any] = {'wildcard_less_best_no_chip': None}
    stability = []
    recommendation = ('TRANSFER' if best_nonchip == 'repair' and repair.steps[0]['action'] == 'TRANSFER' else 'HOLD')
    confidence = 'Medium'
    if wc_samples is not None:
        delta = wc_samples - nonchip_samples
        opt_cost = option['future_wildcard_option_value_points']
        baseline_delta = median_stats(delta)
        baseline_delta['p10'] = round(float(np.quantile(delta, .10)), 2)
        baseline_delta['p90'] = round(float(np.quantile(delta, .90)), 2)
        win_probability = float(np.mean(delta > 0))
        negative_probability = float(np.mean(delta < 0))
        for scenario_name, scale in (('conservative', .85), ('base', 1.0), ('aggressive', 1.15)):
            a = monte.route(wc_steps, scale=scale)
            b = monte.route(repair.steps if best_nonchip == 'repair' else hold_steps, scale=scale, initial_hit=initial_hits)
            conservative_gain = float(np.median(a - b)) - opt_cost
            stability.append({'minutes_assumption': scenario_name, 'start_probability_scale': scale,
                              'wc_net_median_gain': round(conservative_gain, 2), 'supports_wildcard': conservative_gain > AUDIT_MIN_MEDIAN_SURPLUS_PTS})
        stable_count = sum(row['supports_wildcard'] for row in stability)
        net_median = baseline_delta['median'] - opt_cost
        should_wc = (net_median > AUDIT_MIN_MEDIAN_SURPLUS_PTS and win_probability >= AUDIT_MIN_WILDCARD_WIN_RATE
                     and stable_count >= AUDIT_MIN_STABILITY and baseline_delta['p25'] > -8)
        if should_wc:
            recommendation = 'WILDCARD'
        elif best_nonchip == 'repair' and repair.steps[0]['action'] == 'TRANSFER':
            recommendation = 'TRANSFER'
        else:
            recommendation = 'HOLD'
        confidence = ('High' if should_wc and win_probability >= .80 and baseline_delta['p25'] > 0
                      else 'Medium' if (win_probability >= .60 and stable_count >= 2) or (win_probability <= .40 and stable_count <= 1)
                      else 'Low')
        deltas['wildcard_less_hold'] = {**median_stats(wc_samples - hold_samples),
                                          'p_win': round(float(np.mean(wc_samples > hold_samples)), 4),
                                          'p_negative_gain': round(float(np.mean(wc_samples < hold_samples)), 4)}
        deltas['wildcard_less_best_no_chip'] = {**baseline_delta,
                                              'p_win': round(win_probability, 4),
                                              'p_negative_gain': round(negative_probability, 4),
                                              'median_less_wc_opportunity_cost': round(net_median, 2)}
    if not wc_info['available']:
        confidence = 'Medium'
    # Missing current-player match histories make the minute model weak even with 5,000 draws.
    no_recent = sum(not engine.stats[pid]['history_available'] for pid in engine.owned)
    if no_recent >= 4:
        confidence = 'Low'
        issues.append(Diagnostic('WARNING', 'LIMITED_MINUTES_HISTORY', f'{no_recent}/15 players lack recent match logs; confidence capped at Low.'))
    paths = {'hold': median_stats(hold_samples), 'repair': median_stats(repair_samples),
             'wildcard': median_stats(wc_samples) if wc_samples is not None else None}
    weekly_comparison = []
    for index, gw in enumerate(engine.events):
        hold_draws = monte.lineup(tuple(hold_steps[index]['ids']), gw) - (initial_hits if index == 0 else 0)
        repair_draws = monte.lineup(tuple(repair.steps[index]['ids']), gw) - integer(repair.steps[index]['hit']) - (initial_hits if index == 0 else 0)
        wc_draws = (monte.lineup(tuple(wc_steps[index]['ids']), gw) - integer(wc_steps[index]['hit'])) if wc_steps else None
        weekly_comparison.append({'gw': gw, 'hold_mean': round(float(np.mean(hold_draws)), 2),
                                  'hold_median': round(float(np.median(hold_draws)), 2),
                                  'repair_mean': round(float(np.mean(repair_draws)), 2),
                                  'repair_median': round(float(np.median(repair_draws)), 2),
                                  'wildcard_mean': round(float(np.mean(wc_draws)), 2) if wc_draws is not None else None,
                                  'wildcard_median': round(float(np.median(wc_draws)), 2) if wc_draws is not None else None})
    default_action = {'gw': engine.events[0], 'action': 'HOLD'}
    if recommendation == 'TRANSFER':
        default_action = {'gw': engine.events[0], 'action': 'TRANSFER', 'out': repair.steps[0]['out'],
                          'into': repair.steps[0]['into'], 'out_ids':repair.steps[0].get('out_ids'), 'in_ids':repair.steps[0].get('in_ids'), 'hit': repair.steps[0]['hit']}
    elif recommendation == 'WILDCARD':
        default_action = {'gw': engine.events[0], 'action': 'WILDCARD', 'selected': wc_info['selected_ids']}
    candidate_alternatives = {'HOLD': paths['hold']['median'], 'REPAIR': paths['repair']['median']}
    if wc_info['feasible']:
        candidate_alternatives['WILDCARD'] = paths['wildcard']['median'] - option['future_wildcard_option_value_points']
    chosen_path = ('WILDCARD' if recommendation == 'WILDCARD' else ('REPAIR' if recommendation == 'TRANSFER' else 'HOLD'))
    alternatives = {name: score for name, score in candidate_alternatives.items() if name != chosen_path}
    next_best = max(alternatives, key=alternatives.get) if alternatives else 'HOLD'
    return {
        'recommended_action': recommendation, 'current_action_only': default_action,
        'best_alternative': next_best if recommendation != next_best else 'HOLD',
        'next_gw': engine.events[0], 'horizon_gws': engine.events,
        'comparison': paths, 'weekly_comparison': weekly_comparison, 'delta': deltas, 'best_no_chip': best_nonchip,
        'future_flexibility_score': {'hold': round(min(5, integer(engine.ft) + len(engine.events)) * 1.25 + .10 * engine.bank, 2),
                                     'repair': round(max(0, integer(repair.ft)) * 1.25 + .10 * repair.bank, 2),
                                     'wildcard': round(min(5, wc_ft + len(engine.events)) * 1.25 + .10 * wc_info.get('bank_after_wildcard_tenths', 0), 2) if wc_info['feasible'] else None},
        'wildcard': wc_info, 'option_cost': option, 'minutes_sensitivity': stability,
        'captains': get_captains(engine, monte),
        'main_assumptions': ['Unverified post-Wildcard FT rollover is conservatively approximated from current displayed FT' if not wc_ft_verified else 'Post-Wildcard FT count explicitly verified', 'FPL displayed team value and immediate selling-price budget are distinct; WC uses selling prices', 'FPL pre-deadline current state must be independently manager-verified',
                             'Minutes from recent starts/official flags with ±15% minute-sensitivity test',
                             'Fixtures from official FPL; goal models not independently calibrated',
                             'Wildcard future option value is a labelled heuristic',
                             'Multi-period MILP uses limited pool and fixed future prices; legacy beam fallback only on solver failure; global optimality not guaranteed'],
        'main_risks': ['Lineup news/rotation changes before deadline', 'Forecast and clean sheet misspecification',
                       'Transfer paths beyond the current GW are provisional', 'Incomplete injury, penalty-role and effective-ownership feeds'],
        'confidence': confidence, 'scenario_count': monte.n,
        'probabilistic_not_guaranteed': 'This is a probabilistic recommendation, not a guaranteed outcome.',
    }, issues




## Added cell 3 — Diagnostics, audited HTML tab and legacy rendering bridge
All assertions complete before 4K/HTML output. An older dashboard is **not** proof a failed run succeeded.

In [ ]:
def audit_chip_view(engine: Any, state: dict[str, Any], decision: dict[str, Any]) -> dict[str, Any]:
    baseline = decision['comparison'][decision['best_no_chip']]['mean']
    chips: dict[str, Any] = {}
    wc = decision['wildcard']
    delta = decision['delta']['wildcard_less_best_no_chip']
    for chip in ('wildcard', 'freehit', 'triplecaptain', 'benchboost'):
        entry = {'available': state['chip_available'][chip], 'gw': engine.events[0],
                 'mean_gain': None, 'p10_gain': None, 'p90_gain': None,
                 'positive_probability': None, 'hold_reason': 'Not evaluated by this audited hold/repair/Wildcard comparison',
                 'in_ids': [], 'out_ids': []}
        if chip == 'wildcard' and wc['feasible'] and delta is not None:
            entry.update({'mean_gain': delta['mean'], 'p10_gain': delta['p10'], 'p90_gain': delta['p90'],
                          'positive_probability': delta['p_win'], 'selected_ids': wc['selected_ids'],
                          'in_ids': sorted(set(wc['selected_ids']) - set(engine.owned)),
                          'out_ids': sorted(set(engine.owned) - set(wc['selected_ids'])),
                          'has_haaland': any(_squashed_name(engine.name(pid)) == 'haaland' for pid in wc['selected_ids'])})
        chips[chip] = entry
    return {'availability': state['chip_available'], 'normal_6gw': baseline,
            'simulation_count': decision['scenario_count'], 'scenarios': chips,
            'recommendation': ('PLAY WILDCARD (AUDITED)' if decision['recommended_action'] == 'WILDCARD'
                               else f'{decision["recommended_action"]} — HOLD WILDCARD (AUDITED)'),
            'optimal': {'selected': tuple(wc.get('selected_ids', []))} if wc['feasible'] else None}


def assemble_existing_dashboard_result(engine: Any, state: dict[str, Any], decision: dict[str, Any]) -> dict[str, Any]:
    # Preserve V4.1 make_result() and renderer without running its uncalibrated chip decision twice.
    original_chip_solver = engine.chip_solver
    try:
        engine.chip_solver = lambda _route: audit_chip_view(engine, state, decision)
        result = make_result(engine)
    finally:
        engine.chip_solver = original_chip_solver
    result['audited_decision'] = decision
    result['free_transfers_assumed'] = state['ft_after_moves']
    result['free_transfers_verified'] = True
    result['ownership_current_verified'] = True
    result['bank_verified'] = True
    result['verified_state_timestamp_utc'] = state['verified_at_utc']
    result['transfer_audit'] = {name: state[name] for name in ('bank_tenths', 'team_value_tenths', 'ft_before_moves',
                                                               'transfers_made_this_gw', 'displayed_hit_cost',
                                                               'ft_after_moves', 'ft_after_wildcard', 'chip_available')}
    result['recommended_squad_ids'] = (decision['wildcard']['selected_ids'] if decision['recommended_action'] == 'WILDCARD'
                                       else list(engine.owned))
    return result


def html_audit_panel(decision: dict[str, Any], state: dict[str, Any], raw_at: str, source_status: Mapping[str, Any], diagnostics: list[Diagnostic]) -> str:
    def fmt(value: Any) -> str:
        return esc(value if value is not None else 'Not available')
    def points_row(name: str, item: dict[str, float] | None) -> str:
        if item is None:
            return f'<tr><td>{esc(name)}</td><td colspan="4">Unavailable</td></tr>'
        return f'<tr><td>{esc(name)}</td><td>{item["mean"]:+.2f}</td><td>{item["median"]:+.2f}</td><td>{item["p25"]:+.2f}</td><td>{item["p75"]:+.2f}</td></tr>'
    summary = decision['delta']['wildcard_less_best_no_chip']
    difference = (f'<p><b>Wildcard – best no chip:</b> median {summary["median"]:+.2f} points, '
                  f'P25–P75 {summary["p25"]:+.2f} to {summary["p75"]:+.2f}; '
                  f'P(Wildcard wins) {summary["p_win"]:.0%}; P(negative gain) {summary["p_negative_gain"]:.0%}.</p>') if summary else '<p>Wildcard currently unavailable or not evaluated.</p>'
    sensitivity = ''.join(f'<tr><td>{esc(x["minutes_assumption"])}</td><td>{x["start_probability_scale"]:.2f}×</td><td>{x["wc_net_median_gain"]:+.2f}</td><td>{"Supports WC" if x["supports_wildcard"] else "Hold"}</td></tr>' for x in decision['minutes_sensitivity'])
    weekly = ''.join(f'<tr><td>GW{row["gw"]}</td><td>{row["hold_mean"]:.2f}</td><td>{row["repair_mean"]:.2f}</td><td>{fmt(row["wildcard_mean"])}</td></tr>' for row in decision['weekly_comparison'])
    captains = ''.join(f'<tr><td>{fmt(row["type"])}</td><td>{fmt(row["name"])}</td><td>{row["xP"]:.2f}</td><td>{row["p25"]:.1f}</td><td>{fmt(row["global_ownership_percent"])}</td><td>{fmt(row["effective_ownership_percent"])}</td><td>{fmt(row["tradeoff"])}</td></tr>' for row in decision['captains'])
    audit = f'''<section class="page" id="page-auditdecision" role="tabpanel"><h1 class="headline">Verified decision & uncertainty audit</h1>
    <div class="card" style="border-color:#27e5c2"><h2>Current action: {fmt(decision['recommended_action'])} · Confidence: {fmt(decision['confidence'])}</h2>
    <p><b>Verified current manager state:</b> {fmt(state['verified_at_utc'])} UTC; <b>official fetch:</b> {fmt(raw_at)} UTC.</p>
    <p><b>Best alternative:</b> {fmt(decision['best_alternative'])}. <b>Only the first GW action is recommended</b>; re-run before every deadline.</p>
    <p><b>FT before:</b> {fmt(state['ft_before_moves'])} · <b>moves already made:</b> {fmt(state['transfers_made_this_gw'])} · <b>displayed hits:</b> {fmt(state['displayed_hit_cost'])} pts · <b>FT after:</b> {fmt(state['ft_after_moves'])} · <b>bank:</b> £{state['bank_tenths']/10:.1f}m · <b>team value:</b> £{state['team_value_tenths']/10:.1f}m</p>
    <p><b>Projections cover GW{decision['horizon_gws'][0]}–GW{decision['horizon_gws'][-1]}.</b> {fmt(decision['probabilistic_not_guaranteed'])}</p>
    </div><div class="card"><h2>Paired Monte Carlo, {decision['scenario_count']:,} scenarios</h2><div class="table-scroll"><table><thead><tr><th>Path</th><th>Mean</th><th>Median</th><th>P25</th><th>P75</th></tr></thead><tbody>
    {points_row('Hold',decision['comparison']['hold'])}{points_row('Repair (transfers)',decision['comparison']['repair'])}{points_row('Wildcard',decision['comparison']['wildcard'])}</tbody></table></div>
    {difference}<p><b>Wildcard future-option cost:</b> {decision['option_cost']['future_wildcard_option_value_points']:+.1f} pts; this is explicitly a heuristic, not bookmaker or empirical evidence. Never play WC for a trivial positive xP edge.</p></div>
    <div class="card"><h2>Gameweek-by-gameweek projected points (mean, after hit costs)</h2><div class="table-scroll"><table><tr><th>Gameweek</th><th>Hold</th><th>Repair</th><th>Wildcard</th></tr>{weekly}</table></div></div>
    <div class="card"><h2>Expected-minutes sensitivity</h2><div class="table-scroll"><table><tr><th>Assumption</th><th>Start probability</th><th>Median edge after option cost</th><th>Verdict</th></tr>{sensitivity}</table></div></div>
    <div class="card"><h2>Captain choices</h2><div class="table-scroll"><table><tr><th>Style</th><th>Player</th><th>xP</th><th>P25</th><th>Global ownership %</th><th>EO</th><th>Trade-off</th></tr>{captains}</table></div>
    <p class="mini">Global FPL selected_by_percent is not effective ownership of the top-10K. Safe selection does not penalize high ownership.</p></div>
    <div class="card"><h2>Inputs, assumptions, risks and source status</h2><p><b>Official API:</b> {fmt(source_status.get('official_fpl'))}; <b>private state:</b> {fmt(source_status.get('private_state'))}; <b>news overrides:</b> {fmt(source_status.get('verified_news'))}; <b>bookmaker odds:</b> {fmt(source_status.get('odds'))}; <b>social media:</b> {fmt(source_status.get('rumors'))}.</p>
    <p><b>Main assumptions:</b> {fmt('; '.join(decision['main_assumptions']))}.</p><p><b>Main risks:</b> {fmt('; '.join(decision['main_risks']))}.</p>
    <p><b>Diagnostics:</b> {fmt('; '.join(f'{d.code}: {d.message}' for d in diagnostics) or 'All blocking checks passed')}.</p>
    <p class="mini">No projected rank finish is guaranteed. This report does not establish unconfirmed X lineups as fact.</p></div></section>'''
    return audit


def attach_audit_tab(original_html: str, panel: str, recommendation: str, decision: dict[str, Any]) -> str:
    tab = '<button class="tab" role="tab" type="button" data-tab="auditdecision" aria-selected="false">Verified Plan & Audit</button>'
    marker = '</nav></header>'
    if marker not in original_html or 'id="page-audit"' not in original_html:
        raise RuntimeError('Original V4.1 tab layout changed: refusing to inject into unknown HTML structure.')
    original_html = original_html.replace(marker, tab + marker, 1)
    original_html = original_html.replace('<section class="page" id="page-audit"', panel + '<section class="page" id="page-audit"', 1)
    banner = ('<div class="alert" style="border-color:#20e6c5"><strong>AUDITED ACTION: '
              + esc(recommendation) + '</strong> · verified FPL state · 5,000 paired scenarios. '
              + '<button class="button alt" type="button" data-go="auditdecision">See complete audited evidence →</button></div>')
    original_html = original_html.replace('Automatic Wildcard recommendation — current GW7 screenshot squad', 'Audited Wildcard recommendation — verified current squad')
    original_html = original_html.replace('GW7 screenshot transfers', 'verified current-gameweek transfers')
    original_html = original_html.replace('<section class="page active" id="page-overview"', banner + '<section class="page active" id="page-overview"', 1)
    # Remove the V4.1 browser's synthetic 1,500-draw normal re-sampling and show
    # the actual stored 5,000 paired Monte Carlo result instead, without a new API call.
    delta = decision['delta'].get('wildcard_less_best_no_chip')
    javascript_data = json.dumps({'wildcard_delta': delta, 'mc_count': decision['scenario_count']},
                                  ensure_ascii=False, separators=(',', ':')).replace('</', '<\\/')
    audited_script = """<script>
(function(){'use strict';
  const AUDIT_PAYLOAD = __DATA__;
  const output = document.getElementById('sim-output');
  const format = (value) => (Number.isFinite(value) ? value.toFixed(2) : 'N/A');
  const describe = () => {
    const item = AUDIT_PAYLOAD.wildcard_delta;
    if (!item) return 'Wildcard is unavailable or has not passed audited optimization.';
    return '<b>Verified paired Monte Carlo (not a browser re-sample)</b><br>Scenarios: '
      + AUDIT_PAYLOAD.mc_count + ' · Wildcard − best no-chip median: '
      + format(item.median) + ' pts · P25/P75: ' + format(item.p25) + ' / ' + format(item.p75)
      + ' · P(WC wins): ' + format(item.p_win*100) + '% · P(negative gain): '
      + format(item.p_negative_gain*100) + '%.';
  };
  for (const button of document.querySelectorAll('[data-sim]')) {
    const clone = button.cloneNode(true);
    button.parentNode.replaceChild(clone, button);
    clone.addEventListener('click', () => {
      output.textContent = '';
      if (clone.dataset.sim === 'wildcard') output.innerHTML = describe();
      else output.textContent = 'This chip has not been evaluated by the audited 6-GW model. No invented result.';
    });
  }
  const original = document.getElementById('compare-chips');
  if (original) {
    const clone = original.cloneNode(true);
    original.parentNode.replaceChild(clone,original);
    clone.addEventListener('click', () => {output.innerHTML = describe();});
  }
})();
</script>""".replace('__DATA__', javascript_data)
    if '</body></html>' not in original_html:
        raise RuntimeError('Cannot safely attach audited chip simulation behavior')
    original_html = original_html.replace('</body></html>', audited_script+'</body></html>', 1)
    return original_html


def write_diagnostics(destination: Path, diagnostics: list[Diagnostic], data_sources: dict[str, Any], *, status: str) -> tuple[Path, Path]:
    report = {'version': AUDIT_VERSION, 'generated_utc': utc_now().isoformat(), 'status': status,
              'diagnostics': [asdict(d) for d in diagnostics], 'sources': data_sources,
              'recommendation_blocked': status not in ('validated','public-provisional'), 'provisional': status == 'public-provisional'}
    path = destination / 'FPLVORTEX_LATEST_DIAGNOSTICS.json'
    atomic_json(path, report)
    colors = {'ERROR': '#ffad98', 'WARNING': '#ffdf82', 'INFO': '#58f0d4'}
    rows = ''.join(f'<li style="color:{colors.get(d.severity, "white")}"><b>{esc(d.severity)} · {esc(d.code)}</b>: {esc(d.message)}</li>' for d in diagnostics)
    html_file = destination / 'FPLVORTEX_LATEST_DIAGNOSTICS.html'
    html_file.write_text('<!doctype html><html><meta charset="utf-8"><title>FPLVORTEX Diagnostics</title><body style="background:#061524;color:white;font:17px Arial;padding:40px"><h1>FPLVORTEX · AUDIT STATUS: '
                         + esc(status.upper()) + '</h1><p>No new transfer or chip recommendation is issued when required data is unavailable.</p><ul>' + rows
                         + '</ul><p>Public source diagnostics are automatic. No manual data entry is required.</p></body></html>', encoding='utf-8')
    atomic_json(destination / 'FPLVORTEX_LATEST_STATUS.json', {'status': status, 'diagnostics': path.name, 'generated_utc': report['generated_utc']})
    return path, html_file


def export_player_projections(monte: AuditedMonteCarlo, destination: Path, run_id: str) -> Path:
    """Versioned per-player, per-GW JSONL avoids giant in-memory DataFrames."""
    archive = destination / 'Archive'
    archive.mkdir(parents=True, exist_ok=True)
    path = archive / f'projections_{run_id}.jsonl'
    with path.open('x', encoding='utf-8') as stream:
        for gw in monte.engine.events:
            for pid in sorted(monte.engine.players):
                output = monte.player_projections(pid, gw)
                stream.write(json.dumps(output, ensure_ascii=False, separators=(',', ':')) + '\n')
    return path



# V4.6: automatic model research brief, independent of the verification gate.
# The last-published public squad is a modelling input, NOT the user's current private squad.
def automatic_decision_brief(result: dict[str, Any], engine: Any, state: dict[str, Any],
                             decision: dict[str, Any], issues: list[Diagnostic]) -> dict[str, Any]:
    public = bool(state.get('public_provisional'))
    gw = integer(engine.events[0])
    candidate = decision.get('research_candidate_action', decision.get('recommended_action', 'HOLD'))
    published_gw = state.get('published_gw', None)
    repair = (result.get('plan') or [{}])[0]
    def named(ids: Any) -> list[str]:
        return [engine.name(integer(pid)) for pid in (ids or [])]
    def pick_xi(owned: Any) -> dict[str, Any]:
        line = engine.lineup(tuple(owned), gw)
        return {'formation': line['formation'], 'xi': named(line['xi']),
                'bench': named(line['bench']), 'captain': engine.name(line['captain']),
                'vice': engine.name(line['vice']), 'expected_points': round(float(line['expected']), 2)}
    hold = pick_xi(engine.owned)
    mc = decision.get('comparison', {})
    transfer = {
        'moves_out': named(repair.get('out_ids', [])), 'moves_in': named(repair.get('in_ids', [])),
        'hit_points_assumed': integer(repair.get('hit', 0)),
        'free_transfers_assumed': integer(state['ft_after_moves']),
        'bank_after_moves_tenths_estimated': integer(repair.get('bank', state['bank_tenths'])),
        'candidate': repair.get('action', 'HOLD'),
        'first_week_model_points': round(float(repair.get('xp', hold['expected_points'])), 2),
        'valid_for_current_private_account': not public,
    }
    wc = decision.get('wildcard', {})
    wildcard = {'available_from_history_or_verified': bool(state['chip_available']['wildcard']),
                'model_feasible': bool(wc.get('feasible')), 'squad': named(wc.get('selected_ids', [])),
                'captain': engine.name(wc['first_week_lineup']['captain']) if wc.get('first_week_lineup') else None,
                'vice': engine.name(wc['first_week_lineup']['vice']) if wc.get('first_week_lineup') else None,
                'median_gain_vs_best_no_chip': (decision.get('delta', {}).get('wildcard_less_best_no_chip') or {}).get('median'),
                'p25_gain': (decision.get('delta', {}).get('wildcard_less_best_no_chip') or {}).get('p25'),
                'p75_gain': (decision.get('delta', {}).get('wildcard_less_best_no_chip') or {}).get('p75')}
    chip = decision.get('chip_calendar_10gw') or {}
    completeness = chip.get('candidate_completeness') or {}
    incomplete = (completeness.get('wildcard', 0) < completeness.get('expected_wildcard', 0))
    brief = {
        'version': '4.6-auto-conditional', 'gameweek': gw,
        'mode': 'PROVISIONAL – research only' if public else 'VERIFIED – current-week advice',
        'executable': bool(decision.get('executable', False) and not public),
        'model_candidate': candidate,
        'chip_timing_fully_evaluated': not incomplete,
        'published_squad_gw': published_gw, 'last_published_bank_tenths': state['bank_tenths'] if public else None,
        'free_transfers_verified': not public, 'selling_prices_verified': not public,
        'headline': (f'Conditional GW{gw} model comparison — based on GW{published_gw} published squad'
                     if public else f'GW{gw} verified-state model candidates'),
        'hold': {'mean_10gw': (mc.get('hold') or {}).get('mean'), 'lineup': hold},
        'no_chip_transfer_path': {'mean_10gw': (mc.get('repair') or {}).get('mean'), **transfer},
        'wildcard': {'mean_10gw': (mc.get('wildcard') or {}).get('mean'), **wildcard},
        'chip_calendar': [{'gw': x['gw'], 'chip': x['chip'], 'median_gain': x.get('median_gain')}
                          for x in chip.get('recommended_chip_calendar', [])],
        'diagnostic_warnings': [{'code': d.code, 'message': d.message} for d in issues if d.severity != 'ERROR'],
        'blocked_executable_reason': (None if not public else
            'The official FPL public API has only last-locked picks and chip history; it cannot reveal transfers '
            'pending before deadline, exact selling prices, bank after pending transfers or true free transfers.'),
    }
    return brief


def automatic_brief_panel(brief: dict[str, Any]) -> str:
    def fmt(v: Any) -> str:
        return esc(v if v is not None else 'not available')
    def names(v: list[str]) -> str:
        return fmt(', '.join(v) if v else 'none')
    public = not brief['executable']
    mode = ('⚠ PROVISIONAL – research only · NO executable transfer/chip advice'
            if public else 'Verified model — review before deadline')
    t = brief['no_chip_transfer_path']; w = brief['wildcard']; h = brief['hold']
    move = (f'Sell {names(t["moves_out"])}; buy {names(t["moves_in"])}; '
            f'modelled hit {t["hit_points_assumed"]} pts, assumed FTs {t["free_transfers_assumed"]}'
            if t['candidate'] == 'TRANSFER' else 'Hold your squad; no transfers in the route this week')
    wc_squad = names(w['squad']) if w['model_feasible'] else 'No valid Wildcard squad produced'
    chip_rows = ''.join(f'<tr><td>GW{fmt(x["gw"])}</td><td>{fmt(x["chip"])}</td><td>{fmt(x["median_gain"])}</td></tr>'
                        for x in brief['chip_calendar'])
    chip_rows = chip_rows or '<tr><td colspan="3">No chip clears the forecast gates.</td></tr>'
    warn = '' if brief['chip_timing_fully_evaluated'] else (
        '<p style="color:#ffc77d"><b>CHIP TIMING INCOMPLETE:</b> at least one eligible future week failed validation. '
        'These are only evaluated candidates; they are not an optimized chip schedule.</p>')
    return (f'<div class="card" id="auto-research-brief" style="border:2px solid #f0bc63;background:#192333">'
        f'<h2>AUTOMATIC FPL RESEARCH BRIEF · GW{fmt(brief["gameweek"])}</h2>'
        f'<p style="font-weight:800;color:#ffcd84">{mode}</p>'
        f'<p><b>Model candidate:</b> {fmt(brief["model_candidate"])} '
        f'({"conditional research case" if public else "verified-input model output"}).</p>'
        f'<p><b>Squad basis:</b> published GW{fmt(brief["published_squad_gw"])}; '
        f'<b>bank basis:</b> £{brief["last_published_bank_tenths"]/10:.1f}m at last deadline. '
        f'No live private transfers are visible to the public API.</p>' if public else
        f'<div class="card" id="auto-research-brief" style="border:2px solid #27e5c2">'
        f'<h2>AUTOMATIC VERIFIED RESEARCH BRIEF · GW{fmt(brief["gameweek"])}</h2>'
        f'<p><b>Model candidate:</b> {fmt(brief["model_candidate"])}</p>') + (
        f'<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:10px">'
        f'<div class="small-card"><b>HOLD — no new transfers</b><p>10-GW mean {fmt(h["mean_10gw"])} pts. '
        f'Captain {fmt(h["lineup"]["captain"])}; vice {fmt(h["lineup"]["vice"])}; '
        f'formation {fmt(h["lineup"]["formation"])}.</p></div>'
        f'<div class="small-card"><b>TRANSFER PATH — alternative</b><p>{move}.</p>'
        f'<p>10-GW mean {fmt(t["mean_10gw"])} pts (with modelled hits).</p></div>'
        f'<div class="small-card"><b>WILDCARD — evaluated candidate</b><p>10-GW mean {fmt(w["mean_10gw"])} pts; '
        f'median improvement vs best no-chip {fmt(w["median_gain_vs_best_no_chip"])} pts '
        f'(P25–P75 {fmt(w["p25_gain"])}–{fmt(w["p75_gain"])}).</p>'
        f'<p>Model squad: {wc_squad}.</p>'
        f'<p>Captain {fmt(w["captain"])}; vice {fmt(w["vice"])}.</p></div></div>'
        '<h3>10-GW provisional chip ideas</h3><div class="table-scroll"><table><tr><th>GW</th><th>Model chip candidate</th><th>Median edge</th></tr>'
        +chip_rows+'</table></div>'+warn+
        ('<p><b>Research-only warning:</b> These are genuine automatic model outputs but are NOT confirmed '
         'instructions for your current team. Numbers depend on the outdated published squad, assumed bank, '
         'estimated selling prices and unobserved free transfers. Re-running does not make those hidden '
         'values observable. Read the diagnostics for uncertainties.</p>' if public else
         '<p>Model predictions are uncertain and never guaranteed. Verify news and fixture changes before acting.</p>')+
        '</div>')


def run_audited(output: Path | None = None, snapshot: dict[str, Any] | None = None,
                private_state_override: dict[str, Any] | None = None,
                *, offline_test_mode: bool = False) -> tuple[Path, Path, Path]:
    """Replaces only the notebook's *last* run() invocation; V4.1 engine/renderers stay intact."""
    destination = output if output is not None else google_drive_folder()
    destination.mkdir(parents=True, exist_ok=True)
    source_status: dict[str, Any] = {'official_fpl': 'unavailable', 'private_state': 'unavailable',
                                     'verified_news': 'unavailable', 'odds': 'unavailable', 'rumors': 'not independently verified'}
    diags: list[Diagnostic] = []
    try:
        if snapshot is not None and not offline_test_mode:
            raise AuditBlocked([Diagnostic('ERROR', 'FAKE_DATA_PROHIBITED', 'Manually passed snapshots may only be used in explicitly marked offline tests.')])
        if snapshot is None:
            globals()['SUMMARY_CACHE_HOURS'] = 0  # Require fresh player-summary requests for the audited run.
        raw = snapshot if snapshot is not None else collect(destination)
        fetched_at = utc_now()
        source_status['official_fpl'] = 'OFFLINE TEST FIXTURE ONLY' if offline_test_mode else f'Fetched {fetched_at.isoformat()}'
        raw_path, raw_hash = save_raw_archive(raw, destination, fetched_at)
        diags.extend(settle_previous_replays(raw, destination / 'Archive', fetched_at, allow_network=not offline_test_mode))
        events = determine_events(raw['bootstrap'], fetched_at)
        input_folder = destination.parent / 'Input'
        input_folder.mkdir(parents=True, exist_ok=True)
        if offline_test_mode:
            private_path = input_folder / 'manager_state_OFFLINE_TEST.json'
            if private_state_override is None:
                raise AuditBlocked([Diagnostic('ERROR','OFFLINE_STATE_REQUIRED','Offline tests require explicit test state.')])
            atomic_json(private_path, private_state_override)
            state, issues = load_private_state(private_path, raw, events, fetched_at)
        else:
            if private_state_override is not None:
                raise AuditBlocked([Diagnostic('ERROR','PRIVATE_OVERRIDE_PROHIBITED', 'Live mode cannot accept an injected unverified private state.')])
            private_path = input_folder / 'manager_state.json'
            if private_path.exists():
                verified, private_issues = load_private_state(private_path, raw, events, fetched_at)
            else:
                verified, private_issues = None, [Diagnostic('WARNING','VERIFIED_STATE_MISSING',
                    'No manager_state.json. Running public-only research automatically.')]
            # Invalid or stale private state must NEVER authorize a transfer/chip action.
            # Continue with a visibly labelled public research report instead of
            # stopping the model, and retain the complete private-state diagnostics.
            if verified is not None and not any(d.severity == 'ERROR' for d in private_issues):
                state, issues = verified, private_issues
            else:
                state, issues = derive_public_manager_state(raw, events, fetched_at)
                issues += [Diagnostic('WARNING', 'PRIVATE_GATE_' + d.code, d.message)
                           for d in private_issues]
                issues.append(Diagnostic('WARNING','RESEARCH_ONLY',
                    'PROVISIONAL – research only: executable transfers/chips need a fully verified manager_state.json.'))
        diags.extend(issues)
        if state is None or any(d.severity == 'ERROR' for d in issues):
            raise AuditBlocked(diags)
        state_path, state_hash = save_state_archive(state, destination, fetched_at)
        source_status['private_state'] = ('PROVISIONAL – research only; official public last-locked GW state'
                                          if state.get('public_provisional') else
                                          ('OFFLINE TEST - not executable' if offline_test_mode else
                                           f'VERIFIED manager_state.json checked {state["verified_at_utc"]}'))
        data = apply_verified_state(raw, state)
        data['news_radar'] = collect_news_radar(data, fetch_external=not offline_test_mode)
        data['markets'] = collect_polymarket_context(data, fetch_external=not offline_test_mode)
        # Reconcile live private values against V4.1's engine and do not use its obsolete screenshot switches.
        globals()['FREE_TRANSFERS_ASSUMED'] = state['ft_after_moves']
        globals()['BANK_OVERRIDE_TENTHS'] = state['bank_tenths']
        globals()['SIMULATIONS'] = AUDIT_MONTE_CARLO
        globals()['USE_GW7_SCREENSHOT_SQUAD'] = False
        model = DecisionEngine(data, input_folder)
        apply_role_priors(model)
        diags.extend(assert_pre_render(model, state, fetched_at))
        if any(d.severity == 'ERROR' for d in diags):
            raise AuditBlocked(diags)
        source_status['verified_news'] = model.news_count
        source_status['odds'] = model.odds_count
        monte = AuditedMonteCarlo(model, sample_count=AUDIT_MONTE_CARLO)
        decision, issues = compare_strategies(model, state, monte)
        diags.extend(issues)
        mpc_diagnostic=(getattr(model,'_mpc_full_horizon_audit',None)
                        or getattr(model,'mpc_last_audit',None))
        if mpc_diagnostic and mpc_diagnostic.get('status')=='FALLBACK_BEAM':
            diags.append(Diagnostic('WARNING','MPC_FALLBACK_BEAM',mpc_diagnostic['reason']))
        if decision is not None and mpc_diagnostic:
            decision['mpc_solver_audit']=mpc_diagnostic
        if any(d.severity == 'ERROR' for d in diags):
            raise AuditBlocked(diags)
        if offline_test_mode:
            # Offline fixtures exercise code paths but must never be published as live recommendations.
            diags.append(Diagnostic('WARNING', 'OFFLINE_TEST', 'Simulation uses invented test fixture records, not real FPL. Do not act.'))
        # Strict presentation/output action gate: no executable claims anywhere
        # in current_action_only, recommendation, chip calendar or derived reports.
        research_only = bool(state.get('public_provisional') or offline_test_mode)
        decision['executable'] = not research_only
        if research_only:
            original_action = decision.get('recommended_action', 'HOLD')
            decision['research_candidate_action'] = original_action
            decision['recommended_action'] = 'PROVISIONAL – research only'
            decision['current_action_only'] = {'gw': events[0], 'action': 'RESEARCH ONLY',
                'executable': False, 'model_candidate_action': original_action}
            cal = decision.get('chip_calendar_10gw')
            if isinstance(cal, dict):
                cal['research_candidate_next_action'] = cal.get('next_action')
                cal['next_action'] = 'RESEARCH ONLY'
                cal['executable'] = False
            decision['not_executable_without_private_state'] = True
            decision['confidence'] = 'Research only (private state unverified)'
        else:
            decision['current_action_only']['executable'] = True
            decision['not_executable_without_private_state'] = False
        decision['api_snapshot'] = {'raw_path': str(raw_path), 'sha256': raw_hash, 'retrieved_utc': fetched_at.isoformat()}
        decision['verified_manager_snapshot'] = {'path': str(state_path), 'sha256': state_hash,
                                                  'verified_utc': state['verified_at_utc']}
        decision['data_sources'] = source_status
        decision['current_team_state'] = state
        cut, replay_issues = freeze_predeadline_replay(model, decision, state, destination / 'Archive', fetched_at)
        diags.extend(replay_issues)
        if any(d.severity == 'ERROR' for d in diags):
            raise AuditBlocked(diags)
        decision['predeadline_backtest_cut'] = str(cut) if cut else None
        result = assemble_existing_dashboard_result(model, state, decision)
        result['generated_utc'] = utc_now().isoformat()
        result['audited_decision'] = decision
        result['data_sources']['official_fpl'] = not offline_test_mode
        if state.get('public_provisional'):
            result['free_transfers_verified'] = False
            result['ownership_current_verified'] = False
            result['bank_verified'] = False
            result['verified_state_timestamp_utc'] = None
            result['state_quality'] = 'PUBLIC_PROVISIONAL_LAST_LOCKED'
            result['can_execute_recommendation'] = False
            result['reliability_notes'] = state['unobservable_fields']
            decision['confidence'] = 'Low (public-only scenario)'
            decision['not_executable_without_private_state'] = True
            decision['main_assumptions'].append('0 FTs and 0 pending hit are public-only MODEL ASSUMPTIONS, not confirmed.')
            decision['main_risks'].append('Actual pending transfers and sale values may change the best squad and chip advice.')
        result['can_execute_recommendation'] = not research_only
        # Always offer concrete, labelled AUTOMATIC model cases even when the public
        # API does not expose this manager's pre-deadline private state.
        result['auto_research_brief'] = automatic_decision_brief(result, model, state, decision, diags)
        result['state_quality'] = 'OFFLINE_TEST' if offline_test_mode else ('PUBLIC_PROVISIONAL_LAST_LOCKED' if research_only else 'VERIFIED_CURRENT_MANAGER_STATE')
        result['executable_action'] = decision['current_action_only'] if not research_only else None
        if research_only:
            for row in result.get('plan', []):row['research_only'] = True
            result['chip']['recommendation']='PROVISIONAL – research only'
        projection_path = export_player_projections(monte, destination, fetched_at.strftime('%Y%m%dT%H%M%S%fZ'))
        result['projection_artifact'] = str(projection_path)
        archive_path = destination / 'Archive' / f'decision_{fetched_at.strftime("%Y%m%dT%H%M%S%fZ")}.json'
        atomic_json(archive_path, {'schema_version': AUDIT_VERSION, 'generated_utc': utc_now().isoformat(),
                                   'decision': decision, 'diagnostics': [asdict(d) for d in diags]})
        # Every assertion and recommendation is complete before either 4K or dashboard rendering.
        diags.extend(assert_pre_render(model, state, fetched_at))
        if any(d.severity == 'ERROR' for d in diags):
            raise AuditBlocked(diags)
        if not offline_test_mode:
            embed_visual_assets(model, destination)
        png_path = destination / 'FPLVORTEX_TOP5K_LATEST_4K.png'
        html_path = destination / 'FPLVORTEX_TOP5K_AUTOPILOT_V4.html'
        json_path = destination / 'FPLVORTEX_TOP5K_LATEST.json'
        draw_four_k(result, model, png_path)
        base_html = render_html(result, model, png_path)
        audit_panel = html_audit_panel(decision, state, fetched_at.isoformat(), source_status, diags)
        final_html = attach_audit_tab(base_html, audit_panel, decision['recommended_action'], decision)
        # The original overview had a generic action banner; put the specific
        # scenario comparison at the FIRST visible dashboard view instead.
        brief_html = automatic_brief_panel(result['auto_research_brief'])
        main_marker = '<section class="page active" id="page-overview" role="tabpanel">'
        assert main_marker in final_html, 'Legacy dashboard overview markup changed.'
        final_html = final_html.replace(main_marker, main_marker + brief_html, 1)
        if state.get('public_provisional'):
            notice = ('<div class="alert" style="border:2px solid #ffcf82;background:#403015;color:white;padding:17px;">'
                      '<strong>PROVISIONAL – research only.</strong> The public API shows the last '
                      'locked team and recorded chips, NOT this week’s pending moves, exact sell prices, '
                      'true free transfers or chips awaiting activation. £ bank is LAST-PUBLISHED. '
                      '10,000 Monte Carlo draws quantify model variance, not uncertainty in hidden account data. '
                      'No manual input required. Treat all chip/transfer calls as research candidates, '
                      'NOT executable live recommendations.</div>')
            final_html = final_html.replace('<body', '<body', 1)  # keep original CSS body intact
            if '<section class="page active"' in final_html:
                final_html = final_html.replace('<section class="page active"', notice + '<section class="page active"', 1)
            final_html = final_html.replace('verified FPL state', 'PROVISIONAL – research only')
            final_html = final_html.replace('Suggested move', 'Research scenario (NOT executable)')
            final_html = final_html.replace('Review news before lock', 'Provisional model path only')
            final_html = final_html.replace('Verified Plan & Audit', 'PROVISIONAL – research only')
            final_html = final_html.replace('Verified current manager state:', 'Public last-published snapshot fetched:')
            final_html = final_html.replace('Verified decision & uncertainty audit', 'Provisional decision & uncertainty audit')
            final_html = final_html.replace('Verified Plan & Audit', 'Public Data & Audit')
            final_html = final_html.replace('10 future chip windows evaluated with the same verified manager state',
                                            '10 chip windows evaluated from public last-locked squad')
            final_html = final_html.replace('Verified at ', 'Public snapshot fetched at ')
            final_html = final_html.replace('Timing is provisional; only this week’s action is actionable.',
                                            'All actions are provisional until live private state becomes available.')
        assert final_html.count('data-tab=') >= 8 and 'id="page-auditdecision"' in final_html
        atomic_json(json_path, result)
        html_path.write_text(final_html, encoding='utf-8')
        write_diagnostics(destination, diags, source_status, status=('public-provisional' if state.get('public_provisional') else 'offline-test' if offline_test_mode else 'verified'))
        summary_brief = result['auto_research_brief']
        print('📊 AUTOMATIC MODEL RESEARCH BRIEF • GW', summary_brief['gameweek'])
        print('   Model candidate:', summary_brief['model_candidate'])
        move = summary_brief['no_chip_transfer_path']
        print('   No-chip alternative:', move['candidate'], 'OUT:', ', '.join(move['moves_out']) or 'none',
              'IN:', ', '.join(move['moves_in']) or 'none', 'model hits:', move['hit_points_assumed'])
        print('   Hold captain:', summary_brief['hold']['lineup']['captain'])
        print('   Wildcard candidate:', ', '.join(summary_brief['wildcard']['squad']) or 'unavailable')
        print('   10-GW chip candidates:', [(row['gw'], row['chip']) for row in summary_brief['chip_calendar']])
        if research_only:
            print('🔒 PROVISIONAL – research only: last-published squad, unknown private changes; nothing above is an executable instruction.')
        print('✅ FPLVORTEX', 'PROVISIONAL – research only' if research_only else 'VERIFIED executable model', decision['recommended_action'], decision['confidence'])
        print('✅ Decision & scenario audit:', archive_path)
        print('✅ Raw snapshot:', raw_path)
        print('✅ Per-player/GW projections:', projection_path)
        print('✅ Existing HTML / 4K output:', html_path, png_path)
        return html_path, png_path, json_path
    except (AuditBlocked, ValueError, AssertionError, RuntimeError, KeyError, OSError) as error:
        if isinstance(error, AuditBlocked):
            diags = error.diagnostics
        else:
            diags.append(Diagnostic('ERROR', type(error).__name__, str(error)))
        diag_path, diag_html = write_diagnostics(destination, diags, source_status, status='blocked')
        print('⛔ FPLVORTEX recommendation blocked. Read', diag_html)
        raise AuditBlocked(diags) from error




## Added cell 4 — Historical replay and immutable predeadline feature cuts
No current-season statistics may masquerade as a prior deadline forecast. Missing actuals or baseline cohorts are reported as unavailable.

In [ ]:
def run_historical_replay(archive: Path, output: Path | None = None) -> tuple[Path, Path]:
    """Opt-in strict pre-deadline replay. No historical feature reconstruction from present-day API.

    Input pairs per completed GW:
      predeadline_GWNN.json: {snapshot_utc,deadline_utc,gw,projections:[{player_id,mean,p_start,expected_minutes}],
            captain_id, transfer_hit, transfer_predicted_gain, wc_predicted_gain,
            fixture_baseline, template_baseline, no_transfer_baseline}
      settled_GWNN.json: {settled_utc,gw,actual:[{player_id,points,minutes,started}],
            captain_best_available_id, transfer_realized_gain, wildcard_realized_gain,
            no_transfer_total, fixture_baseline_total, template_baseline_total}
    Every baseline must have been computed using pre-deadline data; missing data yields NA, not invented scores.
    """
    import csv
    archive = Path(archive)
    output = output or archive / 'Backtests'
    output.mkdir(parents=True, exist_ok=True)
    records: list[dict[str, Any]] = []
    diagnostics: list[str] = []
    for before_path in sorted(archive.glob('predeadline_GW*.json')):
        label = before_path.stem.removeprefix('predeadline_')
        after_path = archive / f'settled_{label}.json'
        if not after_path.is_file():
            diagnostics.append(f'{label}: missing settled actuals; excluded')
            continue
        try:
            before = json.loads(before_path.read_text(encoding='utf-8'))
            after = json.loads(after_path.read_text(encoding='utf-8'))
            deadline, captured, settled = (iso_time(before.get('deadline_utc')), iso_time(before.get('snapshot_utc')),
                                           iso_time(after.get('settled_utc')))
            if (deadline is None or captured is None or settled is None or not captured < deadline < settled
                    or before.get('gw') != after.get('gw')):
                diagnostics.append(f'{label}: temporal leakage or GW mismatch; excluded')
                continue
            actual = {row['player_id']: row for row in after.get('actual', [])}
            projections = before.get('projections', [])
            if not projections or not actual:
                diagnostics.append(f'{label}: required historical data missing; excluded')
                continue
            safe = lambda value: value if isinstance(value, (int, float)) and math.isfinite(value) else None
            eligible = [(p, actual[p['player_id']]) for p in projections if p['player_id'] in actual]
            if not eligible:
                diagnostics.append(f'{label}: no matched player results; excluded')
                continue
            xp_error = float(np.mean([(p['mean'] - a['points']) ** 2 for p, a in eligible if safe(p.get('mean')) is not None]))
            minute_eligible = [(p, a) for p, a in eligible if safe(p.get('p_start')) is not None and isinstance(a.get('started'), bool)]
            minute_brier = (float(np.mean([(p['p_start'] - (1 if a['started'] else 0)) ** 2 for p, a in minute_eligible]))
                            if minute_eligible else None)
            minute_mae = (float(np.mean([abs(p['expected_minutes'] - a['minutes']) for p, a in eligible
                                          if safe(p.get('expected_minutes')) is not None])) if eligible else None)
            cap_id = before.get('captain_id')
            captain_points = actual.get(cap_id, {}).get('points')
            benchmark_id = after.get('captain_best_available_id')
            bench_cap = actual.get(benchmark_id, {}).get('points')
            captain_regret = max(0, bench_cap - captain_points) if captain_points is not None and bench_cap is not None else None
            computed_transfer, computed_wildcard = replay_realized_gains(before, archive, after)
            hit = before.get('transfer_hit')
            realized = after.get('transfer_realized_gain')
            after_hit = (realized - hit if safe(realized) is not None and safe(hit) is not None else computed_transfer)
            wc_realized = after.get('wildcard_realized_gain')
            if wc_realized is None:
                wc_realized = computed_wildcard
            wc_predicted = before.get('wc_predicted_gain')
            wc_quality = (wc_realized - wc_predicted if safe(wc_realized) is not None and safe(wc_predicted) is not None else None)
            records.append({'gw': before['gw'], 'snapshot_utc': captured.isoformat(), 'deadline_utc': deadline.isoformat(),
                            'matched_players': len(eligible), 'xp_mse': round(xp_error, 4),
                            'minutes_start_brier': round(minute_brier, 4) if minute_brier is not None else None,
                            'minutes_mae': round(minute_mae, 4) if minute_mae is not None else None,
                            'captain_regret': captain_regret,
                            'transfer_gain_after_hits': after_hit,
                            'wc_forecast_error': wc_quality,
                            'no_transfer_baseline_points': after.get('no_transfer_total'),
                            'simple_fixture_baseline_points': after.get('fixture_baseline_total') if before.get('fixture_baseline') is not None else None,
                            'template_baseline_points': after.get('template_baseline_total') if before.get('template_baseline') is not None else None,
                            'source': 'versioned frozen predeadline artifacts + subsequent settled outcomes'})
        except (KeyError, OSError, ValueError, TypeError, ZeroDivisionError) as error:
            diagnostics.append(f'{label}: invalid replay ({error}); excluded')
    now = utc_now()
    ident = now.strftime('%Y%m%dT%H%M%S%fZ')
    json_path = output / f'FPLVORTEX_BACKTEST_{ident}.json'
    csv_path = output / f'FPLVORTEX_BACKTEST_{ident}.csv'
    with csv_path.open('x', encoding='utf-8', newline='') as stream:
        columns = ['gw', 'snapshot_utc', 'deadline_utc', 'matched_players', 'xp_mse', 'minutes_start_brier',
                   'minutes_mae', 'captain_regret', 'transfer_gain_after_hits', 'wc_forecast_error',
                   'no_transfer_baseline_points', 'simple_fixture_baseline_points', 'template_baseline_points', 'source']
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows(records)
    atomic_json(json_path, {'schema_version': AUDIT_VERSION, 'run_utc': now.isoformat(), 'evaluated_gws': len(records),
                            'rows': records, 'diagnostics': diagnostics,
                            'limitations': ['Requires genuine frozen snapshots captured before each deadline',
                                            'No historical FPL features reconstructed using current-season aggregate fields',
                                            'Candidate-policy regret and Wildcard quality require full counterfactual player actuals',
                                            'Missing benchmark data reported as null, never invented']})
    print('🧪 Historical replay:', json_path, csv_path, 'GWs:', len(records))
    return csv_path, json_path


def freeze_predeadline_replay(engine: Any, decision: dict[str, Any], state: dict[str, Any],
                              archive: Path, now: datetime) -> tuple[Path | None, list[Diagnostic]]:
    """Capture a true predeadline feature cut. Existing cut is never overwritten."""
    diagnostics: list[Diagnostic] = []
    gw = engine.events[0]
    event = next((e for e in engine.snapshot['bootstrap']['events'] if integer(e.get('id')) == gw), None)
    deadline = iso_time(event.get('deadline_time')) if event else None
    if deadline is None or not now < deadline:
        return None, [Diagnostic('ERROR', 'PREDEADLINE_LEAK', f'GW{gw} deadline is missing or passed; no backtest features frozen.')]
    archive.mkdir(parents=True, exist_ok=True)
    target = archive / f'predeadline_GW{gw}.json'
    if target.exists():
        return target, [Diagnostic('INFO', 'REPLAY_ALREADY_FROZEN', f'GW{gw} historical cut already frozen; kept immutable.')]
    projections = [{'player_id': pid, 'mean': engine.xp(pid, gw),
                    'p_start': engine.forecasts[(pid, gw)].p_start,
                    'expected_minutes': engine.forecasts[(pid, gw)].minutes,
                    'fixture_difficulty': engine.forecasts[(pid, gw)].fdr}
                   for pid in sorted(engine.players)]
    xi = engine.lineup(engine.owned, gw)
    fixture_captain = min(xi['xi'], key=lambda pid: (engine.forecasts[(pid, gw)].fdr, -engine.xp(pid, gw)))
    hold_steps = hold_route(engine)
    repair_steps = engine.plan().steps
    wc_steps = decision['wildcard'].get('path_full') or []
    def frozen_xis(steps: list[dict[str, Any]]) -> list[dict[str, Any]]:
        results = []
        for step in steps:
            line = engine.lineup(tuple(step['ids']), integer(step['gw']))
            results.append({'gw': integer(step['gw']), 'squad': list(step['ids']), 'xi': line['xi'],
                            'bench': line['bench'], 'captain_id': line['captain'],
                            'vice_id': line['vice'], 'hit': integer(step.get('hit', 0)),
                            'positions': {str(pid): integer(engine.players[pid]['element_type']) for pid in step['ids']}})
        return results
    frozen_paths = {'hold': frozen_xis(hold_steps), 'repair': frozen_xis(repair_steps),
                    'wildcard': frozen_xis(wc_steps) if wc_steps else None}
    data = {
        'schema_version': AUDIT_VERSION, 'gw': gw, 'snapshot_utc': now.isoformat(),
        'deadline_utc': deadline.isoformat(), 'raw_retrieved_utc': engine.snapshot.get('collected_utc'),
        'entry_id': integer(engine.snapshot['profile']['id']),
        'squad_ids_at_deadline_plan': list(engine.owned),
        'squad_xi': xi['xi'], 'captain_id': xi['captain'],
        'fixture_baseline': {'squad': list(engine.owned), 'captain_id': fixture_captain,
                             'policy': 'Use same XI, fixture-difficulty captain; no forecast lookahead'},
        'no_transfer_baseline': {'squad': list(engine.owned), 'captain_id': xi['captain']},
        'template_baseline': global_ownership_template_baseline(engine),  # Global ownership proxy, never top-10K EO.
        'transfer_hit': state['displayed_hit_cost'],
        'transfer_predicted_gain': decision['comparison']['repair']['mean'] - decision['comparison']['hold']['mean'],
        'wc_predicted_gain': (decision['delta']['wildcard_less_best_no_chip'] or {}).get('mean'),
        'wc_candidate_ids': (decision['wildcard'].get('selected_ids') if decision['wildcard']['feasible'] else None),
        'repair_steps': repair_steps,
        'policy_lineups': frozen_paths,
        'projections': projections,
    }
    try:
        atomic_json(target, data, immutable=True)
    except FileExistsError:
        return target, [Diagnostic('INFO', 'REPLAY_ALREADY_FROZEN', 'Concurrent or previous run froze this predeadline cut.')]
    return target, diagnostics


def settle_previous_replays(raw: dict[str, Any], archive: Path, now: datetime, *, allow_network: bool = True) -> list[Diagnostic]:
    """Observed player rows are permitted after deadline; never modify their frozen predictions."""
    diagnostics: list[Diagnostic] = []
    archived = sorted(archive.glob('predeadline_GW*.json'))
    summaries = raw.get('summaries', {})
    for path in archived:
        try:
            before = json.loads(path.read_text(encoding='utf-8'))
            gw = integer(before['gw'])
            deadline = iso_time(before['deadline_utc'])
            if deadline is None or now <= deadline + timedelta(hours=30):
                continue
            fixture_rows = [f for f in raw.get('fixtures', []) if integer(f.get('event')) == gw]
            if fixture_rows and not all(f.get('finished') for f in fixture_rows):
                diagnostics.append(Diagnostic('WARNING', 'GW_NOT_FINAL', f'GW{gw} still has unfinished fixtures; settlement postponed.'))
                continue
            target = archive / f'settled_GW{gw}.json'
            if target.exists():
                continue
            required = set(before.get('squad_ids_at_deadline_plan', []))
            # Missing counterfactual players should suppress only the associated benchmark.
            wished = set(required)
            wished.update(before.get('wc_candidate_ids') or [])
            template = before.get('template_baseline')
            if isinstance(template, dict):
                wished.update(template.get('squad', []))
            for plan in (before.get('policy_lineups') or {}).values():
                if plan:
                    for week in plan:
                        if integer(week.get('gw')) == gw:
                            wished.update(week.get('squad', []))
            source_summaries = dict(summaries)
            missing_ids = sorted(wished - {integer(pid) for pid in source_summaries})
            extra_summaries = {}
            if allow_network and missing_ids:
                with get_session() as session:
                    for pid in missing_ids[:36]:  # Bounded extra post-deadline reads.
                        payload = fetch(session, f'element-summary/{pid}/', optional=True)
                        if isinstance(payload, dict) and isinstance(payload.get('history'), list):
                            source_summaries[str(pid)] = payload
                            extra_summaries[str(pid)] = payload
                if extra_summaries:
                    extra_name = f'settlement_raw_GW{gw}_{now.strftime("%Y%m%dT%H%M%S%fZ")}.json'
                    atomic_json(archive / extra_name,
                                {'retrieved_utc': now.isoformat(), 'gw': gw, 'element_summaries': extra_summaries},
                                immutable=True)
            # All 15 no-transfer players are required. Missing WC/template details
            # remain null instead of being silently assigned zero points.
            actual: list[dict[str, Any]] = []
            for player_id, summary in source_summaries.items():
                pid = integer(player_id)
                rows = [r for r in summary.get('history', []) if integer(r.get('round')) == gw]
                if rows:
                    actual.append({'player_id': pid, 'points': sum(num(row.get('total_points')) for row in rows),
                                   'minutes': sum(num(row.get('minutes')) for row in rows),
                                   'started': any(integer(row.get('starts')) > 0 for row in rows) if any('starts' in r for r in rows) else None})
            observed = {r['player_id']: r for r in actual}
            if not required.issubset(observed):
                diagnostics.append(Diagnostic('WARNING', 'SETTLEMENT_PARTIAL',
                                              f'GW{gw}: missing actuals for {len(required - set(observed))} historical squad players; settlement postponed.'))
                continue
            baseline = before['no_transfer_baseline']
            xi = before['squad_xi']
            if len(xi) != 11 or baseline['captain_id'] not in observed:
                diagnostics.append(Diagnostic('WARNING', 'SETTLEMENT_XI', f'GW{gw}: invalid original XI snapshot.'))
                continue
            no_transfer_total = sum(observed[pid]['points'] for pid in xi) + observed[baseline['captain_id']]['points']
            fixture_baseline_total = sum(observed[pid]['points'] for pid in xi) + observed[before['fixture_baseline']['captain_id']]['points']
            top_real_captain = max(xi, key=lambda pid: observed[pid]['points'])
            template = before.get('template_baseline')
            template_actual = None
            if isinstance(template, dict) and set(template.get('xi', [])).issubset(observed) and template.get('captain_id') in observed:
                template_actual = sum(observed[pid]['points'] for pid in template['xi']) + observed[template['captain_id']]['points']
            outcome = {'gw': gw, 'settled_utc': now.isoformat(), 'actual': actual,
                       'captain_best_available_id': top_real_captain,
                       'no_transfer_total': no_transfer_total,
                       'fixture_baseline_total': fixture_baseline_total,
                       'template_baseline_total': template_actual,
                       'transfer_realized_gain': None,  # Counterfactual path requires full historical future-gameweek actuals.
                       'wildcard_realized_gain': None}
            atomic_json(target, outcome, immutable=True)
        except (ValueError, KeyError, TypeError, OSError, FileExistsError) as error:
            diagnostics.append(Diagnostic('WARNING', 'SETTLEMENT_INVALID', f'{path.name}: {error}'))
    return diagnostics


def global_ownership_template_baseline(engine: Any) -> dict[str, Any] | None:
    """Frozen global-ownership proxy for replay only, NEVER represented as top-10K effective ownership."""
    candidates = [pid for pid, p in engine.players.items()
                  if integer(p.get('element_type')) in POSITION_COUNTS and
                  p.get('status') not in ('u', 's') and
                  0 <= num(p.get('selected_by_percent'), -1) <= 100]
    if len(candidates) < 60:
        return None
    objective = np.asarray([-num(engine.players[pid]['selected_by_percent']) for pid in candidates], dtype=float)
    n = len(candidates)
    constraints: list[tuple[list[int], int, int]] = [(list(range(n)), 15, 15)]
    for pos, count in POSITION_COUNTS.items():
        constraints.append(([i for i, pid in enumerate(candidates) if integer(engine.players[pid]['element_type']) == pos], count, count))
    for club_id in engine.teams:
        rows = [i for i, pid in enumerate(candidates) if integer(engine.players[pid]['team']) == club_id]
        if rows:
            constraints.append((rows, 0, 3))
    constraint_matrix = lil_matrix((len(constraints) + 1, n), dtype=float)
    lower, upper = [], []
    for row_index, (cols, min_count, max_count) in enumerate(constraints):
        for col in cols:
            constraint_matrix[row_index, col] = 1
        lower.append(min_count)
        upper.append(max_count)
    for col, pid in enumerate(candidates):
        constraint_matrix[len(constraints), col] = integer(engine.players[pid]['now_cost'])
    lower.append(0)
    upper.append(engine.budget)
    try:
        solution = milp(objective, integrality=np.ones(n), bounds=Bounds(np.zeros(n), np.ones(n)),
                        constraints=LinearConstraint(constraint_matrix.tocsr(), lower, upper),
                        options={'time_limit': 12, 'mip_rel_gap': .03})
        if solution.x is None:
            return None
        squad = tuple(sorted(pid for i, pid in enumerate(candidates) if solution.x[i] > .5))
        if not engine.valid_squad(squad) or sum(integer(engine.players[pid]['now_cost']) for pid in squad) > engine.budget:
            return None
        line = engine.lineup(squad, engine.events[0])
        return {'squad': list(squad), 'xi': line['xi'], 'captain_id': line['captain'],
                'reference_cohort': 'Global ownership (FPL selected_by_percent), NOT top-10K managers',
                'selection_policy': 'Most globally owned legal/budget-feasible XV; xP-selected XI',
                'price_source': 'Official bootstrap-static at predeadline timestamp'}
    except (RuntimeError, ValueError, TypeError):
        return None


def realized_policy_gw(policy: dict[str, Any], observed: dict[int, dict[str, Any]]) -> float | None:
    """Score a frozen predeadline XI using settled player outcomes and legal auto-subs."""
    squad = [integer(x) for x in policy['squad']]
    if not set(squad).issubset(observed):
        return None
    xi = [integer(x) for x in policy['xi']]
    bench = [integer(x) for x in policy['bench']]
    if len(xi) != 11 or len(bench) != 4 or len(set(xi + bench)) != 15:
        return None
    pos = {integer(key): integer(value) for key, value in policy['positions'].items()}
    points = {pid: num(observed[pid]['points']) for pid in squad}
    played = {pid: num(observed[pid]['minutes']) > 0 for pid in squad}
    total = sum(points[pid] for pid in xi)
    if not played[xi[0]] and played[bench[0]]:
        total += points[bench[0]]
    outfield = [pid for pid in xi if pos[pid] != 1]
    onfield = [pid for pid in outfield if played[pid]]
    counts = {ptype: sum(pos[pid] == ptype for pid in onfield) for ptype in (2,3,4)}
    available = [pid for pid in bench[1:] if played[pid]]
    needed = 10 - len(onfield)
    for slot in range(needed):
        choice = None
        for candidate in available:
            candidate_pos = pos[candidate]
            new_counts = dict(counts)
            new_counts[candidate_pos] += 1
            unfilled = sum(max(0, req - new_counts[ptype]) for ptype, req in ((2,3), (3,2), (4,1)))
            if unfilled <= needed-slot-1:
                choice = candidate
                break
        if choice is None:
            break
        total += points[choice]
        counts[pos[choice]] += 1
        available.remove(choice)
    captain = integer(policy['captain_id'])
    vice = integer(policy['vice_id'])
    if captain in points and played[captain]:
        total += points[captain]
    elif vice in points and played[vice]:
        total += points[vice]
    return total - integer(policy.get('hit',0))


def replay_realized_gains(before: dict[str, Any], archive: Path, current_settled: dict[str, Any]) -> tuple[float | None, float | None]:
    policies = before.get('policy_lineups') or {}
    hold, repair, wildcard = policies.get('hold') or [], policies.get('repair') or [], policies.get('wildcard') or []
    if not hold or not repair:
        return None, None
    actuals = {int(row['player_id']): row for row in current_settled.get('actual',[])}
    realized_hold = realized_policy_gw(hold[0], actuals)
    realized_repair = realized_policy_gw(repair[0], actuals)
    transfer_gain = (realized_repair - realized_hold if realized_hold is not None and realized_repair is not None else None)
    if not wildcard:
        return transfer_gain, None
    totals: dict[str,float] = {'hold': 0.0, 'repair': 0.0, 'wildcard': 0.0}
    for name, plan in (('hold', hold), ('repair', repair), ('wildcard', wildcard)):
        for week in plan:
            gameweek = integer(week['gw'])
            settle_path = archive / f'settled_GW{gameweek}.json'
            if settle_path.exists():
                result = json.loads(settle_path.read_text(encoding='utf-8'))
            elif gameweek == before['gw']:
                result = current_settled
            else:
                return transfer_gain, None
            observed = {int(row['player_id']): row for row in result.get('actual', [])}
            points = realized_policy_gw(week, observed)
            if points is None:
                return transfer_gain, None
            totals[name] += points
    pending_hit = integer(before.get('transfer_hit',0))
    no_chip_best = max(totals['hold'],totals['repair']) - pending_hit
    return transfer_gain, totals['wildcard'] - no_chip_best


## Final settings and execution
`BACKTEST_ONLY=False` runs the audited live engine. For strict historical-only analysis set it to `True`. Never populate the manager state with guesses.

## Automatic FPL API state — no manual entry

The notebook uses public official FPL entry/picks/history/transfers and published chip usage. No player-input form is needed. Where FPL hides information until the deadline, the model uses conservative **explicit assumptions** and labels decisions **PROVISIONAL**. Exact current selling prices, pending transfers/FTs and chips activated for the next GW are not public, and cannot be truthfully authenticated without user account authorization.


In [ ]:
# FPLVORTEX V4.4: completely hands-free, official FPL API public-mode inputs.
# This DOES NOT claim access to private pre-deadline transfers or sell prices.
# It replaces the old 15-player manual form, not the existing forecast/optimizer.

def derive_public_manager_state(raw: dict[str, Any], events: list[int], now: datetime) -> tuple[dict[str, Any], list[Diagnostic]]:
    """Derive a transparent, conservative model state from PUBLIC official FPL endpoints.

    The last-published picks and bank are observed, the chip usage history is observed,
    while current predeadline moves, FTs, and precise liquidation prices remain unknown.
    Never pretend the public API reveals those private fields.
    """
    alerts = []
    profile, history = raw['profile'], raw['history']
    picks = raw['picks'].get('picks', [])
    if len(picks) != 15 or not events:
        raise AuditBlocked([Diagnostic('ERROR', 'PUBLIC_PICKS_UNAVAILABLE', 'The official API has no complete 15-player published squad or next deadline.')])
    published_gw = integer(raw.get('picked_gw'))
    if published_gw < 1:
        raise AuditBlocked([Diagnostic('ERROR', 'PUBLIC_PICKS_UNAVAILABLE', 'No completed public pick history yet.')])
    ids = [integer(p.get('element')) for p in picks]
    players = {integer(p['id']): p for p in raw['bootstrap']['elements']}
    if len(ids) != len(set(ids)) or any(pid not in players for pid in ids):
        raise AuditBlocked([Diagnostic('ERROR','PUBLIC_PICKS_INCONSISTENT','Official last-published picks have duplicate or unknown player IDs.')])
    history_rows = {integer(row.get('event')): row for row in history.get('current', []) if isinstance(row, dict)}
    published = raw['picks'].get('entry_history') or history_rows.get(published_gw) or {}
    last_bank = published.get('bank', profile.get('last_deadline_bank'))
    if type(last_bank) is not int or last_bank < 0:
        raise AuditBlocked([Diagnostic('ERROR', 'PUBLIC_BANK_UNAVAILABLE', 'The official last-deadline bank could not be read; cannot guarantee squad budgets.')])
    current_value = published.get('value', profile.get('last_deadline_value'))
    if type(current_value) is not int or current_value < 0:
        current_value = last_bank + sum(integer(players[pid].get('now_cost')) for pid in ids)
        alerts.append(Diagnostic('WARNING','VALUE_ESTIMATED','Last-published team value unavailable; market-price proxy used.'))
    # Use only public transfer ledger records not later than the displayed locked GW.
    acquired = {}
    transfers = raw.get('transfers')
    if isinstance(transfers, list):
        for txn in sorted((r for r in transfers if isinstance(r,dict)), key=lambda t: (integer(t.get('event')), str(t.get('time','')))):
            if integer(txn.get('event')) <= published_gw:
                pid, price = integer(txn.get('element_in')), txn.get('element_in_cost')
                if type(price) is int and price >= 25: acquired[pid] = price
    rows = []
    purchase_sources = {}
    for pid in ids:
        player = players[pid]
        now_cost = integer(player.get('now_cost'))
        if now_cost < 25:
            raise AuditBlocked([Diagnostic('ERROR','PRICE_MISSING',f'No current public market price for {pid}.')])
        purchase = acquired.get(pid)
        if purchase is None:
            matches = [r for r in raw.get('summaries',{}).get(str(pid),{}).get('history',[])
                       if isinstance(r,dict) and type(r.get('value')) is int and integer(r.get('round')) <= published_gw]
            if matches:
                matches.sort(key=lambda r: integer(r.get('round')))
                purchase = integer(matches[0]['value'])
                source = 'public first-match value proxy; unknown initial purchase price'
            else:
                purchase = now_cost
                source = 'current market price proxy; unknown acquisition cost'
        else:
            source = 'official published transfer purchase cost'
        # FPL sale price: purchase + floor(profit/2), or market price on a loss.
        raw_est = min(now_cost, purchase + max(0,now_cost-purchase)//2)
        # A conservative 0.1m reduction for unknown first-GW acquisitions avoids
        # assuming the exact 0.1m profit split when original purchase price is hidden.
        estimate = max(25, raw_est - (1 if 'proxy' in source and now_cost > 25 else 0))
        rows.append({'element': pid, 'selling_price': estimate, 'purchase_price_estimate': purchase,
                     'selling_price_source': source})
        purchase_sources[str(pid)] = source
    known_chips = history.get('chips')
    if not isinstance(known_chips, list):
        raise AuditBlocked([Diagnostic('ERROR','CHIP_HISTORY_UNAVAILABLE','Official public chip history unavailable. Cannot infer chip eligibility.')])
    current_gw = events[0]
    half_start, half_end = (1,19) if current_gw <= 19 else (20,38)
    used = {str(c.get('name')) for c in known_chips if isinstance(c,dict)
            and half_start <= integer(c.get('event')) <= half_end}
    codes={'wildcard':'wildcard','freehit':'freehit','triplecaptain':'3xc','benchboost':'bboost'}
    chips = {name: code not in used for name,code in codes.items()}
    if current_gw==20 and any(isinstance(c,dict) and c.get('name')=='freehit' and integer(c.get('event'))==19 for c in known_chips):
        chips['freehit']=False
    # Public FT ledger cannot disclose already-committed NEXT-GW moves.
    # Set ZERO free transfers as a pessimistic lower bound, not an observed value.
    # Do not invent pre-deadline transfer counts or point costs.
    ft_lower_bound = 0
    state = {
        'entry_id': integer(profile.get('id')), 'verified_at_utc': now.isoformat(),
        'source': 'Official public FPL last-published entry, picks, history and transfers (PROVISIONAL)',
        'confirmed': False, 'public_provisional': True, 'next_event': current_gw,
        'published_gw': published_gw, 'current_squad': rows,
        'bank_tenths': last_bank, 'team_value_tenths': current_value,
        'ft_before_moves': ft_lower_bound, 'transfers_made_this_gw': 0,
        'displayed_hit_cost': 0, 'ft_after_moves': ft_lower_bound,
        'ft_after_wildcard': None, 'chip_available': chips, 'active_chip': None,
        'known_used_chip_history': known_chips, 'price_provenance': purchase_sources,
        'unobservable_fields': ['unpublished transfers and Wildcard activation', 'exact current selling prices',
                                'current bank after unpublished transfers', 'true free transfers and pending hit'],
        'notes': ('Automatic public-API scenario. Squad and bank are from last published GW; '
                  'selling prices conservatively estimated from public records; 0 FTs and 0 pending hits are '
                  'MODEL ASSUMPTIONS, not the account display. Not executable pre-deadline instructions.'),
    }
    alerts.extend([
        Diagnostic('WARNING','PUBLIC_LOCKED_STATE',f'Using official last-published GW{published_gw} squad and bank; GW{current_gw} private moves are not available to public API.'),
        Diagnostic('WARNING','PUBLIC_SELLING_ESTIMATE','Sale prices inferred from public transfer purchase costs or historical prices; real budget may differ.'),
        Diagnostic('WARNING','PUBLIC_FT_LOWER_BOUND','0 free transfers modelled conservatively; actual FTs and pending hit are not publicly observable.'),
        Diagnostic('WARNING','CHIP_AVAILABILITY_PUBLIC','Chip usage from official history; a chip activated for the upcoming GW may not appear until that GW locks.'),
    ])
    print(f'✅ AUTO PUBLIC DATA: locked GW{published_gw} squad (15 players), £{last_bank/10:.1f}m last-published bank; published chip history read.')
    print('⚠️ PUBLIC-ONLY: pre-deadline transfers, current selling prices and actual FTs cannot be authenticated from this public data.')
    print('🧪 Running a PROVISIONAL 10-GW strategy/chip scenario automatically. No form required.')
    return state, alerts


## Added cell 6 — Ten-gameweek chip calendar and larger paired simulation

The baseline V4.2 audited optimizer, private state editor, FPL API, dashboard and PNG remain intact. This module compares chip timing in every upcoming Gameweek using 10,000 paired draws. Future chip weeks are provisional; do not activate a chip before verified model evidence passes.


## V4.5 · Verified-state gate, richer expected minutes, and multi-period transfer MILP
Research runs automatically on public FPL data, but executable actions require validated private manager state. The bounded MPC MILP respects squad-flow, positions, clubs, budget, FTs, hits, XI and captain over the 10-GW horizon. Only the current GW is ever an actionable recommendation. Future prices and set-piece roles are uncertain.


In [ ]:

# V4.5: multi-period mixed-integer squad / XI / captain / FT / bank planner.
# scipy.optimize.milp (HiGHS): no additional Colab pip dependency.
from scipy.sparse import coo_matrix

MPC_DISCOUNT=0.93
MPC_MAX_MOVES_PER_GW=2  # Explicit bounded candidate formulation, NOT global optimum.
MPC_TIME_LIMIT_SECONDS=12.0


def mpc_transfer_pool(engine: Any) -> list[int]:
    """Score by entire horizon; keep ALL owned players and position/value alternatives."""
    scores={pid:sum(MPC_DISCOUNT**t * engine.xp(pid,gw)
                    for t,gw in enumerate(engine.events)) for pid in engine.players}
    ids=set(engine.owned)
    for pos in POSITION_COUNTS:
        eligible=[p for p in engine.players if integer(engine.players[p]['element_type'])==pos
                  and engine.players[p].get('status') not in ('s','u')]
        # Balanced cohort ~35-45 vars/week, including defenders/keepers/cheap cover.
        best=sorted(eligible,key=lambda pid:scores[pid],reverse=True)
        cheap=sorted(eligible,key=lambda pid:integer(engine.players[pid]['now_cost']))
        ids.update(best[:(6 if pos==1 else 9 if pos in (2,3) else 8)])
        ids.update(cheap[:(2 if pos==1 else 3)])
    return sorted(ids)


def mpc_plan(engine: Any) -> Route:
    """MPC 10-week MILP; only current-week transfer is eligible for execution.

    Decision variables for every GW: squad x, XI y, captain c, buy b, sell s,
    integer bank, and one-hot (FT_start, num_moves) transition z. Transition
    z exactly encodes hits=max(0,moves-FT) and next FT=min(5,max(0,FT-moves)+1).
    Club/position limits, bank and player flow are hard constraints. All
    forecast prices after today are constant-price scenarios, not guarantees.
    """
    if not engine.events: raise ValueError('MPC requires a nonempty GW horizon')
    pool=mpc_transfer_pool(engine); n=len(pool); H=len(engine.events)
    ix={pid:j for j,pid in enumerate(pool)}
    owned=set(engine.owned)
    if not owned.issubset(ix):raise ValueError('MPC candidate pool omitted an owned player')
    cost=np.array([integer(engine.players[pid]['now_cost']) for pid in pool],dtype=int)
    sell=np.array([integer(engine.owned_prices.get(pid,cost[j])) for j,pid in enumerate(pool)],dtype=int)
    owned_ids=np.array([int(pid in owned) for pid in pool])
    K=MPC_MAX_MOVES_PER_GW; F=range(6); M=range(K+1)
    # Centralized variable allocator enables independent legality audit.
    cobj=[]; lower=[];upper=[];integral=[]
    def addvar(lb=0,ub=1,obj=0,integ=1):
        j=len(cobj);cobj.append(float(obj));lower.append(float(lb));upper.append(float(ub));integral.append(int(integ));return j
    x=[[addvar() for j in range(n)] for t in range(H)]
    y=[[addvar(obj=-MPC_DISCOUNT**t*engine.xp(pid,engine.events[t])) for pid in pool] for t in range(H)]
    captain=[[addvar(obj=-MPC_DISCOUNT**t*engine.xp(pid,engine.events[t])) for pid in pool] for t in range(H)]
    buy=[[addvar(obj=MPC_DISCOUNT**t*.35) for j in range(n)] for t in range(H)]
    sold=[[addvar() for j in range(n)] for t in range(H)]
    banks=[addvar(lb=0,ub=2000,integ=1) for t in range(H)]
    z=[{(f,k):addvar(obj=MPC_DISCOUNT**t*4*max(0,k-f)) for f in F for k in M} for t in range(H)]
    coeffs=[]; lo=[];hi=[]
    def constraint(terms,l=-np.inf,u=np.inf):
        coeffs.append(terms);lo.append(float(l));hi.append(float(u))
    ft0=integer(engine.ft)
    if not 0<=ft0<=5:raise ValueError('Initial FT count must be between 0 and 5')
    for t in range(H):
        # 15-player legal squad + 11-player XI and single XI captain.
        constraint({x[t][j]:1 for j in range(n)},15,15)
        constraint({y[t][j]:1 for j in range(n)},11,11)
        constraint({captain[t][j]:1 for j in range(n)},1,1)
        for pos,count in POSITION_COUNTS.items():
            constraint({x[t][j]:1 for j,pid in enumerate(pool) if integer(engine.players[pid]['element_type'])==pos}, count,count)
        for pos,L,U in ((1,1,1),(2,3,5),(3,2,5),(4,1,3)):
            constraint({y[t][j]:1 for j,pid in enumerate(pool) if integer(engine.players[pid]['element_type'])==pos},L,U)
        for team in engine.teams:
            idx=[j for j,pid in enumerate(pool) if integer(engine.players[pid]['team'])==team]
            if idx:constraint({x[t][j]:1 for j in idx},0,3)
        for j in range(n):
            constraint({y[t][j]:1,x[t][j]:-1},u=0)
            constraint({captain[t][j]:1,y[t][j]:-1},u=0)
            # Squad flow: x(t) = x(t-1) + buy - sold.
            flow={x[t][j]:1,buy[t][j]:-1,sold[t][j]:1}
            if t:flow[x[t-1][j]]=-1
            prior=0 if t else int(owned_ids[j])
            constraint(flow,prior,prior)
            # No instantaneous buy-and-sell same GW.
            constraint({buy[t][j]:1,sold[t][j]:1},u=1)
        # Each player is bought at most once during the horizon; never
        # re-buy an originally owned player (preserves exact sale-price accounting).
        constraint({z[t][(f,k)]:1 for f in F for k in M},1,1)
        constraint({**{buy[t][j]:1 for j in range(n)},**{z[t][(f,k)]:-k for f in F for k in M}},0,0)
        constraint({**{sold[t][j]:1 for j in range(n)},**{z[t][(f,k)]:-k for f in F for k in M}},0,0)
        if t==0:
            constraint({z[t][(f,k)]:1 for k in M for f in F if f==ft0},1,1)
        else:
            for f in F:
                constraint({**{z[t][(f,k)]:1 for k in M},
                            **{z[t-1][(fp,kp)]:-1 for fp in F for kp in M if min(5,max(0,fp-kp)+1)==f}},0,0)
        money={banks[t]:1}
        if t:money[banks[t-1]]=-1
        for j in range(n):
            money[buy[t][j]]=int(cost[j])
            money[sold[t][j]]=-int(sell[j])
        constraint(money, engine.bank if t==0 else 0,engine.bank if t==0 else 0)
    for j,pid in enumerate(pool):
        constraint({buy[t][j]:1 for t in range(H)},0,0 if pid in owned else 1)
        constraint({sold[t][j]:1 for t in range(H)},0,1)
    rr=[];cc=[];vv=[]
    for r,terms in enumerate(coeffs):
        for col,val in terms.items():
            if val:rr.append(r);cc.append(col);vv.append(val)
    A=coo_matrix((vv,(rr,cc)),shape=(len(coeffs),len(cobj))).tocsr()
    ans=milp(np.array(cobj),integrality=np.array(integral),
             bounds=Bounds(np.array(lower),np.array(upper)),
             constraints=LinearConstraint(A,np.array(lo),np.array(hi)),
             options={'time_limit':MPC_TIME_LIMIT_SECONDS,'mip_rel_gap':.065})
    if ans.x is None:raise RuntimeError('Multi-period MILP produced no feasible incumbent: '+str(ans.message))
    val=ans.x
    steps=[];current=set(owned);bank=integer(engine.bank);FT=ft0;cost_basis={pid:engine.owned_prices[pid] for pid in owned};score=0.
    for t,gw in enumerate(engine.events):
        selected=tuple(sorted(pid for j,pid in enumerate(pool) if val[x[t][j]]>.5))
        outs=sorted(pid for j,pid in enumerate(pool) if val[sold[t][j]]>.5)
        ins=sorted(pid for j,pid in enumerate(pool) if val[buy[t][j]]>.5)
        m=len(ins)
        if len(outs)!=m or m>K or set(selected)!=(current-set(outs))|set(ins) or not engine.valid_squad(selected):
            raise ValueError(f'MPC GW{gw}: squad flow illegal')
        if len([i for i in selected if val[y[t][ix[i]]]> .5])!=11 or len([i for i in selected if val[captain[t][ix[i]]]> .5])!=1:
            raise ValueError(f'MPC GW{gw}: lineup/captain illegal')
        for pid in outs:bank+=cost_basis.pop(pid)
        for pid in ins:bank-=integer(engine.players[pid]['now_cost']);cost_basis[pid]=integer(engine.players[pid]['now_cost'])
        hit=4*max(0,m-FT)
        if bank<0 or bank!=round(val[banks[t]]):raise ValueError(f'MPC GW{gw}: bank reconciliation failed')
        step={'gw':gw,'action':'TRANSFER' if m else 'HOLD','out':outs[0] if m==1 else None,
              'into':ins[0] if m==1 else None,'out_ids':outs,'in_ids':ins,
              'hit':hit,'xp':engine.lineup(selected,gw)['expected'],
              'free_transfers_before':FT,'bank':bank,'ids':selected}
        steps.append(step);score+=MPC_DISCOUNT**t*(step['xp']-hit-.35*m)
        FT=min(5,max(0,FT-m)+1);current=set(selected)
    answer=Route(steps[-1]['ids'],bank,FT,score,steps)
    engine.mpc_last_audit={'status':str(ans.message),'gap':getattr(ans,'mip_gap',None),
        'candidate_pool':n,'horizon':H,'max_moves_per_gw':K,
        'future_price_model':'fixed market prices; original owned selling prices frozen',
        'exact_global_optimum':bool(ans.status==0 and getattr(ans,'mip_gap',1)==0)}
    return answer

# Preserve the legacy beam as a diagnostic fallback only; do not silently
# pretend it is a MILP when the external solver cannot find a feasible incumbent.
_V45_ORIGINAL_BEAM_PLAN=DecisionEngine.plan

def v45_plan(self: DecisionEngine) -> Route:
    key=(tuple(sorted(self.owned)),integer(self.bank),integer(self.ft),
         tuple(self.events),tuple(sorted(self.owned_prices.items())))
    if getattr(self,'_mpc_cached_key',None)==key:
        return self._mpc_cached_route
    try:
        result=mpc_plan(self)
        self._mpc_cached_key=key;self._mpc_cached_route=result
        if len(self.events) >= 5:
            self._mpc_full_horizon_audit = dict(getattr(self, 'mpc_last_audit', {}))
        return result
    except (RuntimeError,ValueError) as problem:
        self.mpc_last_audit={'status':'FALLBACK_BEAM','reason':str(problem),
          'warning':'MPC MILP failed; old beam planner used with reduced completeness.'}
        print('⚠️ MPC MILP fallback to legacy beam:',problem)
        return _V45_ORIGINAL_BEAM_PLAN(self)
DecisionEngine.plan=v45_plan


def v45_route_audit(engine, steps, verified_ft):
    """Hard-check EVERY multi-transfer step, not only the first named pair."""
    issues=[]; current=set(engine.owned);bank=integer(engine.bank);FT=integer(verified_ft)
    prices=dict(engine.owned_prices)
    if len(steps)!=len(engine.events):
        return [Diagnostic('ERROR','ROUTE_LENGTH','Route does not cover entire MPC horizon.')]
    for t,step in enumerate(steps):
        gw=engine.events[t];selected=set(step['ids'])
        outs=step.get('out_ids') if 'out_ids' in step else ([step['out']] if step.get('action')=='TRANSFER' else [])
        ins=step.get('in_ids') if 'in_ids' in step else ([step['into']] if step.get('action')=='TRANSFER' else [])
        outs=list(outs or []);ins=list(ins or [])
        if (step.get('gw')!=gw or not engine.valid_squad(tuple(selected)) or
            len(outs)!=len(ins) or len(set(outs))!=len(outs) or len(set(ins))!=len(ins) or
            not set(outs).issubset(current) or set(ins)&current or
            selected!=(current-set(outs))|set(ins)):
            issues.append(Diagnostic('ERROR','ROUTE_FLOW',f'GW{gw} transfer flow/squad is invalid.'));continue
        if bool(ins)!=(step['action']=='TRANSFER'):
            issues.append(Diagnostic('ERROR','ROUTE_ACTION',f'GW{gw} action and transfer list mismatch.'))
        for p in outs:bank+=prices.pop(p)
        for p in ins:
            cost=integer(engine.players[p]['now_cost']);bank-=cost;prices[p]=cost
        expected_hit=4*max(0,len(ins)-FT)
        if bank<0 or step.get('bank')!=bank or step.get('hit')!=expected_hit or step.get('free_transfers_before')!=FT:
            issues.append(Diagnostic('ERROR','ROUTE_BUDGET',f'GW{gw} bank, FT, or hits did not reconcile.'))
        current=selected;FT=min(5,max(0,FT-len(ins))+1)
    return issues

# Rebind global auditor used by compare_strategies; chip10_validate_path
# below is independently upgraded to support multiple moves.
route_audit=v45_route_audit


In [ ]:
# V4.3 additive chip-timing layer. Existing V4.2 fetch, verified-state gate,
# stochastic player model, optimizer, HTML and 4K renderer remain unchanged.
from contextlib import contextmanager
from typing import Iterator

CHIP10_VERSION = '4.3-chip-calendar'
CHIP10_HORIZON_WEEKS = 10
CHIP10_MIN_MC = 10000
CHIP10_MIN_PROBABILITY = 0.70
CHIP10_MIN_NET_MEDIAN = 3.0
CHIP10_MIN_STABLE_MINUTES = 3


@contextmanager
def chip10_engine_state(engine: Any, steps: list[dict[str, Any]], index: int) -> Iterator[None]:
    """Forecast a future transfer state, then restore the engine even on solver errors."""
    original = (engine.owned, engine.bank, engine.ft, engine.owned_prices, engine.budget, engine.events)
    step = steps[index]
    prior_step = steps[index-1] if index > 0 else None
    current_ids = tuple(sorted(prior_step['ids'])) if prior_step else original[0]
    try:
        engine.owned = current_ids
        engine.bank = integer(prior_step['bank']) if prior_step else integer(original[1])
        engine.ft = integer(step['free_transfers_before'])
        # Reconstruct modeled ownership cost basis from each earlier transfer.
        # Future price levels remain fixed-market assumptions, never 'verified'.
        modeled_basis=dict(original[3]); modeled_bank=integer(original[1])
        for previous in steps[:index]:
            outs=previous.get('out_ids', [previous['out']] if previous.get('out') is not None else [])
            ins=previous.get('in_ids', [previous['into']] if previous.get('into') is not None else [])
            for pid in outs:
                modeled_bank+=modeled_basis.pop(pid)
            for pid in ins:
                price=integer(engine.players[pid]['now_cost'])
                modeled_basis[pid]=price;modeled_bank-=price
        if prior_step and modeled_bank!=integer(prior_step['bank']):
            raise ValueError('Future GW bank and cost basis failed to reconcile')
        engine.owned_prices = {pid:modelled for pid,modelled in modeled_basis.items() if pid in current_ids}
        engine.budget = engine.bank + sum(engine.owned_prices.values())
        engine.events = original[5][index:]
        if engine.bank < 0 or not engine.valid_squad(current_ids):
            raise ValueError(f'GW{step["gw"]} pre-chip baseline is not budget-legal.')
        yield
    finally:
        engine.owned, engine.bank, engine.ft, engine.owned_prices, engine.budget, engine.events = original


def chip10_week_scores(monte: AuditedMonteCarlo, steps: list[dict[str, Any]], scale: float = 1.0) -> list[np.ndarray]:
    if len(steps) != len(monte.engine.events):
        raise ValueError('Chip schedule steps do not cover every upcoming Gameweek.')
    return [monte.lineup(tuple(step['ids']), integer(step['gw']), minute_scale=scale)
            - integer(step.get('hit', 0)) for step in steps]


def chip10_validate_path(engine: Any, steps: list[dict[str, Any]], baseline: list[dict[str, Any]], wc_index: int) -> None:
    if len(steps) != len(engine.events):
        raise ValueError('Wildcard candidate does not cover the complete 10-week horizon.')
    wc_bank = None
    wc_prices: dict[int, int] = {}
    for index, step in enumerate(steps):
        gw = engine.events[index]
        ids = tuple(sorted(step['ids']))
        if gw != step['gw'] or not engine.valid_squad(ids):
            raise ValueError(f'GW{gw}: illegal candidate squad.')
        lineup = engine.lineup(ids, gw)
        if len(lineup['xi']) != 11 or len(set(lineup['xi'])) != 11 or lineup['captain'] not in lineup['xi'] or lineup['vice'] not in lineup['xi']:
            raise ValueError(f'GW{gw}: illegal starting XI/captain/vice.')
        if index < wc_index and ids != tuple(sorted(baseline[index]['ids'])):
            raise ValueError(f'GW{gw}: future Wildcard altered an earlier squad.')
        if index == wc_index and step['action'] != 'WILDCARD':
            raise ValueError(f'GW{gw}: missing Wildcard on proposed chip week.')
        if integer(step['bank']) < 0 or integer(step.get('hit', 0)) < 0:
            raise ValueError(f'GW{gw}: illegal bank/hit state.')
        if index == wc_index:
            wc_bank = integer(step['bank'])
            wc_prices = {pid: engine.owned_prices.get(pid, integer(engine.players[pid]['now_cost'])) for pid in ids}
        elif index > wc_index:
            if step['action'] == 'TRANSFER':
                outs = step.get('out_ids', [step['out']] if step.get('out') is not None else [])
                ins = step.get('in_ids', [step['into']] if step.get('into') is not None else [])
                if len(outs) != len(ins) or any(p not in wc_prices for p in outs) or any(p in wc_prices for p in ins):
                    raise ValueError(f'GW{gw}: invalid post-Wildcard transfers')
                for outgoing in outs:
                    wc_bank = int(wc_bank) + wc_prices.pop(outgoing)
                for incoming in ins:
                    purchase_price = integer(engine.players[incoming]['now_cost'])
                    wc_bank -= purchase_price
                    wc_prices[incoming] = purchase_price
            if wc_bank != integer(step['bank']) or set(wc_prices) != set(ids):
                raise ValueError(f'GW{gw}: post-Wildcard transfer bank/roster did not reconcile')


def chip10_valid_chips_by_week(engine: Any, state: dict[str, Any], event: int) -> dict[str, bool]:
    """Do not infer a fresh chip after a half-season boundary from last-half private state."""
    current_half = 1 if engine.events[0] <= 19 else 2
    target_half = 1 if event <= 19 else 2
    if target_half != current_half:
        return {chip: False for chip in ('wildcard', 'freehit', 'triplecaptain', 'benchboost')}
    return {chip: state['chip_available'].get(chip) is True
            for chip in ('wildcard', 'freehit', 'triplecaptain', 'benchboost')}


def chip10_candidate_wc(engine: Any, monte: AuditedMonteCarlo,
                       baseline_steps: list[dict[str, Any]], index: int,
                       verified_ft_after_wc: int | None) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    with chip10_engine_state(engine, baseline_steps, index):
        event = engine.events[0]
        optimized = engine.optimize_squad('wildcard', gw=event)
        if not optimized.get('success'):
            raise ValueError(f'GW{event} WC MILP failed: {optimized.get("reason", "unknown")}')
        selected = tuple(sorted(optimized['selected']))
        if not engine.valid_squad(selected):
            raise ValueError(f'GW{event} illegal Wildcard XV')
        if integer(optimized['remaining_bank']) < 0:
            raise ValueError(f'GW{event} Wildcard exceeds the verified model budget')
        # Future FTs remain provisional. Current verified post-WC FT can be used only for this GW.
        wc_ft = (verified_ft_after_wc if index == 0 and type(verified_ft_after_wc) is int
                 else integer(baseline_steps[index]['free_transfers_before']))
        suffix = wc_repair_route(engine, optimized, wc_ft)
        detailed = {'gw': event, 'selected_ids': list(selected),
                    'remaining_bank_tenths': integer(optimized['remaining_bank']),
                    'solver_status': str(optimized.get('status')), 'future_free_transfers_assumed': wc_ft,
                    'future_prices_assumed_constant': True,
                    'model_budget_tenths': integer(engine.budget)}
    combined = baseline_steps[:index] + suffix
    chip10_validate_path(engine, combined, baseline_steps, index)
    return combined, detailed


def chip10_minutes_sensitivity(monte: AuditedMonteCarlo,
                              candidate: list[dict[str, Any]], baseline: list[dict[str, Any]],
                              future_option_cost: float) -> list[dict[str, Any]]:
    rows = []
    for name, scale in [('conservative', 0.85), ('base', 1.0), ('aggressive', 1.15)]:
        candidate_total = np.sum(chip10_week_scores(monte, candidate, scale), axis=0)
        baseline_total = np.sum(chip10_week_scores(monte, baseline, scale), axis=0)
        net_median = float(np.median(candidate_total - baseline_total)) - future_option_cost
        rows.append({'case': name, 'minute_scale': scale, 'net_median_gain': round(net_median, 2),
                     'supports_chip': net_median >= CHIP10_MIN_NET_MEDIAN})
    return rows


def chip10_one_week_chip_gains(engine: Any, monte: AuditedMonteCarlo,
                               baseline_steps: list[dict[str, Any]],
                               availability: dict[str, bool]) -> tuple[dict[str, list[dict[str, Any]]], list[Diagnostic]]:
    alerts: list[Diagnostic] = []
    results: dict[str, list[dict[str, Any]]] = {'triplecaptain': [], 'benchboost': [], 'freehit': []}
    for index, step in enumerate(baseline_steps):
        gw = integer(step['gw'])
        underlying = monte.lineup(tuple(step['ids']), gw)
        for chip in ('triplecaptain', 'benchboost', 'freehit'):
            if not availability.get(chip):
                continue
            # A Free Hit cannot meaningfully be compared with a normal transfer
            # in the same GW without modeling its transfer rollover.
            if chip == 'freehit' and step['action'] == 'TRANSFER':
                results[chip].append({'gw': gw, 'eligible': False,
                                      'reason': 'Baseline schedules a normal transfer; FT rollover not independently modeled.'})
                continue
            try:
                if chip == 'freehit':
                    with chip10_engine_state(engine, baseline_steps, index):
                        optimized = engine.optimize_squad('freehit', gw=gw)
                        if not optimized.get('success'):
                            raise ValueError(str(optimized.get('reason', 'No valid Free Hit XV')))
                        selected = tuple(sorted(optimized['selected']))
                        if not engine.valid_squad(selected) or integer(optimized['remaining_bank']) < 0:
                            raise ValueError('Free Hit legality/budget failed')
                    candidate = monte.lineup(selected, gw)
                    squads = list(selected)
                else:
                    candidate = monte.lineup(tuple(step['ids']), gw, chip=chip)
                    squads = list(step['ids'])
                gain = candidate - underlying
                stats = median_stats(gain)
                results[chip].append({'gw': gw, 'eligible': True, 'chip': chip,
                                      'mean_gain': stats['mean'], 'median_gain': stats['median'],
                                      'p25_gain': stats['p25'], 'p75_gain': stats['p75'],
                                      'p_gain_positive': round(float(np.mean(gain > 0)), 4),
                                      'selected_ids': squads, 'assumes_no_price_changes': True})
            except (ValueError, RuntimeError) as error:
                alerts.append(Diagnostic('WARNING', 'CHIP_CANDIDATE_FAILED', f'GW{gw} {chip}: {error}'))
                results[chip].append({'gw': gw, 'eligible': False, 'reason': str(error)})
    return results, alerts


def chip10_pick_calendar(engine: Any, state: dict[str, Any], monte: AuditedMonteCarlo,
                         original: dict[str, Any]) -> tuple[dict[str, Any], list[Diagnostic]]:
    alerts: list[Diagnostic] = []
    if len(engine.events) != CHIP10_HORIZON_WEEKS:
        return {}, [Diagnostic('ERROR', 'CHIP10_HORIZON',
                               f'Expected {CHIP10_HORIZON_WEEKS} future official GWs, got {len(engine.events)}. Cannot claim 10-GW evaluation.')]
    if monte.n < CHIP10_MIN_MC:
        return {}, [Diagnostic('ERROR', 'CHIP10_MC',
                               f'Expected at least {CHIP10_MIN_MC} paired scenarios, got {monte.n}.')]
    if original.get('best_no_chip') not in ('repair', 'hold'):
        return {}, [Diagnostic('ERROR', 'CHIP10_BASELINE', 'Best verified no-chip route is unavailable.')]
    route = engine.plan()
    baseline = route.steps if original['best_no_chip'] == 'repair' else hold_route(engine)
    if route_audit(engine, route.steps, state['ft_after_moves']):
        return {}, [Diagnostic('ERROR', 'CHIP10_ROUTE', 'A baseline route failed the original transfer audit.')]
    base_scores = chip10_week_scores(monte, baseline)
    baseline_total = np.sum(base_scores, axis=0) - integer(state['displayed_hit_cost'])
    source = {'state_derived_utc': state['verified_at_utc'], 'api_fetched_utc': engine.snapshot.get('collected_utc', engine.snapshot.get('retrieved_utc', 'ARCHIVED_OFFLINE_SNAPSHOT')),
              'entry_id': engine.snapshot['entry_id'], 'horizon_gws': list(engine.events), 'draws': monte.n}
    options = adaptive_option_cost(engine)
    # The option cost is kept when GWs remain before the actual half-season expiry.
    boundary = 19 if engine.events[0] <= 19 else 38
    remaining_beyond_horizon = max(0, boundary - engine.events[-1])
    option_cost = round(options['future_wildcard_option_value_points'] * min(1, remaining_beyond_horizon / 8.0), 2)
    wc_candidates: list[dict[str, Any]] = []
    computed_paths: dict[int, list[dict[str, Any]]] = {}
    if state['chip_available']['wildcard']:
        for index, gw in enumerate(engine.events):
            print(f'  🧮 Chip-timing study: GW{gw} ({index + 1}/{len(engine.events)}); {monte.n:,} shared scenarios')
            if not chip10_valid_chips_by_week(engine, state, gw)['wildcard']:
                wc_candidates.append({'gw': gw, 'eligible': False, 'reason': 'Chip entitlement not verified across the half-season boundary.'})
                continue
            try:
                candidate_steps, details = chip10_candidate_wc(engine, monte, baseline, index, state.get('ft_after_wildcard'))
                candidate_samples = np.sum(chip10_week_scores(monte, candidate_steps), axis=0)
                candidate_samples -= integer(state['displayed_hit_cost']) if index > 0 else 0
                delta = candidate_samples - baseline_total
                stats = median_stats(delta)
                p_win = float(np.mean(delta > 0))
                net_median = float(stats['median']) - option_cost
                candidate = {'gw': gw, 'eligible': True, 'chip': 'wildcard',
                             'mean_gain': stats['mean'], 'median_gain': stats['median'],
                             'p25_gain': stats['p25'], 'p75_gain': stats['p75'],
                             'p_win': round(p_win, 4), 'p_negative_gain': round(float(np.mean(delta < 0)), 4),
                             'option_cost_points': option_cost, 'net_median_gain': round(net_median, 2),
                             'selected_ids': details['selected_ids'], 'budget': details['model_budget_tenths'],
                             'solver_status': details['solver_status'],
                             'is_provisional_future_projection': index > 0,
                             'qualifies': net_median >= CHIP10_MIN_NET_MEDIAN and p_win >= CHIP10_MIN_PROBABILITY
                                          and stats['p25'] > -8}
                wc_candidates.append(candidate)
                computed_paths[gw] = candidate_steps
            except (ValueError, RuntimeError, KeyError, AssertionError) as error:
                alerts.append(Diagnostic('WARNING', 'CHIP10_CANDIDATE_FAILED', f'GW{gw} Wildcard: {error}'))
                wc_candidates.append({'gw': gw, 'eligible': False, 'reason': str(error)})
    valid_wc = sorted((c for c in wc_candidates if c.get('eligible') and c.get('qualifies')),
                      key=lambda row: (row['net_median_gain'], row['p25_gain']), reverse=True)
    chosen_wc = None
    # Sensitivity-test candidate timing in preference order; a later stable option
    # may still be useful even when a higher-point alternative proves fragile.
    for candidate in valid_wc:
        try:
            minutes_cases = chip10_minutes_sensitivity(monte, computed_paths[candidate['gw']], baseline, option_cost)
            candidate['minutes_sensitivity'] = minutes_cases
            if sum(case['supports_chip'] for case in minutes_cases) >= CHIP10_MIN_STABLE_MINUTES:
                chosen_wc = candidate
                break
            candidate['qualifies'] = False
            candidate['reason'] = 'Fails conservative/base/aggressive expected-minutes stability.'
        except (ValueError, RuntimeError) as error:
            candidate['qualifies'] = False
            candidate['reason'] = f'Sensitivity could not be verified: {error}'
            alerts.append(Diagnostic('WARNING', 'CHIP10_SENSITIVITY_FAILED', f'GW{candidate["gw"]}: {error}'))
    chips, other_alerts = chip10_one_week_chip_gains(engine, monte, baseline, state['chip_available'])
    alerts.extend(other_alerts)
    chip_options = []
    for name, rows in chips.items():
        for row in rows:
            if row.get('eligible') and row.get('median_gain', 0) >= 3 and row.get('p_gain_positive', 0) >= .60:
                chip_options.append({'gw': row['gw'], 'chip': name, 'median_gain': row['median_gain'],
                                     'p25_gain': row['p25_gain'], 'p75_gain': row['p75_gain'],
                                     'provisional': bool(state.get('public_provisional')) or row['gw'] != engine.events[0]})
    if chosen_wc:
        chip_options.append({'gw': chosen_wc['gw'], 'chip': 'wildcard',
                             'median_gain': chosen_wc['median_gain'], 'p25_gain': chosen_wc['p25_gain'],
                             'p75_gain': chosen_wc['p75_gain'], 'provisional': bool(state.get('public_provisional')) or chosen_wc['gw'] != engine.events[0]})
    # Exact finite DP for the *approximated additive chip values*: each verified chip
    # at most once and no more than one chip in a Gameweek. Joint player selection
    # after multiple different chips is explicitly NOT claimed to be globally exact.
    chip_names = ['wildcard', 'freehit', 'triplecaptain', 'benchboost']
    chip_mask = {name: 1 << index for index, name in enumerate(chip_names)}
    dp: dict[int, tuple[float, list[dict[str, Any]]]] = {0: (0.0, [])}
    for gw in engine.events:
        next_dp = dict(dp)
        for used_mask, (utility, entries) in dp.items():
            for option in chip_options:
                if option['gw'] != gw or used_mask & chip_mask[option['chip']]:
                    continue
                # Wildcard utility has already cleared the option-cost and downside gates.
                effective = option['median_gain'] + .1 * option['p25_gain']
                if option['chip'] == 'wildcard':
                    effective -= option_cost
                if effective <= 0:
                    continue
                new_mask = used_mask | chip_mask[option['chip']]
                proposal = utility + effective
                if proposal > next_dp.get(new_mask, (-1e12, []))[0]:
                    next_dp[new_mask] = (proposal, entries + [option])
        dp = next_dp
    planned = max(dp.values(), key=lambda item: item[0])[1]
    planned.sort(key=lambda item: item['gw'])
    first = next((row for row in planned if row['gw'] == engine.events[0]), None)
    now = first['chip'].upper() if first else ('TRANSFER' if baseline[0]['action'] == 'TRANSFER' else 'HOLD')
    if now == 'WILDCARD' and not chosen_wc:
        raise RuntimeError('No qualified current Wildcard plan was found.')
    result = {'version': CHIP10_VERSION, 'source': {**source, 'public_provisional': bool(state.get('public_provisional'))},
              'baseline': 'best no-chip path using the input state, flagged public-provisional when applicable',
              'baseline_median': round(float(np.median(baseline_total)), 2),
              'wildcard_opportunity_cost_points': option_cost, 'wildcard_candidates': wc_candidates,
              'other_chip_candidates': chips, 'recommended_chip_calendar': planned,
              'next_gameweek_chip': first['chip'] if first else None, 'next_action': now,
              'candidate_completeness': {'wildcard': sum(c.get('eligible') is True for c in wc_candidates),
                                         'expected_wildcard': sum(chip10_valid_chips_by_week(engine, state, e)['wildcard'] for e in engine.events),
                                         'other_chips': {k: sum(row.get('eligible') is True for row in v) for k, v in chips.items()}},
              'limits': ['Gameweek 2+ transfers and prices are estimates; regenerate every deadline.',
                         'Current pre-deadline transfers, displayed hit and selling prices cannot be read from public API. All output is provisional.',
                         'Best single chip-GW choices, not an exact joint multi-chip dynamic program.',
                         'Cross-half chip renewal is not assumed unless reverified.',
                         'Future Wildcard assumes current prices, no injuries not yet reported, and estimated FT rollover.',
                         'Chip scores reflect model uncertainty, not calibrated probability of rank finish.'],
              'probabilistic_not_guaranteed': True}
    coverage_complete = (result['candidate_completeness']['wildcard'] ==
                         result['candidate_completeness']['expected_wildcard'])
    result['chip_timing_coverage_complete'] = coverage_complete
    # A missing eligible week invalidates claims of optimal timing; retaining
    # evaluated alternatives is useful for research but not chip execution.
    if not coverage_complete:
        result['optimization_verdict'] = 'INCOMPLETE — research candidates, not an optimized schedule'
        for idea in result['recommended_chip_calendar']:
            idea['provisional'] = True
        if not state.get('public_provisional'):
            result['next_action'] = ('TRANSFER' if original['best_no_chip'] == 'repair' and baseline[0]['action'] == 'TRANSFER'
                                     else 'HOLD')
            result['next_gameweek_chip'] = None
            result['recommended_chip_calendar'] = []
    else:
        result['optimization_verdict'] = 'EVALUATED-POOL HEURISTIC — not globally optimal'
    if state['chip_available']['wildcard'] and not coverage_complete:
        alerts.append(Diagnostic('WARNING', 'CHIP10_INCOMPLETE',
                                'Not every possible Wildcard week could be evaluated; refusing an optimized chip-timing verdict.'))
    return result, alerts


def chip10_render_tab(decision: dict[str, Any]) -> str:
    report = decision.get('chip_calendar_10gw', {})
    weeks = report.get('source', {}).get('horizon_gws', [])
    candidates = report.get('wildcard_candidates', [])
    rows = ''.join(f'<tr><td>GW{esc(c.get("gw"))}</td><td>{"Evaluated" if c.get("eligible") else "Unavailable"}</td>'
                   f'<td>{esc(c.get("median_gain", "—"))}</td><td>{esc(c.get("p25_gain", "—"))}–{esc(c.get("p75_gain", "—"))}</td>'
                   f'<td>{esc(c.get("p_win", "—"))}</td><td>{esc(c.get("net_median_gain", "—"))}</td>'
                   f'<td>{esc(c.get("reason", "Eligible" if c.get("qualifies") else "Below decision threshold"))}</td></tr>' for c in candidates)
    calendar_rows = ''.join(f'<tr><td>GW{esc(r["gw"])}</td><td>{esc(r["chip"])}</td>'
                            f'<td>{esc(r["median_gain"])}</td><td>{"Recheck before deadline" if r["provisional"] else "Review now"}</td></tr>'
                            for r in report.get('recommended_chip_calendar', []))
    return (f'<section class="page" id="page-chipcalendar" role="tabpanel"><h1 class="headline">Next 10 Gameweeks · Chip Timing</h1>'
            f'<div class="card"><h2>Recommended current action: {esc(report.get("next_action", "UNAVAILABLE"))}</h2>'
            f'<p>Public/model state derived at {esc(report.get("source", {}).get("state_verified_utc", "unknown"))}; '
            f'official data {esc(report.get("source", {}).get("api_fetched_utc", "unknown"))}.</p>'
            f'<p>Evaluated GW{esc(weeks[0] if weeks else "?")}–GW{esc(weeks[-1] if weeks else "?")} '
            f'with {esc(report.get("source", {}).get("draws", 0))} paired Monte Carlo draws.</p>'
            f'<p>Wildcard future-option penalty: {esc(report.get("wildcard_opportunity_cost_points", 0))} points. '
            f'Timing is provisional; only this week’s action is actionable.</p></div>'
            '<div class="card"><h2>Proposed chip calendar</h2><div class="table-scroll"><table><tr><th>GW</th><th>Chip</th><th>Median gain</th><th>Status</th></tr>'
            + (calendar_rows or '<tr><td colspan="4">No chip clears the evidence and risk thresholds now.</td></tr>')
            + '</table></div></div><div class="card"><h2>Wildcard: evaluate every available Gameweek</h2><div class="table-scroll"><table><tr><th>GW</th><th>Coverage</th><th>Median edge</th><th>P25–P75</th><th>P(win)</th><th>Net edge</th><th>Audit</th></tr>'
            + (rows or '<tr><td colspan="7">Wildcard unavailable in this half.</td></tr>')
            + '</table></div></div><div class="card"><h2>Limitations</h2><p>'
            + esc(' • '.join(report.get('limits', [])))
            + '</p><p>Probabilistic estimates, not guarantees. Re-run after every deadline and before confirming any chip.</p></div></section>')


_CHIP10_COMPARE_ORIGINAL = compare_strategies
_CHIP10_ATTACH_ORIGINAL = attach_audit_tab
_CHIP10_RUN_ORIGINAL = run_audited


def compare_strategies(engine: Any, state: dict[str, Any], monte: AuditedMonteCarlo) -> tuple[dict[str, Any], list[Diagnostic]]:
    existing, issues = _CHIP10_COMPARE_ORIGINAL(engine, state, monte)
    # Save the actual full-horizon audit before later chip scenarios temporarily
    # replace the engine events with one or a few GWs.
    full_horizon_mpc_audit = dict(getattr(engine, 'mpc_last_audit', {}) or {})
    if any(d.severity == 'ERROR' for d in issues):
        return existing, issues
    schedule, new_issues = chip10_pick_calendar(engine, state, monte, existing)
    issues.extend(new_issues)
    if any(d.severity == 'ERROR' for d in issues):
        return existing, issues
    existing['chip_calendar_10gw'] = schedule
    if full_horizon_mpc_audit:
        engine.mpc_last_audit = full_horizon_mpc_audit
        engine._mpc_full_horizon_audit = full_horizon_mpc_audit
        existing['mpc_solver_audit'] = full_horizon_mpc_audit
    next_action = schedule['next_action']
    # The new ten-week chip audit is authoritative for current-chip timing.
    if next_action in ('WILDCARD', 'FREEHIT', 'TRIPLECAPTAIN', 'BENCHBOOST'):
        existing['recommended_action'] = next_action
        current = {'gw': engine.events[0], 'action': next_action}
        if next_action == 'WILDCARD':
            candidate = next(r for r in schedule['wildcard_candidates'] if r['gw'] == engine.events[0])
            current['selected'] = candidate['selected_ids']
            existing['wildcard']['selected_ids'] = candidate['selected_ids']
        existing['current_action_only'] = current
    elif existing['recommended_action'] == 'WILDCARD':
        # Prior six-week mean-only rule must not override the audited 10-week timing choice.
        existing['recommended_action'] = next_action
        existing['current_action_only'] = {'gw': engine.events[0], 'action': next_action}
    existing['main_assumptions'].append('Ten chip windows evaluated from available FPL input data; predeadline private state may differ.')
    existing['main_risks'].append('Chip schedule is not a global joint chip+transfer optimum; replanning is mandatory.')
    if schedule['candidate_completeness']['wildcard'] < schedule['candidate_completeness']['expected_wildcard']:
        existing['confidence'] = 'Low'
    return existing, issues


def attach_audit_tab(original_html: str, panel: str, recommendation: str, decision: dict[str, Any]) -> str:
    original = _CHIP10_ATTACH_ORIGINAL(original_html, panel, recommendation, decision)
    if 'chip_calendar_10gw' not in decision:
        return original
    marker = '</nav></header>'
    original = original.replace(marker,
                                '<button class="tab" role="tab" type="button" data-tab="chipcalendar" aria-selected="false">10-GW Chip Calendar</button>' + marker, 1)
    original = original.replace('<section class="page" id="page-auditdecision"',
                                chip10_render_tab(decision) + '<section class="page" id="page-auditdecision"', 1)
    original = original.replace('5,000 paired scenarios', f'{decision["scenario_count"]:,} paired scenarios')
    original = original.replace('6-GW Plan', '10-GW Plan')
    original = original.replace('6-GW', '10-GW')
    return original


def run_audited(*args: Any, **kwargs: Any) -> tuple[Path, Path, Path]:
    result = _CHIP10_RUN_ORIGINAL(*args, **kwargs)
    payload = json.loads(result[2].read_text(encoding='utf-8'))
    chip_report = payload.get('audited_decision', {}).get('chip_calendar_10gw')
    if chip_report:
        out = result[2].parent
        atomic_json(out / 'FPLVORTEX_CHIP_CALENDAR_LATEST.json', chip_report)
        archive = out / 'Archive'
        archive.mkdir(exist_ok=True, parents=True)
        stamp = utc_now().strftime('%Y%m%dT%H%M%S%fZ')
        atomic_json(archive / f'chip_calendar_{stamp}.json', chip_report, immutable=True)
        print('✅ 10-GW chip calendar:', out / 'FPLVORTEX_CHIP_CALENDAR_LATEST.json')
    return result


In [ ]:
# FPLVORTEX V4.5 · one-click research; verified manager_state.json automatically detected if present.
HORIZON = 10
AUDIT_HORIZON = 10
AUDIT_MONTE_CARLO = 10000
ENABLE_REAL_HEADSHOTS = True
BACKTEST_ONLY = False

if BACKTEST_ONLY:
    REPORT_DIR = google_drive_folder()
    BACKTEST_CSV, BACKTEST_JSON = run_historical_replay(REPORT_DIR / 'Archive')
    print('Historical replay saved:', BACKTEST_CSV, BACKTEST_JSON)
else:
    try:
        HTML_FILE, PNG_FILE, JSON_FILE = run_audited()
        # Display the model candidates immediately in Colab (including mobile),
        # not only inside a separate Google Drive HTML file.
        from IPython.display import display, HTML
        _auto_payload = json.loads(Path(JSON_FILE).read_text(encoding='utf-8'))
        _auto_brief = _auto_payload.get('auto_research_brief')
        if _auto_brief:
            display(HTML('<div style="background:#091829;color:#eefdf9;border:1px solid #38d8c1;border-radius:14px;padding:18px;margin:12px 0">'
                         + automatic_brief_panel(_auto_brief) + '</div>'))
        print('✅ Saved 10-GW audited report, HTML dashboard, 4K PNG and chip calendar to Google Drive.')
        print('📌 Open the HTML report and read AUTOMATIC FPL RESEARCH BRIEF on the first page.')
        print('🔒 Only the verified-state report can contain executable first-GW advice; otherwise PROVISIONAL – research only.')
    except AuditBlocked as problem:
        print('⛔ The FPL API or solver could not supply a safe public-data analysis. No manual form will be requested.')
        for item in problem.diagnostics:
            if item.severity == 'ERROR':
                print(f' - {item.code}: {item.message}')
